In [1]:
# Notebook 06 — Gradient-boosted regression
# Objective: build two joint price-change models, for 30 and 60 seconds.
# Training loss: MAE. Primary score: MAE; secondary score: RMSE, both in pips.
# Predicted EURUSD rate = latest known real mid + predicted pips * 0.0001.
# MAE training targets a conditional median point forecast, not P&L.
#
# Prerequisites: reviewed full-history Notebooks 02, 04 and 05.
# Inputs: their saved Parquet files, contracts and checkpoints.
# Stored timestamps: integer UTC nanoseconds; readable displays: Athens time.
# Size units: million EUR. Price units: USD per EUR.
#
# Six groups contain 29 numeric inputs:
# - Six past mid changes: 10/30/60 seconds and 3/5/15 minutes.
# - Bid size, ask size and spread.
# - All ten existing completed-candle SMA/EMA distances, in pips.
# - SampEn and conditional entropy scores at all five agreed windows.
# Current mid/spread/sizes use the last known real quote.
# Historical interpolation is usable only once its information is known.
# Actual price gaps require fresh history; MA history keeps its agreed behavior.
# Undefined inputs stay missing. No extra support/quality predictors are added.
#
# Fit: 2018. Choose tree count: January-March 2019.
# Compare: April-June 2019. Reserve 2026; do not model it here.
# One conservative parameter configuration; no tuning search.
# Benchmarks: zero change, trailing change and ridge regression.
# Ridge is a linear forecast with a penalty on large coefficients.
# Forecasts use zero modeled inference latency; revisit before market replay.
#
# Outputs: monthly input tables, two models, April-June forecasts and reports.
# Main displays: a lean score table and four forecast rows spanning 30 seconds.
# Inline checks cover timing, missing values, split boundaries and saved models.
# Initial group-removal/significance tests are deferred until output review.

In [2]:
# Settings and current-kernel environment.
# Install only missing modeling libraries; keep the current Python environment.
from pathlib import Path
import importlib
import importlib.util
import subprocess
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "data" / "processed" / "targets_05_v2").is_dir(), (
    "Start Jupyter from the repository or its notebooks folder."
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUN_ID = "regression_06_v1"
MEMORY_LIMIT = "2GB"
THREADS = 2
WORKING_MEMORY_BUDGET_GIB = 3.0
MAXIMUM_TREES = 1000
EARLY_STOPPING_ROUNDS = 50
EXAMPLE_START_ATHENS = "2019-04-01 10:00:00"

packages = {
    "lightgbm": "lightgbm",
    "sklearn": "scikit-learn",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing missing modeling libraries:", missing)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--only-binary=:all:",
        *missing,
    ])
    importlib.invalidate_caches()

PARTS = {}

print("Repository:", ROOT)
print("Run:", RUN_ID)

Installing missing modeling libraries: ['lightgbm']
Repository: C:\Users\Ijzd4\Desktop\Inventory-Aware FX Pricing and Hedging Simulator
Run: regression_06_v1


In [3]:
# Helper section: core. Stored in memory; written once in Cell 9.
PARTS["core"] = r'''
import gc
import hashlib
import importlib.metadata
import json
import pickle
import time
from collections import deque
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from functions.targets import forecast_targets_v2 as tg
from functions.volatility import multiscale_volatility as util

NS, STEP, PIP = tg.NS, tg.STEP, tg.PIP
TZ = "Europe/Athens"
TARGET_RUN = "targets_05_v2"
SAMPLE_RUN = "sampling_02_v1"
ENTROPY_RUN = "entropy_04_v1"

EXPECTED = {
    "targets": "a3ca7b4d579202995b60e4fb7ad1c2e9b0f146aa2317a7bafa9508f510e8072c",
    "sampling": "ad5235dc14f854d9f26ef04dd2c1fdf846faf05cff43eb9f84afccc229628e54",
    "entropy": "355d1378dab75041dc3a5af4a02aea55040be758a3a7eefa1a482c0d3ec6e841",
}

LAGS = {
    "10s": 1,
    "30s": 3,
    "60s": 6,
    "3m": 18,
    "5m": 30,
    "15m": 90,
}
WINDOWS = (15, 30, 60, 100, 200)
ENTROPY_WINDOWS = ("3m", "5m", "15m", "30m", "60m")

GROUPS = {
    "Price changes": [f"past_change_{k}_pips" for k in LAGS],
    "Displayed sizes": ["bid_size_million_eur", "ask_size_million_eur"],
    "Spread": ["spread_pips"],
    "MA distances": [
        f"close_minus_{kind}_{n}_pips"
        for kind in ("sma", "ema")
        for n in WINDOWS
    ],
    "Sample entropy": [
        f"sampen_score_{w}" for w in ENTROPY_WINDOWS
    ],
    "Conditional entropy": [
        f"ce_score_{w}" for w in ENTROPY_WINDOWS
    ],
}
FEATURES = [
    column
    for columns in GROUPS.values()
    for column in columns
]

PERIODS = {
    "train": (tg.ns("2018-01-01"), tg.ns("2019-01-01")),
    "early_stopping": (tg.ns("2019-01-01"), tg.ns("2019-04-01")),
    "comparison": (tg.ns("2019-04-01"), tg.ns("2019-07-01")),
}

BASE = [
    "issue_ns",
    "reference_quote_ns",
    "reference_mid",
    "reference_age_ns",
]
for _h in ("30s", "60s"):
    BASE += [
        f"usable_{_h}",
        f"label_available_ns_{_h}",
        f"target_change_{_h}_pips",
        f"target_future_mid_{_h}",
        f"endpoint_is_interpolated_{_h}",
    ]

def versions():
    return {
        **util.versions(),
        **{
            n: importlib.metadata.version(n)
            for n in ("lightgbm", "scikit-learn")
        },
    }

def dependencies():
    return {
        "regression_helper": util.sha(__file__),
        "target_helper": util.sha(tg.__file__),
        "shared_utilities": util.sha(util.__file__),
        "etl_utilities": util.sha(tg.fx.__file__),
    }

def read(con, paths, columns, order, where="TRUE"):
    con.read_parquet(
        [str(p) for p in paths],
        hive_partitioning=False,
    ).create_view("_input", replace=True)

    return con.sql(
        f"SELECT {', '.join(columns)} "
        f"FROM _input WHERE {where} ORDER BY {order}"
    ).to_arrow_table()

def matrix(table):
    return np.column_stack([
        tg.get(table, c, np.nan) for c in FEATURES
    ]).astype(np.float32, copy=False)

def role_mask(table, role, horizon):
    lower, upper = PERIODS[role]
    t = tg.get(table, "issue_ns")

    return (
        (t >= lower)
        & (t < upper)
        & tg.get(table, f"usable_{horizon}")
        & (
            tg.get(table, f"label_available_ns_{horizon}", 0)
            < upper
        )
    )

def publish(events, times, clock, availability, columns):
    valid = ~events[availability].is_null().to_numpy(
        zero_copy_only=False
    )
    nominal = tg.get(events, clock)[valid]
    av = tg.get(events, availability, 0)[valid]

    assert np.all(av >= nominal)
    assert np.all(np.diff(nominal) > 0)

    order = np.lexsort((nominal, av))

    # Among published rows, prefer the most recent nominal observation.
    latest = np.maximum.accumulate(order)
    index = np.searchsorted(av[order], times, side="right") - 1
    found = index >= 0
    safe = np.maximum(index, 0)

    info = np.zeros(len(times), dtype=np.int64)
    values = {
        c: np.full(len(times), np.nan)
        for c in columns
    }

    if len(order):
        chosen = latest[safe[found]]
        info[found] = av[chosen]

        for c in columns:
            data = tg.get(events, c, np.nan)[valid]
            values[c][found] = data[chosen]

    assert np.all(info <= times)
    return values, info

def rolling_max(values, width):
    out = np.zeros(len(values), dtype=np.int64)
    queue = deque()

    for i, value in enumerate(values):
        while queue and queue[0] <= i - width:
            queue.popleft()

        while queue and values[queue[-1]] <= value:
            queue.pop()

        queue.append(i)

        if i >= width - 1:
            out[i] = values[queue[0]]

    return out

def past_changes(grid, times, reference, reference_time):
    gt = tg.get(grid, "sample_ns")
    av = tg.get(grid, "available_ns")
    has = tg.get(grid, "has_price")
    price = tg.get(grid, "mid", np.nan)

    assert np.all(np.diff(gt) == STEP)
    position = np.searchsorted(gt, times)
    assert np.array_equal(gt[position], times)

    bad = np.r_[0, np.cumsum(~has)]
    values, information = {}, {}

    for key, width in LAGS.items():
        start = position - width
        safe = np.maximum(start, 0)
        enough = start >= 0
        clean = enough & ((bad[position] - bad[safe]) == 0)

        maximum = rolling_max(av, width)
        info = np.maximum(
            reference_time,
            maximum[np.maximum(position - 1, 0)],
        )
        ready = clean & (info <= times)
        column = f"past_change_{key}_pips"

        values[column] = np.where(
            ready,
            (reference - price[safe]) / PIP,
            np.nan,
        )
        information[column] = np.where(
            ready, info, 0
        ).astype(np.int64)

    return values, information

def save_text(path, text):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    tg.retry(lambda: temporary.write_text(text, encoding="utf-8"))
    tg.retry(lambda: temporary.replace(path))

def save_bytes(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    tg.retry(lambda: temporary.write_bytes(payload))
    tg.retry(lambda: temporary.replace(path))

def ridge_model():
    return make_pipeline(
        SimpleImputer(strategy="mean", copy=True),
        StandardScaler(copy=False),
        Ridge(alpha=1.0, solver="cholesky"),
    )

def check_evidence(ctx):
    path = ctx["reports"] / "synthetic_checks.json"
    assert path.exists(), (
        "Synthetic-check evidence is required before historical processing."
    )
    evidence = tg.fx.load_json(path)
    assert evidence["fingerprint"] == ctx["fingerprint"]
    assert evidence["passed"]
'''

In [4]:
# Helper section: inputs. Stored in memory; written once in Cell 9.
PARTS["inputs"] = r'''
def prepare(
    root, run_id, memory="2GB", threads=2, working_gb=3.0,
    maximum_trees=1000, early_stopping_rounds=50,
):
    root = Path(root).resolve()
    assert run_id and Path(run_id).name == run_id
    assert maximum_trees > 0 and early_stopping_rounds > 0

    sample = root / "data" / "processed" / SAMPLE_RUN
    target = root / "data" / "processed" / TARGET_RUN
    entropy = root / "outputs" / "features" / "entropy" / ENTROPY_RUN

    locations = {
        "sampling": root / "reports" / "tables" / "sampling" / SAMPLE_RUN,
        "targets": root / "reports" / "tables" / "targets" / TARGET_RUN,
        "entropy": root / "reports" / "tables" / "entropy" / ENTROPY_RUN,
    }

    for name, folder in locations.items():
        summary = tg.fx.load_json(folder / "execution_summary.json")
        assert summary["fingerprint"] == EXPECTED[name]
        assert summary["full_history_complete"]
        assert summary["nonzero_machine_checks"] == 0

        if name != "targets":
            completed = tg.fx.load_json(
                folder / "full_history_completion_checks.json"
            )
            assert all(completed["checks"].values())

    source_config = tg.fx.load_json(sample / "configuration.json")
    calendar = (
        root / "data" / "processed" / "etl_01_v3"
        / "merged_exclusion_windows.json"
    )
    assert util.sha(calendar) == source_config["calendar_sha256"]

    inputs = []

    def capture(kind, path, expected, checkpoint):
        info = util.file_info(path)
        assert all(info[k] == expected[k] for k in info), str(path)

        inputs.append({
            "kind": kind,
            "path": path.relative_to(root).as_posix(),
            **info,
            "checkpoint": checkpoint.relative_to(root).as_posix(),
            "checkpoint_sha": util.sha(checkpoint),
        })

    months = [
        str(m)
        for m in pd.period_range("2018-01", "2019-06", freq="M")
    ]
    source_months = ["2017-12", *months]
    raw = {
        p["year_month"]: p
        for p in source_config["inputs"]
    }

    for month in source_months:
        checkpoint = sample / "_checkpoints" / f"{month}.json"
        saved = tg.fx.load_json(checkpoint)

        assert saved["fingerprint"] == EXPECTED["sampling"]
        assert all(v == 0 for v in saved["checks"].values())

        for kind in ("sampled_10s", "candles_3min"):
            capture(
                kind,
                sample / kind / f"{month}.parquet",
                saved["outputs"][kind],
                checkpoint,
            )

        original = raw[month]
        cp = (
            root / "data" / "processed" / "etl_01_v3"
            / "_checkpoints" / f"observed_{month}.json"
        )
        assert util.sha(cp) == original["checkpoint_sha256"]
        capture("observed", root / original["path"], original, cp)

    for day in pd.date_range(
        "2017-12-31", "2019-06-30", freq="D"
    ):
        date = day.strftime("%Y-%m-%d")
        checkpoint = entropy / "_checkpoints" / f"{date}.json"
        saved = tg.fx.load_json(checkpoint)

        assert saved["fingerprint"] == EXPECTED["entropy"]
        assert all(v == 0 for v in saved["checks"].values())

        capture(
            "entropy",
            entropy / f"{date}.parquet",
            saved["output"],
            checkpoint,
        )

        if date >= "2018-01-01":
            checkpoint = target / "_checkpoints" / f"{date}.json"
            saved = tg.fx.load_json(checkpoint)

            assert saved["fingerprint"] == EXPECTED["targets"]
            assert all(v == 0 for v in saved["checks"].values())

            capture(
                "targets",
                target / f"{date}.parquet",
                saved["output"],
                checkpoint,
            )

    contracts = {
        path.relative_to(root).as_posix(): util.sha(path)
        for folder in (sample, target, entropy)
        for path in (
            folder / "configuration.json",
            folder / "output_contract.json",
        )
    }
    contracts[
        "data/processed/etl_01_v3/merged_exclusion_windows.json"
    ] = util.sha(calendar)
    contracts[
        "data/processed/targets_05_v2/forecast_output_contract.json"
    ] = util.sha(target / "forecast_output_contract.json")

    params = {
        "objective": "regression_l1",
        "metric": ["l1", "rmse"],
        "learning_rate": 0.03,
        "num_leaves": 15,
        "max_depth": 5,
        "min_data_in_leaf": 1000,
        "lambda_l2": 1.0,
        "max_bin": 127,
        "num_threads": threads,
        "verbosity": -1,
        "deterministic": True,
        "force_col_wise": True,
        "seed": 42,
        "use_missing": True,
        "zero_as_missing": False,
    }

    config = {
        "definition": "regression_06_v1",
        "run_id": run_id,
        "upstream": EXPECTED,
        "inputs": inputs,
        "contracts": contracts,
        "dependencies": dependencies(),
        "versions": versions(),
        "features": GROUPS,
        "periods_ns": PERIODS,
        "params": params,
        "maximum_trees": maximum_trees,
        "early_stopping_rounds": early_stopping_rounds,
        "inference_latency_ns": 0,
        "price_change_continuity": (
            "all intervening historical samples known; "
            "reset after actual gaps"
        ),
        "missing_features": "native NaN; no entropy zero imputation",
        "forecast_estimate": "conditional median point forecast",
        "model_available_ns": PERIODS["comparison"][0],
        "comparison_scope": "April-June 2019; final 2026 unopened",
    }

    # Normalize tuples and lists before restart comparisons.
    config = json.loads(json.dumps(config))
    fingerprint = hashlib.sha256(json.dumps(
        config, sort_keys=True, separators=(",", ":")
    ).encode()).hexdigest()

    feature_dir = root / "outputs" / "features" / run_id
    model_dir = root / "outputs" / "models" / run_id
    forecast_dir = root / "outputs" / "forecasts" / run_id
    reports = root / "reports" / "tables" / "regression" / run_id

    for path in (
        feature_dir,
        feature_dir / "_checkpoints",
        model_dir,
        forecast_dir,
        reports,
    ):
        path.mkdir(parents=True, exist_ok=True)

    path = feature_dir / "configuration.json"
    if path.exists():
        old = tg.fx.load_json(path)
        changed = [
            k for k in config
            if old.get(k) != config[k]
        ]
        assert not changed, (
            f"Changed configuration {changed}; use a new RUN_ID."
        )
    else:
        tg.write_json(path, config)

    tg.write_csv(
        pd.DataFrame(inputs),
        reports / "input_inventory.csv",
    )
    tg.write_json(reports / "feature_groups.json", GROUPS)

    return {
        "root": root,
        "feature_dir": feature_dir,
        "model_dir": model_dir,
        "forecast_dir": forecast_dir,
        "reports": reports,
        "months": months,
        "raw": raw,
        "sample": sample,
        "entropy": entropy,
        "target": target,
        "config": config,
        "fingerprint": fingerprint,
        "run_id": run_id,
        "memory": memory,
        "threads": threads,
        "working_gb": working_gb,
    }

def check_snapshot(ctx):
    config = ctx["config"]

    assert dependencies() == config["dependencies"]
    assert versions() == config["versions"]

    for relative, expected in config["contracts"].items():
        assert util.sha(ctx["root"] / relative) == expected, relative

    for item in config["inputs"]:
        actual = util.file_info(ctx["root"] / item["path"])
        assert all(actual[k] == item[k] for k in actual), item["path"]
        assert util.sha(
            ctx["root"] / item["checkpoint"]
        ) == item["checkpoint_sha"]

def paths_for(ctx, kind, lower, upper):
    selected = []

    for item in ctx["config"]["inputs"]:
        if item["kind"] != kind:
            continue

        stem = Path(item["path"]).stem

        if kind in ("entropy", "targets"):
            a = tg.ns(stem)
            b = int((
                pd.Timestamp(stem, tz=TZ)
                + pd.DateOffset(days=1)
            ).value)
        else:
            a, b = tg.fx.month_bounds(stem)

        if a < upper and b > lower:
            selected.append(ctx["root"] / item["path"])

    assert selected, (kind, lower, upper)
    return selected
'''

In [5]:
# Helper section: features. Stored in memory; written once in Cell 9.
PARTS["features"] = r'''
def make_month(ctx, month, con):
    lower, upper = tg.fx.month_bounds(month)

    base = read(
        con,
        paths_for(ctx, "targets", lower, upper),
        BASE,
        "issue_ns",
    )
    times = tg.get(base, "issue_ns")

    assert len(times) == (upper - lower) // STEP
    assert times[0] == lower
    assert times[-1] == upper - STEP
    assert np.all(np.diff(times) == STEP)

    reference = tg.get(base, "reference_mid")
    reference_time = tg.get(base, "reference_quote_ns")

    real = read(
        con,
        paths_for(ctx, "observed", lower, upper),
        ["ts_ns", "bid", "ask", "bid_size", "ask_size"],
        "ts_ns",
    )

    previous_month = str(pd.Period(month, freq="M") - 1)
    checkpoint = (
        ctx["root"] / "data" / "processed" / "etl_01_v3"
        / "_checkpoints" / f"observed_{previous_month}.json"
    )
    seed = tg.fx.load_json(checkpoint)["last"]

    quote_times = np.r_[
        np.int64(seed["ts_ns"]),
        tg.get(real, "ts_ns"),
    ]
    assert np.all(np.diff(quote_times) > 0)

    index = np.searchsorted(
        quote_times, times, side="right"
    ) - 1

    assert np.all(index >= 0)
    assert np.array_equal(quote_times[index], reference_time)

    quotes = {
        c: np.r_[float(seed[c]), tg.get(real, c)][index]
        for c in ("bid", "ask", "bid_size", "ask_size")
    }
    assert np.allclose(
        (quotes["bid"] + quotes["ask"]) / 2,
        reference,
        rtol=0,
        atol=1e-12,
    )

    values = {
        "bid_size_million_eur": quotes["bid_size"],
        "ask_size_million_eur": quotes["ask_size"],
        "spread_pips": (quotes["ask"] - quotes["bid"]) / PIP,
    }
    information = {
        "market_information_ns": reference_time,
    }
    del real, quote_times, quotes

    grid = read(
        con,
        paths_for(ctx, "sampled_10s", lower - 90 * STEP, upper),
        ["sample_ns", "available_ns", "has_price", "mid"],
        "sample_ns",
        f"sample_ns >= {lower - 90 * STEP} "
        f"AND sample_ns < {upper}",
    )
    changes, change_times = past_changes(
        grid, times, reference, reference_time
    )
    values.update(changes)
    information.update({
        c + "_information_ns": v
        for c, v in change_times.items()
    })
    del grid

    ma_columns = GROUPS["MA distances"]
    candles = read(
        con,
        paths_for(
            ctx, "candles_3min",
            lower - 32 * 86400 * NS, upper,
        ),
        ["candle_end_ns", "ma_available_ns", *ma_columns],
        "candle_end_ns",
        "ma_eligible",
    )
    selected, info = publish(
        candles,
        times,
        "candle_end_ns",
        "ma_available_ns",
        ma_columns,
    )
    values.update(selected)
    information["ma_information_ns"] = info
    del candles

    entropy_columns = ["sample_ns"]
    for family in ("sampen", "ce"):
        for w in ENTROPY_WINDOWS:
            entropy_columns += [
                f"{family}_score_{w}",
                f"{family}_available_ns_{w}",
            ]

    events = read(
        con,
        paths_for(ctx, "entropy", lower - 86400 * NS, upper),
        entropy_columns,
        "sample_ns",
        f"sample_ns >= {lower - 86400 * NS} "
        f"AND sample_ns < {upper}",
    )

    # Keep null/unready events: they replace earlier defined values.
    for family in ("sampen", "ce"):
        for w in ENTROPY_WINDOWS:
            column = f"{family}_score_{w}"
            selected, info = publish(
                events,
                times,
                "sample_ns",
                f"{family}_available_ns_{w}",
                [column],
            )
            values.update(selected)
            information[column + "_information_ns"] = info

    assert set(values) == set(FEATURES)

    output = {
        c: base[c]
        for c in base.column_names
    }

    for c in FEATURES:
        output[c] = pa.array(
            values[c],
            type=pa.float32(),
            from_pandas=True,
        )

    for c, value in information.items():
        assert np.all(value <= times), c
        output[c] = tg.arr(value, value > 0)

    output["input_information_max_ns"] = pa.array(
        np.maximum.reduce(list(information.values())),
        type=pa.int64(),
    )

    table = pa.table(output)
    x = matrix(table)

    checks = {
        "infinite_features": int(np.isinf(x).sum()),
        "information_after_issue": int(np.count_nonzero(
            tg.get(table, "input_information_max_ns") > times
        )),
        "reference_after_issue": int(np.count_nonzero(
            reference_time > times
        )),
        "wrong_feature_count": int(x.shape[1] != 29),
    }

    for c in GROUPS["Sample entropy"] + GROUPS["Conditional entropy"]:
        v = values[c]
        checks[c + "_outside_0_1"] = int(np.count_nonzero(
            np.isfinite(v) & ((v < 0) | (v > 1))
        ))

    assert all(v == 0 for v in checks.values()), checks

    counts = {
        "month": month,
        "rows": table.num_rows,
    }
    for role in PERIODS:
        for h in ("30s", "60s"):
            counts[f"{role}_{h}"] = int(
                role_mask(table, role, h).sum()
            )

    missing = [
        {
            "month": month,
            "feature": c,
            "missing": int(table[c].null_count),
            "rows": table.num_rows,
        }
        for c in FEATURES
    ]

    return table, checks, counts, missing

def assemble(ctx):
    check_evidence(ctx)
    check_snapshot(ctx)
    records = []

    with duckdb.connect(config={
        "memory_limit": ctx["memory"],
        "threads": str(ctx["threads"]),
    }) as con:
        for month in ctx["months"]:
            path = ctx["feature_dir"] / f"{month}.parquet"
            checkpoint = (
                ctx["feature_dir"] / "_checkpoints"
                / f"{month}.json"
            )

            if checkpoint.exists():
                saved = tg.fx.load_json(checkpoint)
                assert saved["fingerprint"] == ctx["fingerprint"]
                assert util.file_info(path) == saved["output"]
                assert all(
                    v == 0 for v in saved["checks"].values()
                )
            else:
                started = time.perf_counter()
                table, checks, counts, missing = make_month(
                    ctx, month, con
                )
                tg.write_parquet(table, path)

                saved = {
                    "fingerprint": ctx["fingerprint"],
                    "checks": checks,
                    "counts": counts,
                    "missing": missing,
                    "output": util.file_info(path),
                }
                tg.write_json(checkpoint, saved)

                print(
                    f"{month}: {table.num_rows:,} rows, "
                    f"{time.perf_counter() - started:.1f}s"
                )
                del table

            records.append(saved)

    check_snapshot(ctx)

    counts = pd.DataFrame([
        r["counts"] for r in records
    ])
    missing = pd.DataFrame([
        v for r in records for v in r["missing"]
    ])
    checks = pd.DataFrame([
        {"month": r["counts"]["month"], **r["checks"]}
        for r in records
    ])

    assert int(
        checks.drop(columns="month").to_numpy().sum()
    ) == 0

    tg.write_csv(
        counts, ctx["reports"] / "feature_month_counts.csv"
    )
    tg.write_csv(
        missing, ctx["reports"] / "feature_missing_counts.csv"
    )
    tg.write_csv(
        checks, ctx["reports"] / "feature_checks.csv"
    )

    rows = []
    for role in PERIODS:
        for h in ("30s", "60s"):
            n = int(counts[f"{role}_{h}"].sum())
            rows.append({
                "Period": role,
                "Horizon": h,
                "Usable labels": n,
            })

    return pd.DataFrame(rows)

def memory_report(ctx, counts):
    rows = []

    for h in ("30s", "60s"):
        n = int(counts.loc[
            counts["Horizon"].eq(h)
            & counts["Period"].isin(["train", "early_stopping"]),
            "Usable labels",
        ].sum())

        raw_gb = n * (4 * len(FEATURES) + 8) / 1024**3
        working_gb = 6 * raw_gb

        rows.append({
            "Horizon": h,
            "Input arrays GiB": raw_gb,
            "Estimated working GiB": working_gb,
            "Budget GiB": ctx["working_gb"],
        })

    report = pd.DataFrame(rows)
    tg.write_csv(
        report, ctx["reports"] / "memory_estimate.csv"
    )

    assert report["Estimated working GiB"].le(
        ctx["working_gb"]
    ).all(), (
        "Memory estimate exceeds the settings budget. "
        "Review before fitting."
    )

    return report
'''

In [6]:
# Helper section: models. Stored in memory; written once in Cell 9.
PARTS["models"] = r'''
def check_features(ctx):
    for month in ctx["months"]:
        checkpoint = (
            ctx["feature_dir"] / "_checkpoints"
            / f"{month}.json"
        )
        saved = tg.fx.load_json(checkpoint)

        assert saved["fingerprint"] == ctx["fingerprint"]
        assert all(v == 0 for v in saved["checks"].values())
        assert util.file_info(
            ctx["feature_dir"] / f"{month}.parquet"
        ) == saved["output"]

def load_role(ctx, role, horizon):
    lower, upper = PERIODS[role]

    paths = [
        ctx["feature_dir"] / f"{m}.parquet"
        for m in ctx["months"]
        if tg.fx.month_bounds(m)[0] < upper
        and tg.fx.month_bounds(m)[1] > lower
    ]

    where = (
        f"issue_ns >= {lower} AND issue_ns < {upper} "
        f"AND usable_{horizon} "
        f"AND label_available_ns_{horizon} < {upper}"
    )

    with duckdb.connect(config={
        "memory_limit": ctx["memory"],
        "threads": str(ctx["threads"]),
    }) as con:
        table = read(
            con,
            paths,
            [
                *FEATURES,
                f"target_change_{horizon}_pips",
                f"label_available_ns_{horizon}",
            ],
            f"label_available_ns_{horizon}",
            where,
        )

    assert table.num_rows > 0, (
        role, horizon, "No usable labels"
    )

    x = matrix(table)
    y = tg.get(
        table, f"target_change_{horizon}_pips"
    ).astype(float, copy=True)
    information = int(
        tg.get(table, f"label_available_ns_{horizon}").max()
    )

    assert len(y) and np.isfinite(y).all()
    assert not np.isinf(x).any()
    return x, y, information

def losses(actual, predicted):
    error = np.asarray(predicted) - np.asarray(actual)
    assert len(error) and np.isfinite(error).all()

    return {
        "mae": float(np.abs(error).mean()),
        "rmse": float(np.sqrt(np.square(error).mean())),
    }

def fit_models(ctx):
    check_evidence(ctx)
    check_snapshot(ctx)
    check_features(ctx)

    saved_counts = [
        tg.fx.load_json(
            ctx["feature_dir"] / "_checkpoints" / f"{m}.json"
        )["counts"]
        for m in ctx["months"]
    ]
    counts = pd.DataFrame([
        {
            "Period": role,
            "Horizon": h,
            "Usable labels": sum(
                c[f"{role}_{h}"] for c in saved_counts
            ),
        }
        for role in PERIODS
        for h in ("30s", "60s")
    ])
    memory_report(ctx, counts)

    bundles, fit_rows, gains = {}, [], []

    for h in ("30s", "60s"):
        meta_path = ctx["model_dir"] / f"{h}_metadata.json"
        model_path = ctx["model_dir"] / f"{h}_lightgbm.txt"
        ridge_path = ctx["model_dir"] / f"{h}_ridge.pkl"

        if meta_path.exists():
            meta = tg.fx.load_json(meta_path)

            assert meta["fingerprint"] == ctx["fingerprint"]
            assert util.sha(model_path) == meta["lightgbm_sha"]
            assert util.sha(ridge_path) == meta["ridge_sha"]

            booster = lgb.Booster(
                model_str=model_path.read_text(encoding="utf-8")
            )
            ridge = pickle.loads(ridge_path.read_bytes())

        else:
            started = time.perf_counter()

            x, y, fit_info = load_role(ctx, "train", h)
            stop_x, stop_y, stop_info = load_role(
                ctx, "early_stopping", h
            )

            supported = np.isfinite(x).any(axis=0)
            assert supported.all(), (
                "Features unavailable throughout training: "
                + str([
                    c for c, ok in zip(FEATURES, supported)
                    if not ok
                ])
            )

            train_data = lgb.Dataset(
                x,
                label=y,
                feature_name=FEATURES,
            )
            stop_data = lgb.Dataset(
                stop_x,
                label=stop_y,
                reference=train_data,
                feature_name=FEATURES,
            )

            booster = lgb.train(
                ctx["config"]["params"],
                train_data,
                num_boost_round=ctx["config"]["maximum_trees"],
                valid_sets=[stop_data],
                valid_names=["early_stopping"],
                callbacks=[
                    lgb.early_stopping(
                        ctx["config"]["early_stopping_rounds"],
                        first_metric_only=True,
                        verbose=False,
                    ),
                    lgb.log_evaluation(0),
                ],
            )

            assert booster.best_iteration >= 1
            del train_data, stop_data
            gc.collect()

            ridge = ridge_model()
            ridge.fit(x, y)

            predicted = booster.predict(
                stop_x,
                num_iteration=booster.best_iteration,
            )
            assert np.isfinite(predicted).all()

            cuts = np.unique(np.quantile(
                predicted, [0.2, 0.4, 0.6, 0.8]
            ))
            cuts = cuts[
                (cuts > predicted.min())
                & (cuts < predicted.max())
            ]

            latest = max(fit_info, stop_info)
            ready = PERIODS["comparison"][0]
            assert latest < ready

            text = booster.model_to_string(
                num_iteration=booster.best_iteration
            )
            restored = lgb.Booster(model_str=text)

            assert np.allclose(
                predicted[:1000],
                restored.predict(stop_x[:1000]),
                rtol=0,
                atol=1e-10,
            )

            payload = pickle.dumps(
                ridge, protocol=pickle.HIGHEST_PROTOCOL
            )
            restored_ridge = pickle.loads(payload)

            assert np.allclose(
                ridge.predict(stop_x[:1000]),
                restored_ridge.predict(stop_x[:1000]),
                rtol=0,
                atol=1e-10,
            )

            save_text(model_path, text)
            save_bytes(ridge_path, payload)

            meta = {
                "fingerprint": ctx["fingerprint"],
                "model_identifier": f"{ctx['run_id']}_{h}",
                "horizon": h,
                "features": FEATURES,
                "objective": "regression_l1",
                "estimate": "conditional median point forecast",
                "training_rows": len(y),
                "early_stopping_rows": len(stop_y),
                "best_iteration": int(booster.best_iteration),
                "fit_information_cutoff_ns": fit_info,
                "selection_information_cutoff_ns": stop_info,
                "training_cutoff_ns": latest,
                "model_available_ns": ready,
                "cutoff_convention": "latest included information time",
                "early_stopping_end_exclusive_ns": ready,
                "inference_latency_ns": ctx["config"]["inference_latency_ns"],
                "bucket_edges": cuts.tolist(),
                "bucket_basis": (
                    "January-March forecasts; "
                    "fixed before April comparison"
                ),
                "early_stopping_scores": losses(stop_y, predicted),
                "lightgbm_sha": util.sha(model_path),
                "ridge_sha": util.sha(ridge_path),
                "fit_minutes": (
                    time.perf_counter() - started
                ) / 60,
            }
            tg.write_json(meta_path, meta)

            del (
                x, y, stop_x, stop_y, predicted,
                restored, restored_ridge, payload
            )
            gc.collect()

        assert (
            meta["training_cutoff_ns"]
            < meta["model_available_ns"]
        )

        bundles[h] = {
            "booster": booster,
            "ridge": ridge,
            "meta": meta,
        }
        fit_rows.append({
            "Horizon": h,
            "Training examples": meta["training_rows"],
            "Early-stopping examples": meta["early_stopping_rows"],
            "Trees retained": meta["best_iteration"],
        })

        importance = booster.feature_importance(
            importance_type="gain"
        )
        for c, value in zip(FEATURES, importance):
            group = next(
                k for k, cols in GROUPS.items()
                if c in cols
            )
            gains.append({
                "horizon": h,
                "group": group,
                "feature": c,
                "gain": float(value),
            })

    check_snapshot(ctx)
    check_features(ctx)

    tg.write_csv(
        pd.DataFrame(gains),
        ctx["reports"] / "feature_gain_diagnostic.csv",
    )
    result = pd.DataFrame(fit_rows)
    tg.write_csv(
        result, ctx["reports"] / "model_fit_summary.csv"
    )

    return bundles, result
'''

In [7]:
# Helper section: evaluation. Stored in memory; written once in Cell 9.
PARTS["evaluation"] = r'''
def evaluate(
    ctx, bundles,
    example_start="2019-04-01 10:00:00",
):
    check_evidence(ctx)
    check_snapshot(ctx)
    check_features(ctx)

    scores, bucket_rows, all_checks = [], [], []
    examples, example_date = None, None
    demo_start = tg.ns(example_start)

    contract = tg.fx.load_json(
        ctx["target"] / "forecast_output_contract.json"
    )

    with duckdb.connect(config={
        "memory_limit": ctx["memory"],
        "threads": str(ctx["threads"]),
    }) as con:
        for month in ("2019-04", "2019-05", "2019-06"):
            table = read(
                con,
                [ctx["feature_dir"] / f"{month}.parquet"],
                [*BASE, *FEATURES, "input_information_max_ns"],
                "issue_ns",
            )
            times = tg.get(table, "issue_ns")
            reference = tg.get(table, "reference_mid")
            x = matrix(table)

            forecast = {
                c: table[c]
                for c in (
                    "issue_ns",
                    "reference_quote_ns",
                    "reference_mid",
                    "reference_age_ns",
                )
            }
            benchmark = {
                "issue_ns": table["issue_ns"],
            }

            masks, predicted_prices, outcome_prices = {}, {}, {}
            checks = {"month": month}

            for h, bundle in bundles.items():
                meta = bundle["meta"]
                assert np.all(
                    times >= meta["model_available_ns"]
                )

                predicted = bundle["booster"].predict(x)
                ridge = bundle["ridge"].predict(x)

                trailing = tg.get(
                    table, f"past_change_{h}_pips", np.nan
                )
                trailing = np.where(
                    np.isfinite(trailing), trailing, 0.0
                )

                forecasts = {
                    "LightGBM": predicted,
                    "Zero change": np.zeros(len(times)),
                    "Trailing change": trailing,
                    "Ridge": ridge,
                }
                assert all(
                    np.isfinite(v).all()
                    for v in forecasts.values()
                )

                availability = (
                    times + meta["inference_latency_ns"]
                )
                price = tg.predicted_mid(reference, predicted)

                forecast.update({
                    f"predicted_change_{h}_pips": pa.array(predicted),
                    f"expected_eurusd_rate_{h}": pa.array(price),
                    f"forecast_available_ns_{h}": pa.array(availability),
                    f"model_identifier_{h}": pa.array(
                        [meta["model_identifier"]] * len(times)
                    ),
                    f"training_cutoff_ns_{h}": pa.array(
                        np.full(
                            len(times),
                            meta["training_cutoff_ns"],
                            dtype=np.int64,
                        )
                    ),
                })

                for name, value in forecasts.items():
                    if name != "LightGBM":
                        column = (
                            f"{name.lower().replace(' ', '_')}"
                            f"_{h}_pips"
                        )
                        benchmark[column] = pa.array(value)

                mask = role_mask(table, "comparison", h)
                actual = tg.get(
                    table, f"target_change_{h}_pips", np.nan
                )
                actual_price = tg.get(
                    table, f"target_future_mid_{h}", np.nan
                )

                n = int(mask.sum())
                assert n > 0
                assert np.isfinite(actual[mask]).all()
                assert np.isfinite(actual_price[mask]).all()

                for name, value in forecasts.items():
                    error = value[mask] - actual[mask]
                    scores.append({
                        "month": month,
                        "horizon": h,
                        "model": name,
                        "count": n,
                        "eligible": n,
                        "absolute_error_sum": float(
                            np.abs(error).sum()
                        ),
                        "squared_error_sum": float(
                            np.square(error).sum()
                        ),
                    })

                bucket = np.searchsorted(
                    meta["bucket_edges"],
                    predicted,
                    side="right",
                ) + 1

                for b in np.unique(bucket[mask]):
                    use = mask & (bucket == b)
                    bucket_rows.append({
                        "horizon": h,
                        "bucket": int(b),
                        "count": int(use.sum()),
                        "predicted_sum": float(
                            predicted[use].sum()
                        ),
                        "outcome_sum": float(
                            actual[use].sum()
                        ),
                        "absolute_error_sum": float(
                            np.abs(
                                predicted[use] - actual[use]
                            ).sum()
                        ),
                    })

                checks[h + "_nonpositive_forecast_price"] = int(
                    np.count_nonzero(price <= 0)
                )
                checks[h + "_late_inputs"] = int(
                    np.count_nonzero(
                        tg.get(table, "input_information_max_ns")
                        > times
                    )
                )
                checks[h + "_model_from_future"] = int(
                    np.count_nonzero(
                        times <= meta["training_cutoff_ns"]
                    )
                )
                checks[h + "_early_forecast"] = int(
                    np.count_nonzero(availability < times)
                )
                checks[h + "_wrong_outcome_conversion"] = int(
                    np.count_nonzero(
                        mask
                        & (
                            np.abs(
                                tg.predicted_mid(reference, actual)
                                - actual_price
                            ) > 1e-12
                        )
                    )
                )

                masks[h] = mask
                predicted_prices[h] = price
                outcome_prices[h] = actual_price

            assert set(contract["columns"]).issubset(forecast)
            assert all(
                v == 0
                for k, v in checks.items()
                if k != "month"
            ), checks

            all_checks.append(checks)

            # No future outcomes or eligibility flags enter forecast files.
            tg.write_parquet(
                pa.table(forecast),
                ctx["forecast_dir"] / f"{month}.parquet",
            )
            tg.write_parquet(
                pa.table(benchmark),
                ctx["forecast_dir"] / f"{month}_benchmarks.parquet",
            )

            if examples is None:
                eligible = (
                    masks["30s"]
                    & masks["60s"]
                    & (times >= demo_start)
                )
                runs = np.flatnonzero(
                    np.convolve(
                        eligible.astype(int),
                        np.ones(4, dtype=int),
                        mode="valid",
                    ) == 4
                )

                if len(runs):
                    index = np.arange(
                        int(runs[0]), int(runs[0]) + 4
                    )
                    display_time = pd.to_datetime(
                        times[index], unit="ns", utc=True
                    ).tz_convert(TZ)

                    data = {
                        "Forecast time": display_time.strftime("%H:%M:%S"),
                        "Current mid": [
                            f"{v:.6f}"
                            for v in reference[index]
                        ],
                    }

                    for h in ("30s", "60s"):
                        synthetic = tg.get(
                            table,
                            f"endpoint_is_interpolated_{h}",
                            False,
                        )
                        data[f"Predicted +{h}"] = [
                            f"{v:.6f}"
                            for v in predicted_prices[h][index]
                        ]
                        data[f"Outcome +{h}"] = [
                            f"{outcome_prices[h][i]:.6f}"
                            + ("*" if synthetic[i] else "")
                            for i in index
                        ]

                    examples = pd.DataFrame(data)
                    example_date = display_time[0].strftime(
                        "%A, %d %B %Y"
                    )

            print(
                f"{month}: comparison forecasts and checks saved."
            )
            del table, x

    monthly = pd.DataFrame(scores)
    sums = [
        "count",
        "eligible",
        "absolute_error_sum",
        "squared_error_sum",
    ]
    overall = (
        monthly.groupby(
            ["horizon", "model"], sort=False
        )[sums]
        .sum()
        .reset_index()
    )

    for frame in (monthly, overall):
        frame["mae_pips"] = (
            frame["absolute_error_sum"] / frame["count"]
        )
        frame["rmse_pips"] = np.sqrt(
            frame["squared_error_sum"] / frame["count"]
        )
        frame["coverage"] = (
            frame["count"] / frame["eligible"]
        )

    assert overall["mae_pips"].le(
        overall["rmse_pips"] + 1e-10
    ).all()
    assert overall.groupby(
        "horizon"
    )["count"].nunique().eq(1).all()

    buckets = (
        pd.DataFrame(bucket_rows)
        .groupby(["horizon", "bucket"], sort=True)
        .sum(numeric_only=True)
        .reset_index()
    )
    buckets["mean_predicted_pips"] = (
        buckets["predicted_sum"] / buckets["count"]
    )
    buckets["mean_outcome_pips"] = (
        buckets["outcome_sum"] / buckets["count"]
    )
    buckets["mae_pips"] = (
        buckets["absolute_error_sum"] / buckets["count"]
    )

    check_frame = pd.DataFrame(all_checks)
    assert int(
        check_frame.drop(columns="month").to_numpy().sum()
    ) == 0
    assert examples is not None, (
        "No four-row example window; review coverage."
    )

    tg.write_csv(
        overall, ctx["reports"] / "performance_summary.csv"
    )
    tg.write_csv(
        monthly, ctx["reports"] / "monthly_performance.csv"
    )
    tg.write_csv(
        buckets, ctx["reports"] / "forecast_buckets.csv"
    )
    tg.write_csv(
        check_frame, ctx["reports"] / "forecast_checks.csv"
    )
    tg.write_csv(
        examples, ctx["reports"] / "forecast_price_examples.csv"
    )

    friendly = pd.DataFrame({
        "Horizon": overall["horizon"],
        "Model / benchmark": overall["model"],
        "MAE (pips)": overall["mae_pips"].round(4),
        "RMSE (pips)": overall["rmse_pips"].round(4),
        "Coverage": [
            f"{n:,} ({rate:.0%})"
            for n, rate in zip(
                overall["count"], overall["coverage"]
            )
        ],
    })

    evidence = tg.fx.load_json(
        ctx["reports"] / "synthetic_checks.json"
    )

    summary = {
        "notebook": "06_gradient_boosted_regression.ipynb",
        "run_id": ctx["run_id"],
        "fingerprint": ctx["fingerprint"],
        "upstream": EXPECTED,
        "feature_groups": 6,
        "numeric_features": 29,
        "feature_months": len(ctx["months"]),
        "lightgbm_models": 2,
        "ridge_benchmarks": 2,
        "objective": "regression_l1",
        "primary_metric": "MAE in pips",
        "secondary_metric": "RMSE in pips",
        "train": "2018",
        "early_stopping": "January-March 2019",
        "comparison": "April-June 2019",
        "final_evaluation_accessed_for_modeling": False,
        "forecast_rows_include_unscorable_times": True,
        "inference_latency_ns": ctx["config"]["inference_latency_ns"],
        "latency_status": (
            "zero modeled inference latency; "
            "revisit before market replay"
        ),
        "synthetic_checks_passed": evidence["passed"],
        "nonzero_machine_checks": 0,
        "human_review": "pending",
        "example_date": example_date,
        "forecast_interpretation": (
            "conditional median price-change forecast; "
            "no P&L forecast"
        ),
        "group_significance": (
            "deferred until full-model performance review"
        ),
        "source_stability": (
            "file metadata and checkpoint hashes; "
            "no bulk content hashes"
        ),
    }

    check_snapshot(ctx)
    check_features(ctx)
    tg.write_json(
        ctx["reports"] / "execution_summary.json",
        summary,
    )

    return friendly, examples, summary
'''

In [8]:
# Helper section: finish. Stored in memory; written once in Cell 9.
PARTS["finish"] = r'''
def synthetic_checks():
    origin = tg.ns("2018-06-15 10:00:00")

    events = pa.table({
        "sample_ns": np.array([
            origin,
            origin + 10 * NS,
            origin + 20 * NS,
            origin + 30 * NS,
        ]),
        "published_ns": np.array([
            origin,
            origin + 20 * NS,
            origin + 20 * NS + 123,
            origin + 50 * NS,
        ]),
        "value": pa.array(
            [1.0, 2.0, None, 9.0],
            type=pa.float64(),
        ),
    })
    times = np.array([
        origin + 19 * NS,
        origin + 20 * NS + 122,
        origin + 20 * NS + 123,
        origin + 50 * NS,
    ], dtype=np.int64)

    values, information = publish(
        events, times, "sample_ns", "published_ns", ["value"]
    )

    assert values["value"][0] == 1
    assert values["value"][1] == 2
    assert np.isnan(values["value"][2])
    assert values["value"][3] == 9
    assert information[2] == origin + 20 * NS + 123

    checks = {
        "publication_time_and_integer_precision": True,
        "published_null_replaces_old_value": True,
    }

    delayed = pa.table({
        "sample_ns": [
            origin, origin + 10 * NS, origin + 20 * NS
        ],
        "published_ns": [
            origin, origin + 40 * NS, origin + 30 * NS
        ],
        "value": [1.0, 2.0, 3.0],
    })
    chosen, _ = publish(
        delayed,
        np.array([origin + 50 * NS]),
        "sample_ns",
        "published_ns",
        ["value"],
    )
    assert chosen["value"][0] == 3
    checks[
        "late_old_publication_does_not_replace_newer_observation"
    ] = True

    gt = origin + np.arange(40, dtype=np.int64) * STEP
    available = gt.copy()
    available[5] = gt[10] + 123

    has = np.ones(40, dtype=bool)
    has[20] = False
    mid = 1.1 + np.arange(40) * PIP

    grid = pa.table({
        "sample_ns": gt,
        "available_ns": available,
        "has_price": has,
        "mid": tg.arr(mid, has),
    })
    selected = np.array([8, 11, 23, 24])

    changes, info = past_changes(
        grid,
        gt[selected],
        mid[selected],
        gt[selected],
    )

    assert np.isnan(changes["past_change_30s_pips"][0])
    assert abs(
        changes["past_change_30s_pips"][1] - 3
    ) < 1e-8
    assert np.isnan(changes["past_change_30s_pips"][2])
    assert abs(
        changes["past_change_30s_pips"][3] - 3
    ) < 1e-8
    assert np.all(
        info["past_change_30s_pips"] <= gt[selected]
    )
    checks[
        "price_changes_wait_for_known_anchors_and_fresh_history"
    ] = True

    cut = PERIODS["early_stopping"][1]
    toy = pa.table({
        "issue_ns": [cut - STEP, cut - STEP],
        "usable_30s": [True, True],
        "label_available_ns_30s": [cut, cut - 1],
    })
    assert np.array_equal(
        role_mask(toy, "early_stopping", "30s"),
        [False, True],
    )
    checks["additional_april_information_cutoff"] = True

    rng = np.random.default_rng(42)
    x = rng.normal(size=(160, 3)).astype(np.float32)
    y = x[:, 0].copy()
    x[::7, 1] = np.nan

    training = lgb.Dataset(x[:100], label=y[:100])
    stopping = lgb.Dataset(
        x[100:], label=y[100:], reference=training
    )
    model = lgb.train(
        {
            "objective": "regression_l1",
            "metric": ["l1", "rmse"],
            "min_data_in_leaf": 5,
            "num_leaves": 4,
            "verbosity": -1,
            "num_threads": 1,
        },
        training,
        num_boost_round=5,
        valid_sets=[stopping],
        callbacks=[
            lgb.early_stopping(
                2, first_metric_only=True, verbose=False
            ),
            lgb.log_evaluation(0),
        ],
    )

    restored = lgb.Booster(
        model_str=model.model_to_string()
    )
    assert np.allclose(
        model.predict(x[100:]),
        restored.predict(x[100:]),
        rtol=0,
        atol=1e-10,
    )

    ridge = ridge_model().fit(x[:100], y[:100])
    assert np.isfinite(
        ridge.predict(x[100:])
    ).all()

    checks[
        "lightgbm_native_missing_early_stopping_and_serialization"
    ] = True
    checks[
        "ridge_training_only_imputation_interface"
    ] = True

    assert np.allclose(
        tg.predicted_mid([1.1, 1.1], [2, -1]),
        [1.10020, 1.09990],
        rtol=0,
        atol=1e-12,
    )
    checks["pip_to_price_conversion"] = True

    return {
        "passed": all(checks.values()),
        "checks": checks,
    }

def record(ctx, summary):
    decisions = """
## Notebook 06 — initial regression experiment
- Agreed: two LightGBM models, predicting 30- and 60-second price changes.
- MAE training loss; conditional median point forecasts. No P&L prediction.
- MAE in pips is primary; RMSE in pips is secondary.
- Six groups: past changes, displayed sizes, spread, MA distances, SampEn and CE.
- Past changes: 10/30/60 seconds and 3/5/15 minutes.
- All ten completed-candle SMA/EMA distance columns: 15/30/60/100/200.
- Both entropy families use 3/5/15/30/60-minute scores.
- Retain valid target examples with missing features; native missing handling.
- Current mid/spread/sizes use the last known real quote.
- Historical reconstructed prices enter only after their information is available.
- Past changes need fresh continuous history after actual data gaps.
- Quotes in excluded clock hours can provide history; target exclusions still apply.
- Preserve MA history and the established entropy restart rules.
- Fit on 2018; choose tree count using January-March 2019.
- Early-stopping label information must be available before April 1.
- Compare the fixed models on April-June 2019. Keep 2026 untouched.
- Conservative single parameter configuration; no tuning search or blending.
- Benchmarks: zero change, trailing horizon change and ridge.
- Trailing baseline falls back to zero change when its history is unavailable.
- Ridge uses training-only mean imputation and scaling; source features stay missing.
- Forecast timestamps cover both fitting and early-stopping information.
- April 1 model readiness is a research clock assumption, not measured fit latency.
- Zero modeled inference latency is provisional until replay design.
- Export forecasts without future outcomes or future eligibility flags.
- Later group-removal tests require a separate agreed diagnostic experiment.
- After April-June informs changes, assess the revised specification on a later development block.
- Feature gain is descriptive; it is not a statistical-significance test.
"""

    contributions = """
## Notebook 06 — agreed contributions
- Raymond Fendi chose LightGBM and two joint models using all six feature groups.
- Requested every agreed SMA and EMA lookback, plus sample and conditional entropy.
- Chose MAE primary, RMSE secondary and MAE training loss.
- Agreed the six price-change lookbacks and the two 2019 validation roles.
- Chose to retain valid examples with missing inputs.
- Requested concise performance tables and a 30-second forecast-price example.
- Clarified that P&L must come from simulated trades and accounting.
- Requested later evaluation of each group's additional forecasting value.
- OpenAI Codex supplied the notebook, implementation and inline validation code.
- This executed run produced two LightGBM models and benchmark comparisons.
- Output interpretation and human review remain pending.
"""

    state = f"""
## Notebook 06 — current state
- Run: {ctx['run_id']}; fingerprint: {ctx['fingerprint']}.
- Features: outputs/features/{ctx['run_id']}.
- Models: outputs/models/{ctx['run_id']}.
- Forecasts: outputs/forecasts/{ctx['run_id']}.
- Reports: reports/tables/regression/{ctx['run_id']}.
- Models fit on 2018; early stopping January-March 2019.
- Comparison forecasts cover April-June 2019; final 2026 was not modeled.
- Machine checks passed; output review remains pending.
- Next: review forecasting performance and coverage before further experiments.
- No pricing, inventory or exit policy was implemented.
- No upstream numerical outputs were changed.
"""

    for path, marker, body in (
        (
            "docs/design_decisions.md",
            "notebook_06_design_v1",
            decisions,
        ),
        (
            "CONTRIBUTIONS.md",
            "notebook_06_contributions_v1",
            contributions,
        ),
        (
            "docs/project_state.md",
            "notebook_06_state_v1",
            state,
        ),
        (
            "README.md",
            "notebook_06_method_v1",
            "## Forecasting development\n"
            "- Notebook 05: complete 30/60-second labels and chronological splits.\n"
            "- Notebook 06: two MAE-trained LightGBM models and benchmark comparisons.\n"
            "- Training: 2018; early stopping: January-March 2019; comparison: April-June 2019.\n"
            "- Final evaluation: 2026, reserved. Forecasts do not predict P&L.\n"
            "- Results pending execution and validation.",
        ),
    ):
        util.write_section(
            ctx["root"] / path,
            marker,
            body.strip(),
        )

    util.write_section(
        ctx["root"] / "docs/project_state.md",
        "risk-interpretation-and-targets-2026-11-05",
        "## Earlier target discussion\n"
        "Superseded by the completed Notebook 05 and current Notebook 06 records.\n"
        "The side-dependent interpretation and 1/30/60-second exit agenda remains for 09-10.",
    )

    util.write_section(
        ctx["root"] / "docs/project_state.md",
        "notebook_05_state_v2",
        "## Notebook 05 — current state\n"
        "- Full-history labels: targets_05_v2, 130 months through October 16, 2026.\n"
        "- Codex reviewed saved checks, coverage and checkpoint/file metadata.\n"
        "- No reconciliation failures found; upstream label definitions unchanged.\n"
        "- Notebook 06 uses the reviewed outputs and an additional April 1 information cutoff.",
    )

    tg.write_json(
        ctx["reports"] / "execution_summary.json",
        summary,
    )
'''

In [9]:
# Check the assembled helper, then write it once.
import ast
import builtins
import symtable

ORDER = (
    "core",
    "inputs",
    "features",
    "models",
    "evaluation",
    "finish",
)
assert set(PARTS) == set(ORDER)

for name in ORDER:
    ast.parse(PARTS[name], filename=f"<{name}>")

MODULE_SOURCE = "\n\n".join(
    PARTS[name] for name in ORDER
)
tree = ast.parse(MODULE_SOURCE)

definitions = [
    n.name for n in tree.body
    if isinstance(n, ast.FunctionDef)
]
assert len(definitions) == len(set(definitions))

bound = set(definitions)

for node in tree.body:
    if isinstance(node, (ast.Import, ast.ImportFrom)):
        bound.update(
            a.asname or a.name.split(".")[0]
            for a in node.names
        )

    if isinstance(node, (ast.Assign, ast.AnnAssign)):
        targets_in_node = (
            node.targets
            if isinstance(node, ast.Assign)
            else [node.target]
        )
        for target in targets_in_node:
            bound.update(
                n.id for n in ast.walk(target)
                if isinstance(n, ast.Name)
            )

    if isinstance(node, ast.For):
        bound.update(
            n.id for n in ast.walk(node.target)
            if isinstance(n, ast.Name)
        )

allowed = bound | set(dir(builtins)) | {
    "__file__",
    "__name__",
    "__package__",
    "__spec__",
}
unresolved = []

def check_symbols(table):
    for symbol in table.get_symbols():
        if (
            symbol.is_global()
            and symbol.is_referenced()
            and not getattr(
                symbol, "is_comp_iter", lambda: False
            )()
            and symbol.get_name() not in allowed
        ):
            unresolved.append((
                table.get_name(),
                symbol.get_name(),
            ))

    for child in table.get_children():
        check_symbols(child)

check_symbols(symtable.symtable(
    MODULE_SOURCE,
    "<regression helper>",
    "exec",
))
assert not unresolved, unresolved

helper = (
    ROOT / "functions" / "regression"
    / "fx_regression.py"
)
compile(tree, str(helper), "exec")
helper.parent.mkdir(parents=True, exist_ok=True)

if (
    not helper.exists()
    or helper.read_text(encoding="utf-8") != MODULE_SOURCE
):
    temporary = helper.with_suffix(".py.tmp")
    temporary.write_text(
        MODULE_SOURCE, encoding="utf-8"
    )
    temporary.replace(helper)

importlib.invalidate_caches()
from functions.regression import fx_regression as regression
regression = importlib.reload(regression)

print("Helper syntax and names checked.")
print("Environment:", regression.versions())

Helper syntax and names checked.
Environment: {'python': '3.14.8', 'numpy': '2.5.4', 'pandas': '3.0.6', 'pyarrow': '26.0.0', 'duckdb': '2.0.0', 'matplotlib': '3.11.2', 'lightgbm': '4.7.0', 'scikit-learn': '1.9.1'}


In [10]:
# Verify input versions and save the agreed configuration.
CTX = regression.prepare(
    ROOT,
    RUN_ID,
    memory=MEMORY_LIMIT,
    threads=THREADS,
    working_gb=WORKING_MEMORY_BUDGET_GIB,
    maximum_trees=MAXIMUM_TREES,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

display(regression.pd.DataFrame([
    {
        "Feature group": group,
        "Numeric columns": len(columns),
    }
    for group, columns in regression.GROUPS.items()
]))

print("Input months to assemble:", len(CTX["months"]))
print("2026 is reserved and will not be modeled.")

,Feature group,Numeric columns
0,Price changes,6
1,Displayed sizes,2
2,Spread,1
3,MA distances,10
4,Sample entropy,5
5,Conditional entropy,5


Input months to assemble: 18
2026 is reserved and will not be modeled.


In [11]:
# Small synthetic checks before processing historical inputs.
CHECKS = regression.synthetic_checks()
assert CHECKS["passed"]

regression.tg.write_json(
    CTX["reports"] / "synthetic_checks.json",
    {
        "fingerprint": CTX["fingerprint"],
        **CHECKS,
    },
)

display(regression.pd.DataFrame(
    CHECKS["checks"].items(),
    columns=["Check", "Passed"],
))

,Check,Passed
0,publication_time_and_integer_precision,True
1,published_null_replaces_old_value,True
2,late_old_publication_does_not_replace_newer_ob...,True
3,price_changes_wait_for_known_anchors_and_fresh...,True
4,additional_april_information_cutoff,True
5,lightgbm_native_missing_early_stopping_and_ser...,True
6,ridge_training_only_imputation_interface,True
7,pip_to_price_conversion,True


In [12]:
# Expensive: assemble 2018 and January-June 2019, one month at a time.
# Verified completed months are reused.
COUNTS = regression.assemble(CTX)
MEMORY_REPORT = regression.memory_report(CTX, COUNTS)

display(COUNTS)
display(MEMORY_REPORT.round(2))

# The working-memory figure is an estimate, not a measured peak.

2018-01: 267,840 rows, 2.2s
2018-02: 241,920 rows, 1.9s
2018-03: 267,480 rows, 2.0s
2018-04: 259,200 rows, 1.9s
2018-05: 267,840 rows, 2.1s
2018-06: 259,200 rows, 2.0s
2018-07: 267,840 rows, 1.9s
2018-08: 267,840 rows, 2.1s
2018-09: 259,200 rows, 1.9s
2018-10: 268,200 rows, 2.0s
2018-11: 259,200 rows, 1.9s
2018-12: 267,840 rows, 1.9s
2019-01: 267,840 rows, 2.0s
2019-02: 241,920 rows, 1.8s
2019-03: 267,480 rows, 2.0s
2019-04: 259,200 rows, 2.0s
2019-05: 267,840 rows, 2.0s
2019-06: 259,200 rows, 1.9s


,Period,Horizon,Usable labels
0,train,30s,1960094
1,train,60s,1959311
2,early_stopping,30s,476052
3,early_stopping,60s,475860
4,comparison,30s,490845
5,comparison,60s,490650


,Horizon,Input arrays GiB,Estimated working GiB,Budget GiB
0,30s,0.28,1.69,3.0
1,60s,0.28,1.69,3.0


In [13]:
# Expensive: fit two LightGBM models and the ridge benchmarks.
# January-March chooses the tree count.
# April-June does not enter fitting.
BUNDLES, FIT_REPORT = regression.fit_models(CTX)

display(FIT_REPORT)

,Horizon,Training examples,Early-stopping examples,Trees retained
0,30s,1960094,476052,218
1,60s,1959311,475860,210


In [14]:
# Fixed-model comparison and the short forecast-price example.
PERFORMANCE, PRICE_EXAMPLES, SUMMARY = regression.evaluate(
    CTX,
    BUNDLES,
    example_start=EXAMPLE_START_ATHENS,
)

display(PERFORMANCE)

print(SUMMARY["example_date"], "| Athens local time")
display(PRICE_EXAMPLES)

print("* marks an outcome based on eligible interpolation.")
print(
    "Coverage = eligible forecasts scored; "
    "benchmarks use matching observations."
)

2019-04: comparison forecasts and checks saved.
2019-05: comparison forecasts and checks saved.
2019-06: comparison forecasts and checks saved.


,Horizon,Model / benchmark,MAE (pips),RMSE (pips),Coverage
0,30s,LightGBM,0.4374,0.7233,"490,845 (100%)"
1,30s,Zero change,0.4380,0.7234,"490,845 (100%)"
2,30s,Trailing change,0.6472,1.0297,"490,845 (100%)"
3,30s,Ridge,0.4382,0.7232,"490,845 (100%)"
4,60s,LightGBM,0.6301,1.0151,"490,650 (100%)"
5,60s,Zero change,0.6309,1.0152,"490,650 (100%)"
6,60s,Trailing change,0.9236,1.4485,"490,650 (100%)"
7,60s,Ridge,0.6309,1.0149,"490,650 (100%)"


Monday, 01 April 2019 | Athens local time


,Forecast time,Current mid,Predicted +30s,Outcome +30s,Predicted +60s,Outcome +60s
0,10:00:00,1.123395,1.123399,1.123275,1.123398,1.123360
1,10:00:10,1.123450,1.123450,1.123350,1.123450,1.123345
2,10:00:20,1.123445,1.123448,1.123350,1.123448,1.123340
3,10:00:30,1.123275,1.123277,1.123360,1.123280,1.123280


* marks an outcome based on eligible interpolation.
Coverage = eligible forecasts scored; benchmarks use matching observations.


In [15]:
# Record completed execution, agreed decisions and contributions.
# Human review and interpretation remain pending.
assert CHECKS["passed"]
assert SUMMARY["nonzero_machine_checks"] == 0
assert not SUMMARY["final_evaluation_accessed_for_modeling"]

regression.record(CTX, SUMMARY)

print("NOTEBOOK_06_EXECUTION_SUMMARY")
print("LightGBM models:", SUMMARY["lightgbm_models"])
print("Numeric inputs per model:", SUMMARY["numeric_features"])
print("Primary: MAE | Secondary: RMSE")
print("Comparison:", SUMMARY["comparison"])
print("Nonzero checks:", SUMMARY["nonzero_machine_checks"])
print("Reports:", CTX["reports"])
print("STOP HERE. Tell Codex: Notebook 06 outputs ready.")

NOTEBOOK_06_EXECUTION_SUMMARY
LightGBM models: 2
Numeric inputs per model: 29
Primary: MAE | Secondary: RMSE
Comparison: April-June 2019
Nonzero checks: 0
Reports: C:\Users\Ijzd4\Desktop\Inventory-Aware FX Pricing and Hedging Simulator\reports\tables\regression\regression_06_v1
STOP HERE. Tell Codex: Notebook 06 outputs ready.


In [17]:
# # Short-MA model variation

# - Keep the SMA 15 and EMA 15 distance inputs; remove the eight longer-MA distances.
# - Keep the other five groups and the original training settings. Each new model has 21 inputs.
# - Reuse all 12 months of 2018 for training and January–March 2019 for early stopping.
# - April–June is a development comparison. July–September is fresh validation.
# - Preserve the original models. Keep 2026 reserved.
# - The new model becomes available on July 1 in the research clock. April–June predictions are retrospective diagnostics.
# - This tests whether the longer-MA inputs hurt forecasting; it does not assume overfitting or improvement.

In [18]:
# Settings and read-only original-run checks.
from pathlib import Path
import copy
import gc
import hashlib
import json
import sys
import time

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow as pa

VAR_ROOT = Path.cwd().resolve()
if VAR_ROOT.name == "notebooks":
    VAR_ROOT = VAR_ROOT.parent

assert (
    VAR_ROOT / "outputs" / "features"
    / "regression_06_v1" / "configuration.json"
).exists(), (
    "Start Jupyter from the repository or its notebooks folder."
)

if str(VAR_ROOT) not in sys.path:
    sys.path.insert(0, str(VAR_ROOT))

from functions.regression import fx_regression as r

VAR_RUN_ID = "regression_06_short_ma_v1"
VAR_MA_PERIODS = (15,)
VAR_MEMORY = "2GB"
VAR_THREADS = 2
VAR_MEMORY_BUDGET_GIB = 3.0

VAR_FRESH_MONTHS = [
    "2019-07", "2019-08", "2019-09"
]
VAR_PHASES = {
    "Training": r.PERIODS["train"],
    "Development": r.PERIODS["comparison"],
    "Fresh validation": (
        r.tg.ns("2019-07-01"),
        r.tg.ns("2019-10-01"),
    ),
}
VAR_READY_NS = VAR_PHASES["Fresh validation"][0]
VAR_EXAMPLE_START = r.tg.ns("2019-07-01 10:00:00")

VAR_PARENT_FEATURES = (
    VAR_ROOT / "outputs" / "features" / "regression_06_v1"
)
VAR_PARENT_MODELS = (
    VAR_ROOT / "outputs" / "models" / "regression_06_v1"
)
VAR_PARENT_REPORTS = (
    VAR_ROOT / "reports" / "tables"
    / "regression" / "regression_06_v1"
)
VAR_PARENT_CONFIG_PATH = (
    VAR_PARENT_FEATURES / "configuration.json"
)

VAR_PARENT_CONFIG = r.tg.fx.load_json(VAR_PARENT_CONFIG_PATH)
assert VAR_RUN_ID != VAR_PARENT_CONFIG["run_id"]

VAR_PARENT_SUMMARY = r.tg.fx.load_json(
    VAR_PARENT_REPORTS / "execution_summary.json"
)
VAR_PARENT_FP = VAR_PARENT_SUMMARY["fingerprint"]

assert VAR_PARENT_FP == (
    "cef05f2e7a7f593b25c126cc8b06cb46e58872e273c34cb1b87062e937814c36"
)
assert VAR_PARENT_SUMMARY["nonzero_machine_checks"] == 0
assert VAR_PARENT_CONFIG["params"]["objective"] == "regression_l1"
assert {
    k: list(v) for k, v in r.PERIODS.items()
} == VAR_PARENT_CONFIG["periods_ns"]
assert r.GROUPS == VAR_PARENT_CONFIG["features"]
assert r.PERIODS["train"] == (
    r.tg.ns("2018-01-01"),
    r.tg.ns("2019-01-01"),
)

VAR_PARENT_CTX = {
    "root": VAR_ROOT,
    "feature_dir": VAR_PARENT_FEATURES,
    "model_dir": VAR_PARENT_MODELS,
    "reports": VAR_PARENT_REPORTS,
    "months": [
        str(m)
        for m in pd.period_range("2018-01", "2019-06", freq="M")
    ],
    "config": VAR_PARENT_CONFIG,
    "fingerprint": VAR_PARENT_FP,
    "memory": VAR_MEMORY,
    "threads": VAR_THREADS,
}

r.check_evidence(VAR_PARENT_CTX)
r.check_snapshot(VAR_PARENT_CTX)
r.check_features(VAR_PARENT_CTX)

VAR_KEEP_MA = [
    f"close_minus_{kind}_{n}_pips"
    for kind in ("sma", "ema")
    for n in VAR_MA_PERIODS
]
assert set(VAR_KEEP_MA).issubset(r.GROUPS["MA distances"])

VAR_COLUMNS = [
    c for c in r.FEATURES
    if c not in r.GROUPS["MA distances"]
    or c in VAR_KEEP_MA
]
VAR_INDICES = np.array([
    r.FEATURES.index(c) for c in VAR_COLUMNS
])

assert len(VAR_COLUMNS) == 21
assert len(set(VAR_COLUMNS)) == 21

VAR_GROUPS = copy.deepcopy(r.GROUPS)
VAR_GROUPS["MA distances"] = VAR_KEEP_MA

VAR_FULL_MODELS = {}
VAR_PARENT_ARTIFACTS = []

for var_h in ("30s", "60s"):
    var_meta_path = VAR_PARENT_MODELS / f"{var_h}_metadata.json"
    var_model_path = VAR_PARENT_MODELS / f"{var_h}_lightgbm.txt"

    var_meta = r.tg.fx.load_json(var_meta_path)
    assert var_meta["fingerprint"] == VAR_PARENT_FP
    assert var_meta["features"] == r.FEATURES
    assert r.util.sha(var_model_path) == var_meta["lightgbm_sha"]

    var_model = lgb.Booster(
        model_str=var_model_path.read_text(encoding="utf-8")
    )
    assert var_model.feature_name() == r.FEATURES

    VAR_FULL_MODELS[var_h] = {
        "model": var_model,
        "meta": var_meta,
    }

    for var_path in (var_meta_path, var_model_path):
        VAR_PARENT_ARTIFACTS.append({
            "path": var_path.relative_to(VAR_ROOT).as_posix(),
            "sha": r.util.sha(var_path),
        })

for var_month in VAR_PARENT_CTX["months"]:
    var_cp = (
        VAR_PARENT_FEATURES / "_checkpoints" / f"{var_month}.json"
    )
    VAR_PARENT_ARTIFACTS.append({
        "path": var_cp.relative_to(VAR_ROOT).as_posix(),
        "sha": r.util.sha(var_cp),
    })

var_path = VAR_PARENT_REPORTS / "performance_summary.csv"
VAR_PARENT_ARTIFACTS.append({
    "path": var_path.relative_to(VAR_ROOT).as_posix(),
    "sha": r.util.sha(var_path),
})

print("Training: January-December 2018 (12 months).")
print("Original inputs:", len(r.FEATURES))
print("Variation inputs:", len(VAR_COLUMNS))
print("Retained MA periods:", VAR_MA_PERIODS)

Training: January-December 2018 (12 months).
Original inputs: 29
Variation inputs: 21
Retained MA periods: (15,)


In [19]:
# Extend the verified inventory through September.
# This does not write to the original run.
VAR_ADDITIONS = []

var_sample = (
    VAR_ROOT / "data" / "processed" / r.SAMPLE_RUN
)
var_target = (
    VAR_ROOT / "data" / "processed" / r.TARGET_RUN
)
var_entropy = (
    VAR_ROOT / "outputs" / "features" / "entropy" / r.ENTROPY_RUN
)

var_sampling_config = r.tg.fx.load_json(
    var_sample / "configuration.json"
)
var_observed = {
    p["year_month"]: p
    for p in var_sampling_config["inputs"]
}

for var_month in VAR_FRESH_MONTHS:
    var_cp = var_sample / "_checkpoints" / f"{var_month}.json"
    var_saved = r.tg.fx.load_json(var_cp)

    assert var_saved["fingerprint"] == r.EXPECTED["sampling"]
    assert all(v == 0 for v in var_saved["checks"].values())

    for var_kind in ("sampled_10s", "candles_3min"):
        VAR_ADDITIONS.append((
            var_kind,
            var_sample / var_kind / f"{var_month}.parquet",
            var_saved["outputs"][var_kind],
            var_cp,
        ))

    var_raw = var_observed[var_month]
    var_cp = (
        VAR_ROOT / "data" / "processed" / "etl_01_v3"
        / "_checkpoints" / f"observed_{var_month}.json"
    )
    assert r.util.sha(var_cp) == var_raw["checkpoint_sha256"]

    VAR_ADDITIONS.append((
        "observed",
        VAR_ROOT / var_raw["path"],
        var_raw,
        var_cp,
    ))

for var_day in pd.date_range(
    "2019-07-01", "2019-09-30", freq="D"
):
    var_date = var_day.strftime("%Y-%m-%d")

    for var_kind, var_folder in (
        ("entropy", var_entropy),
        ("targets", var_target),
    ):
        var_cp = (
            var_folder / "_checkpoints" / f"{var_date}.json"
        )
        var_saved = r.tg.fx.load_json(var_cp)

        assert var_saved["fingerprint"] == r.EXPECTED[var_kind]
        assert all(v == 0 for v in var_saved["checks"].values())

        VAR_ADDITIONS.append((
            var_kind,
            var_folder / f"{var_date}.parquet",
            var_saved["output"],
            var_cp,
        ))

VAR_INPUTS = copy.deepcopy(VAR_PARENT_CONFIG["inputs"])

for var_kind, var_path, var_expected, var_cp in VAR_ADDITIONS:
    var_info = r.util.file_info(var_path)
    assert all(
        var_info[k] == var_expected[k]
        for k in var_info
    ), str(var_path)

    VAR_INPUTS.append({
        "kind": var_kind,
        "path": var_path.relative_to(VAR_ROOT).as_posix(),
        **var_info,
        "checkpoint": var_cp.relative_to(VAR_ROOT).as_posix(),
        "checkpoint_sha": r.util.sha(var_cp),
    })

assert len({
    (p["kind"], p["path"])
    for p in VAR_INPUTS
}) == len(VAR_INPUTS)

VAR_CONFIG = copy.deepcopy(VAR_PARENT_CONFIG)
VAR_CONFIG["stored_feature_groups"] = VAR_CONFIG.pop("features")

VAR_CONFIG.update({
    "definition": "short_ma_variation_v1",
    "run_id": VAR_RUN_ID,
    "parent_run_id": "regression_06_v1",
    "parent_fingerprint": VAR_PARENT_FP,
    "parent_config_sha": r.util.sha(VAR_PARENT_CONFIG_PATH),
    "parent_artifacts": VAR_PARENT_ARTIFACTS,
    "inputs": VAR_INPUTS,
    "model_feature_groups": VAR_GROUPS,
    "model_features": VAR_COLUMNS,
    "retained_ma_periods": list(VAR_MA_PERIODS),
    "evaluation_periods_ns": VAR_PHASES,
    "architecture_information_cutoff_ns": VAR_READY_NS - 1,
    "model_available_ns": VAR_READY_NS,
    "comparison_scope": (
        "April-June development; July-September fresh validation"
    ),
    "development_forecasts": (
        "retrospective diagnostics; not exported for replay"
    ),
})

VAR_CONFIG = json.loads(json.dumps(VAR_CONFIG))
VAR_FP = hashlib.sha256(json.dumps(
    VAR_CONFIG,
    sort_keys=True,
    separators=(",", ":"),
).encode()).hexdigest()

VAR_FEATURE_DIR = (
    VAR_ROOT / "outputs" / "features" / VAR_RUN_ID
)
VAR_MODEL_DIR = (
    VAR_ROOT / "outputs" / "models" / VAR_RUN_ID
)
VAR_FORECAST_DIR = (
    VAR_ROOT / "outputs" / "forecasts" / VAR_RUN_ID
)
VAR_REPORTS = (
    VAR_ROOT / "reports" / "tables" / "regression" / VAR_RUN_ID
)

for var_folder in (
    VAR_FEATURE_DIR,
    VAR_FEATURE_DIR / "_checkpoints",
    VAR_MODEL_DIR,
    VAR_FORECAST_DIR / "original_29",
    VAR_FORECAST_DIR / "short_ma_21",
    VAR_REPORTS,
):
    var_folder.mkdir(parents=True, exist_ok=True)

var_config_path = VAR_FEATURE_DIR / "configuration.json"

if var_config_path.exists():
    assert r.tg.fx.load_json(var_config_path) == VAR_CONFIG, (
        "Variation configuration changed; choose a new VAR_RUN_ID."
    )
else:
    r.tg.write_json(var_config_path, VAR_CONFIG)

VAR_CTX = {
    "root": VAR_ROOT,
    "config": VAR_CONFIG,
    "fingerprint": VAR_FP,
    "feature_dir": VAR_FEATURE_DIR,
    "months": VAR_FRESH_MONTHS,
    "memory": VAR_MEMORY,
    "threads": VAR_THREADS,
}

r.check_snapshot(VAR_CTX)

r.tg.write_csv(
    pd.DataFrame(VAR_INPUTS),
    VAR_REPORTS / "input_inventory.csv",
)
r.tg.write_json(
    VAR_REPORTS / "model_feature_groups.json",
    VAR_GROUPS,
)

print("Fresh feature months:", ", ".join(VAR_FRESH_MONTHS))

Fresh feature months: 2019-07, 2019-08, 2019-09


In [20]:
# Build July-September inputs for both model versions.
# Stored tables keep all 29 columns; only the new models select 21.
# Expensive: three monthly batches. Completed batches are reused.
VAR_BUILD_CHECKS = []
VAR_FRESH_COUNTS = []

with duckdb.connect(config={
    "memory_limit": VAR_MEMORY,
    "threads": str(VAR_THREADS),
}) as var_con:
    for var_month in VAR_FRESH_MONTHS:
        var_path = VAR_FEATURE_DIR / f"{var_month}.parquet"
        var_cp = (
            VAR_FEATURE_DIR / "_checkpoints" / f"{var_month}.json"
        )

        if var_cp.exists():
            var_saved = r.tg.fx.load_json(var_cp)

            assert var_saved["fingerprint"] == VAR_FP
            assert r.util.file_info(var_path) == var_saved["output"]
            assert all(
                v == 0 for v in var_saved["checks"].values()
            )

        else:
            var_started = time.perf_counter()
            var_table, var_checks, _, _ = r.make_month(
                VAR_CTX, var_month, var_con
            )
            var_times = r.tg.get(var_table, "issue_ns")
            var_counts = {
                "month": var_month,
                "rows": var_table.num_rows,
            }

            for var_h in ("30s", "60s"):
                var_use = (
                    (var_times >= VAR_PHASES["Fresh validation"][0])
                    & (var_times < VAR_PHASES["Fresh validation"][1])
                    & r.tg.get(var_table, f"usable_{var_h}")
                    & (
                        r.tg.get(
                            var_table,
                            f"label_available_ns_{var_h}",
                            0,
                        )
                        < VAR_PHASES["Fresh validation"][1]
                    )
                )
                var_counts[f"fresh_{var_h}"] = int(var_use.sum())

            r.tg.write_parquet(var_table, var_path)

            var_saved = {
                "fingerprint": VAR_FP,
                "checks": var_checks,
                "counts": var_counts,
                "output": r.util.file_info(var_path),
            }
            r.tg.write_json(var_cp, var_saved)

            print(
                f"{var_month}: inputs saved in "
                f"{time.perf_counter() - var_started:.1f}s"
            )
            del var_table

        VAR_BUILD_CHECKS.append({
            "month": var_month,
            **var_saved["checks"],
        })
        VAR_FRESH_COUNTS.append(var_saved["counts"])

r.check_snapshot(VAR_CTX)
r.check_features(VAR_CTX)

r.tg.write_csv(
    pd.DataFrame(VAR_BUILD_CHECKS),
    VAR_REPORTS / "feature_checks.csv",
)
r.tg.write_csv(
    pd.DataFrame(VAR_FRESH_COUNTS),
    VAR_REPORTS / "fresh_label_counts.csv",
)

var_parent_counts = [
    r.tg.fx.load_json(
        VAR_PARENT_FEATURES / "_checkpoints" / f"{m}.json"
    )["counts"]
    for m in VAR_PARENT_CTX["months"]
]

VAR_MEMORY_ROWS = []

for var_h in ("30s", "60s"):
    var_n = sum(
        c[f"train_{var_h}"] + c[f"early_stopping_{var_h}"]
        for c in var_parent_counts
    )
    var_estimate = (
        6 * var_n * (4 * len(r.FEATURES) + 8) / 1024**3
    )
    VAR_MEMORY_ROWS.append({
        "Horizon": var_h,
        "Estimated working GiB": var_estimate,
        "Budget GiB": VAR_MEMORY_BUDGET_GIB,
    })

VAR_MEMORY_REPORT = pd.DataFrame(VAR_MEMORY_ROWS)
display(VAR_MEMORY_REPORT.round(2))

assert VAR_MEMORY_REPORT["Estimated working GiB"].le(
    VAR_MEMORY_BUDGET_GIB
).all(), (
    "Estimated working memory exceeds the variation settings budget."
)

r.tg.write_csv(
    VAR_MEMORY_REPORT,
    VAR_REPORTS / "memory_estimate.csv",
)

2019-07: inputs saved in 2.3s
2019-08: inputs saved in 2.3s
2019-09: inputs saved in 2.6s


,Horizon,Estimated working GiB,Budget GiB
0,30s,1.69,3.0
1,60s,1.69,3.0


In [21]:
# Retrain only the two short-MA models.
# Training data, parameters, seed and stopping rules match the original.
# Expensive: two fits. Verified saved models are reused.
assert VAR_MEMORY_REPORT["Estimated working GiB"].le(
    VAR_MEMORY_BUDGET_GIB
).all()

r.check_snapshot(VAR_CTX)
r.check_features(VAR_PARENT_CTX)
r.check_features(VAR_CTX)

assert r.util.sha(
    VAR_PARENT_CONFIG_PATH
) == VAR_CONFIG["parent_config_sha"]

for var_item in VAR_PARENT_ARTIFACTS:
    assert r.util.sha(
        VAR_ROOT / var_item["path"]
    ) == var_item["sha"]

VAR_SHORT_MODELS = {}
VAR_FIT_ROWS = []

for var_h in ("30s", "60s"):
    var_meta_path = VAR_MODEL_DIR / f"{var_h}_metadata.json"
    var_model_path = VAR_MODEL_DIR / f"{var_h}_lightgbm.txt"

    if var_meta_path.exists():
        var_meta = r.tg.fx.load_json(var_meta_path)

        assert var_meta["fingerprint"] == VAR_FP
        assert var_meta["features"] == VAR_COLUMNS
        assert r.util.sha(var_model_path) == var_meta["model_sha"]

        var_model = lgb.Booster(
            model_str=var_model_path.read_text(encoding="utf-8")
        )

    else:
        var_started = time.perf_counter()

        var_full_x, var_y, var_fit_info = r.load_role(
            VAR_PARENT_CTX, "train", var_h
        )
        var_x = np.ascontiguousarray(
            var_full_x[:, VAR_INDICES]
        )
        del var_full_x

        var_full_stop, var_stop_y, var_stop_info = r.load_role(
            VAR_PARENT_CTX, "early_stopping", var_h
        )
        var_stop_x = np.ascontiguousarray(
            var_full_stop[:, VAR_INDICES]
        )
        del var_full_stop

        assert len(var_y) == (
            VAR_FULL_MODELS[var_h]["meta"]["training_rows"]
        )
        assert len(var_stop_y) == (
            VAR_FULL_MODELS[var_h]["meta"]["early_stopping_rows"]
        )
        assert var_x.shape == (
            len(var_y), len(VAR_COLUMNS)
        )
        assert var_stop_x.shape == (
            len(var_stop_y), len(VAR_COLUMNS)
        )
        assert np.isfinite(var_x).any(axis=0).all()

        var_train_data = lgb.Dataset(
            var_x,
            label=var_y,
            feature_name=VAR_COLUMNS,
        )
        var_stop_data = lgb.Dataset(
            var_stop_x,
            label=var_stop_y,
            reference=var_train_data,
            feature_name=VAR_COLUMNS,
        )

        var_model = lgb.train(
            copy.deepcopy(VAR_PARENT_CONFIG["params"]),
            var_train_data,
            num_boost_round=VAR_PARENT_CONFIG["maximum_trees"],
            valid_sets=[var_stop_data],
            valid_names=["early_stopping"],
            callbacks=[
                lgb.early_stopping(
                    VAR_PARENT_CONFIG["early_stopping_rounds"],
                    first_metric_only=True,
                    verbose=False,
                ),
                lgb.log_evaluation(0),
            ],
        )
        assert var_model.best_iteration >= 1

        var_text = var_model.model_to_string(
            num_iteration=var_model.best_iteration
        )
        var_restored = lgb.Booster(model_str=var_text)

        assert np.allclose(
            var_model.predict(var_stop_x[:1000]),
            var_restored.predict(var_stop_x[:1000]),
            rtol=0,
            atol=1e-10,
        )

        r.save_text(var_model_path, var_text)

        var_meta = {
            "fingerprint": VAR_FP,
            "parent_fingerprint": VAR_PARENT_FP,
            "model_identifier": f"{VAR_RUN_ID}_{var_h}",
            "horizon": var_h,
            "features": VAR_COLUMNS,
            "training_period": "January-December 2018",
            "training_months": 12,
            "training_rows": len(var_y),
            "early_stopping_rows": len(var_stop_y),
            "best_iteration": int(var_model.best_iteration),
            "fit_information_cutoff_ns": var_fit_info,
            "early_stopping_information_cutoff_ns": var_stop_info,
            "feature_choice_information_cutoff_ns": VAR_READY_NS - 1,
            "training_cutoff_ns": max(
                var_fit_info,
                var_stop_info,
                VAR_READY_NS - 1,
            ),
            "model_available_ns": VAR_READY_NS,
            "inference_latency_ns": 0,
            "cutoff_convention": "latest included information time",
            "estimate": "conditional median point forecast",
            "development_predictions": (
                "retrospective diagnostics only"
            ),
            "model_sha": r.util.sha(var_model_path),
            "fit_minutes": (
                time.perf_counter() - var_started
            ) / 60,
        }
        r.tg.write_json(var_meta_path, var_meta)

        del (
            var_x,
            var_y,
            var_stop_x,
            var_stop_y,
            var_train_data,
            var_stop_data,
            var_restored,
        )
        gc.collect()

    assert var_model.feature_name() == VAR_COLUMNS
    assert (
        var_meta["training_cutoff_ns"]
        < var_meta["model_available_ns"]
    )

    VAR_SHORT_MODELS[var_h] = {
        "model": var_model,
        "meta": var_meta,
    }
    VAR_FIT_ROWS.append({
        "Horizon": var_h,
        "Inputs": len(VAR_COLUMNS),
        "Training months": 12,
        "Training examples": var_meta["training_rows"],
        "Stopping examples": var_meta["early_stopping_rows"],
        "Original trees": (
            VAR_FULL_MODELS[var_h]["meta"]["best_iteration"]
        ),
        "Short-MA trees": var_meta["best_iteration"],
    })

r.check_snapshot(VAR_CTX)
r.check_features(VAR_PARENT_CTX)

r.tg.write_csv(
    pd.DataFrame(VAR_FIT_ROWS),
    VAR_REPORTS / "model_fit_summary.csv",
)
display(pd.DataFrame(VAR_FIT_ROWS))

,Horizon,Inputs,Training months,Training examples,Stopping examples,Original trees,Short-MA trees
0,30s,21,12,1960094,476052,218,385
1,60s,21,12,1959311,475860,210,209


In [22]:
# Score training, development and fresh validation.
# Training/development scores are retrospective diagnostics.
# Only July-September forecasts are exported for market replay.
# Expensive: evaluate the two fixed versions over 18 monthly batches.
VAR_PLANS = [
    (
        "Training",
        str(m),
        VAR_PARENT_FEATURES / f"{m}.parquet",
    )
    for m in pd.period_range("2018-01", "2018-12", freq="M")
] + [
    (
        "Development",
        m,
        VAR_PARENT_FEATURES / f"{m}.parquet",
    )
    for m in ("2019-04", "2019-05", "2019-06")
] + [
    (
        "Fresh validation",
        m,
        VAR_FEATURE_DIR / f"{m}.parquet",
    )
    for m in VAR_FRESH_MONTHS
]

VAR_SCORES = []
VAR_EVALUATION_CHECKS = []
VAR_PRICE_EXAMPLES = None
VAR_EXAMPLE_DATE = None

var_required = r.tg.fx.load_json(
    VAR_ROOT / "data" / "processed" / r.TARGET_RUN
    / "forecast_output_contract.json"
)["columns"]

with duckdb.connect(config={
    "memory_limit": VAR_MEMORY,
    "threads": str(VAR_THREADS),
}) as var_con:
    for var_phase, var_month, var_path in VAR_PLANS:
        var_table = r.read(
            var_con,
            [var_path],
            [
                *r.BASE,
                *r.FEATURES,
                "input_information_max_ns",
            ],
            "issue_ns",
        )

        var_times = r.tg.get(var_table, "issue_ns")
        var_reference = r.tg.get(var_table, "reference_mid")
        var_x29 = r.matrix(var_table)
        var_x21 = np.ascontiguousarray(
            var_x29[:, VAR_INDICES]
        )
        var_lower, var_upper = VAR_PHASES[var_phase]

        var_checks = {
            "period": var_phase,
            "month": var_month,
            "late_inputs": int(np.count_nonzero(
                r.tg.get(var_table, "input_information_max_ns")
                > var_times
            )),
        }

        var_payloads = {
            name: {
                c: var_table[c]
                for c in (
                    "issue_ns",
                    "reference_quote_ns",
                    "reference_mid",
                    "reference_age_ns",
                )
            }
            for name in ("original_29", "short_ma_21")
        }
        var_prices = {}
        var_masks = {}

        for var_h in ("30s", "60s"):
            var_full_prediction = (
                VAR_FULL_MODELS[var_h]["model"].predict(var_x29)
            )
            var_short_prediction = (
                VAR_SHORT_MODELS[var_h]["model"].predict(var_x21)
            )

            var_predictions = {
                "Original 29": var_full_prediction,
                "Short MA 21": var_short_prediction,
                "Zero change": np.zeros(len(var_times)),
            }
            assert all(
                np.isfinite(v).all()
                for v in var_predictions.values()
            )

            var_use = (
                (var_times >= var_lower)
                & (var_times < var_upper)
                & r.tg.get(var_table, f"usable_{var_h}")
                & (
                    r.tg.get(
                        var_table,
                        f"label_available_ns_{var_h}",
                        0,
                    ) < var_upper
                )
            )

            var_actual = r.tg.get(
                var_table, f"target_change_{var_h}_pips", np.nan
            )
            var_outcome = r.tg.get(
                var_table, f"target_future_mid_{var_h}", np.nan
            )

            var_n = int(var_use.sum())
            assert var_n > 0
            assert np.isfinite(var_actual[var_use]).all()
            assert np.isfinite(var_outcome[var_use]).all()

            var_checks[var_h + "_outcome_conversion"] = int(
                np.count_nonzero(
                    var_use
                    & (
                        np.abs(
                            r.tg.predicted_mid(
                                var_reference, var_actual
                            ) - var_outcome
                        ) > 1e-12
                    )
                )
            )

            for var_name, var_prediction in var_predictions.items():
                var_error = (
                    var_prediction[var_use] - var_actual[var_use]
                )
                VAR_SCORES.append({
                    "period": var_phase,
                    "month": var_month,
                    "horizon": var_h,
                    "model": var_name,
                    "count": var_n,
                    "absolute_error_sum": float(
                        np.abs(var_error).sum()
                    ),
                    "squared_error_sum": float(
                        np.square(var_error).sum()
                    ),
                })

            var_full_price = r.tg.predicted_mid(
                var_reference, var_full_prediction
            )
            var_short_price = r.tg.predicted_mid(
                var_reference, var_short_prediction
            )

            var_checks[var_h + "_nonpositive_price"] = int(
                np.count_nonzero(
                    (var_full_price <= 0)
                    | (var_short_price <= 0)
                )
            )

            var_prices[var_h] = (
                var_full_price,
                var_short_price,
                var_outcome,
            )
            var_masks[var_h] = var_use

            if var_phase == "Fresh validation":
                for (
                    var_name,
                    var_bundle,
                    var_prediction,
                    var_price,
                ) in (
                    (
                        "original_29",
                        VAR_FULL_MODELS[var_h],
                        var_full_prediction,
                        var_full_price,
                    ),
                    (
                        "short_ma_21",
                        VAR_SHORT_MODELS[var_h],
                        var_short_prediction,
                        var_short_price,
                    ),
                ):
                    var_meta = var_bundle["meta"]

                    var_checks[
                        f"{var_h}_{var_name}_model_from_future"
                    ] = int(np.count_nonzero(
                        (
                            var_times
                            < var_meta["model_available_ns"]
                        )
                        | (
                            var_times
                            <= var_meta["training_cutoff_ns"]
                        )
                    ))

                    var_payloads[var_name].update({
                        f"predicted_change_{var_h}_pips":
                            pa.array(var_prediction),
                        f"expected_eurusd_rate_{var_h}":
                            pa.array(var_price),
                        f"forecast_available_ns_{var_h}":
                            pa.array(
                                var_times
                                + var_meta["inference_latency_ns"]
                            ),
                        f"model_identifier_{var_h}":
                            pa.array(
                                [var_meta["model_identifier"]]
                                * len(var_times)
                            ),
                        f"training_cutoff_ns_{var_h}":
                            pa.array(np.full(
                                len(var_times),
                                var_meta["training_cutoff_ns"],
                                dtype=np.int64,
                            )),
                    })

        assert all(
            v == 0
            for k, v in var_checks.items()
            if k not in ("period", "month")
        ), var_checks

        VAR_EVALUATION_CHECKS.append(var_checks)

        if var_phase == "Fresh validation":
            for var_name, var_payload in var_payloads.items():
                assert set(var_required).issubset(var_payload)

                var_forecast_table = pa.table(var_payload)
                assert (
                    var_forecast_table.num_rows
                    == var_table.num_rows
                )

                r.tg.write_parquet(
                    var_forecast_table,
                    VAR_FORECAST_DIR / var_name
                    / f"{var_month}.parquet",
                )

            if VAR_PRICE_EXAMPLES is None:
                var_ok = (
                    var_masks["30s"]
                    & var_masks["60s"]
                    & (var_times >= VAR_EXAMPLE_START)
                )
                var_runs = np.flatnonzero(
                    np.convolve(
                        var_ok.astype(int),
                        np.ones(4, dtype=int),
                        mode="valid",
                    ) == 4
                )

                if len(var_runs):
                    var_index = np.arange(
                        int(var_runs[0]),
                        int(var_runs[0]) + 4,
                    )
                    var_display_time = pd.to_datetime(
                        var_times[var_index],
                        unit="ns",
                        utc=True,
                    ).tz_convert(r.TZ)

                    var_example = {
                        "Time": var_display_time.strftime("%H:%M:%S"),
                        "Current mid": [
                            f"{v:.6f}"
                            for v in var_reference[var_index]
                        ],
                    }

                    for var_h in ("30s", "60s"):
                        (
                            var_full_price,
                            var_short_price,
                            var_outcome,
                        ) = var_prices[var_h]

                        var_synthetic = r.tg.get(
                            var_table,
                            f"endpoint_is_interpolated_{var_h}",
                            False,
                        )

                        var_example[f"Original +{var_h}"] = [
                            f"{v:.6f}"
                            for v in var_full_price[var_index]
                        ]
                        var_example[f"Short MA +{var_h}"] = [
                            f"{v:.6f}"
                            for v in var_short_price[var_index]
                        ]
                        var_example[f"Outcome +{var_h}"] = [
                            f"{var_outcome[i]:.6f}"
                            + ("*" if var_synthetic[i] else "")
                            for i in var_index
                        ]

                    VAR_PRICE_EXAMPLES = pd.DataFrame(var_example)
                    VAR_EXAMPLE_DATE = (
                        var_display_time[0].strftime(
                            "%A, %d %B %Y"
                        )
                    )

        print(
            f"{var_phase} | {var_month}: "
            "scores saved in memory."
        )
        del var_table, var_x29, var_x21

r.check_snapshot(VAR_CTX)
r.check_features(VAR_CTX)
r.check_features(VAR_PARENT_CTX)

assert r.util.sha(
    VAR_PARENT_CONFIG_PATH
) == VAR_CONFIG["parent_config_sha"]

for var_item in VAR_PARENT_ARTIFACTS:
    assert r.util.sha(
        VAR_ROOT / var_item["path"]
    ) == var_item["sha"]

assert VAR_PRICE_EXAMPLES is not None

Training | 2018-01: scores saved in memory.
Training | 2018-02: scores saved in memory.
Training | 2018-03: scores saved in memory.
Training | 2018-04: scores saved in memory.
Training | 2018-05: scores saved in memory.
Training | 2018-06: scores saved in memory.
Training | 2018-07: scores saved in memory.
Training | 2018-08: scores saved in memory.
Training | 2018-09: scores saved in memory.
Training | 2018-10: scores saved in memory.
Training | 2018-11: scores saved in memory.
Training | 2018-12: scores saved in memory.
Development | 2019-04: scores saved in memory.
Development | 2019-05: scores saved in memory.
Development | 2019-06: scores saved in memory.
Fresh validation | 2019-07: scores saved in memory.
Fresh validation | 2019-08: scores saved in memory.
Fresh validation | 2019-09: scores saved in memory.


In [23]:
# Concise paired results.
VAR_MONTHLY = pd.DataFrame(VAR_SCORES)

VAR_TOTALS = (
    VAR_MONTHLY.groupby(
        ["period", "horizon", "model"],
        sort=False,
    )[
        ["count", "absolute_error_sum", "squared_error_sum"]
    ]
    .sum()
    .reset_index()
)

for var_frame in (VAR_MONTHLY, VAR_TOTALS):
    var_frame["mae_pips"] = (
        var_frame["absolute_error_sum"] / var_frame["count"]
    )
    var_frame["rmse_pips"] = np.sqrt(
        var_frame["squared_error_sum"] / var_frame["count"]
    )

assert VAR_TOTALS.groupby(
    ["period", "horizon"]
)["count"].nunique().eq(1).all()

assert VAR_TOTALS["mae_pips"].le(
    VAR_TOTALS["rmse_pips"] + 1e-10
).all()

# Reproduce the original April-June scores.
var_baseline = pd.read_csv(
    VAR_PARENT_REPORTS / "performance_summary.csv"
)
VAR_BASELINE_CHECKS = []

for var_h in ("30s", "60s"):
    var_old = var_baseline.loc[
        var_baseline["horizon"].eq(var_h)
        & var_baseline["model"].eq("LightGBM")
    ]
    var_again = VAR_TOTALS.loc[
        VAR_TOTALS["period"].eq("Development")
        & VAR_TOTALS["horizon"].eq(var_h)
        & VAR_TOTALS["model"].eq("Original 29")
    ]

    assert len(var_old) == len(var_again) == 1
    var_old = var_old.iloc[0]
    var_again = var_again.iloc[0]

    var_check = {
        "horizon": var_h,
        "count_match": (
            int(var_old["count"])
            == int(var_again["count"])
        ),
        "mae_match": bool(np.isclose(
            var_old["mae_pips"],
            var_again["mae_pips"],
            rtol=0,
            atol=1e-9,
        )),
        "rmse_match": bool(np.isclose(
            var_old["rmse_pips"],
            var_again["rmse_pips"],
            rtol=0,
            atol=1e-9,
        )),
    }

    assert all(
        v for k, v in var_check.items()
        if k != "horizon"
    ), var_check

    VAR_BASELINE_CHECKS.append(var_check)

r.tg.write_csv(
    pd.DataFrame(VAR_BASELINE_CHECKS),
    VAR_REPORTS / "baseline_reconciliation.csv",
)

VAR_MAE_ROWS = []
VAR_RMSE_ROWS = []
VAR_RELATIVE_ROWS = []

for var_phase in VAR_PHASES:
    for var_h in ("30s", "60s"):
        var_scope = VAR_TOTALS.loc[
            VAR_TOTALS["period"].eq(var_phase)
            & VAR_TOTALS["horizon"].eq(var_h)
        ].set_index("model")

        var_full_mae = float(
            var_scope.loc["Original 29", "mae_pips"]
        )
        var_short_mae = float(
            var_scope.loc["Short MA 21", "mae_pips"]
        )
        var_zero_mae = float(
            var_scope.loc["Zero change", "mae_pips"]
        )

        var_gain = (
            100 * (var_full_mae - var_short_mae)
            / var_full_mae
            if var_full_mae > 0 else np.nan
        )

        VAR_MAE_ROWS.append({
            "Period": var_phase,
            "Horizon": var_h,
            "Zero MAE": var_zero_mae,
            "Original MAE": var_full_mae,
            "Short-MA MAE": var_short_mae,
            "Short vs original (%)": var_gain,
        })

        VAR_RMSE_ROWS.append({
            "Period": var_phase,
            "Horizon": var_h,
            "Zero RMSE": float(
                var_scope.loc["Zero change", "rmse_pips"]
            ),
            "Original RMSE": float(
                var_scope.loc["Original 29", "rmse_pips"]
            ),
            "Short-MA RMSE": float(
                var_scope.loc["Short MA 21", "rmse_pips"]
            ),
        })

        VAR_RELATIVE_ROWS.append({
            "Period": var_phase,
            "Horizon": var_h,
            "Forecasts": int(
                var_scope.loc["Original 29", "count"]
            ),
            "Original MAE improvement vs zero (%)": (
                100 * (1 - var_full_mae / var_zero_mae)
                if var_zero_mae > 0 else np.nan
            ),
            "Short-MA MAE improvement vs zero (%)": (
                100 * (1 - var_short_mae / var_zero_mae)
                if var_zero_mae > 0 else np.nan
            ),
        })

VAR_MAE_TABLE = pd.DataFrame(VAR_MAE_ROWS)
VAR_RMSE_TABLE = pd.DataFrame(VAR_RMSE_ROWS)
VAR_RELATIVE_TABLE = pd.DataFrame(VAR_RELATIVE_ROWS)

r.tg.write_csv(
    VAR_MONTHLY, VAR_REPORTS / "monthly_performance.csv"
)
r.tg.write_csv(
    VAR_TOTALS, VAR_REPORTS / "performance_summary.csv"
)
r.tg.write_csv(
    VAR_MAE_TABLE, VAR_REPORTS / "mae_comparison.csv"
)
r.tg.write_csv(
    VAR_RMSE_TABLE, VAR_REPORTS / "rmse_comparison.csv"
)
r.tg.write_csv(
    VAR_RELATIVE_TABLE,
    VAR_REPORTS / "baseline_relative_performance.csv",
)
r.tg.write_csv(
    pd.DataFrame(VAR_EVALUATION_CHECKS),
    VAR_REPORTS / "evaluation_checks.csv",
)
r.tg.write_csv(
    VAR_PRICE_EXAMPLES,
    VAR_REPORTS / "forecast_price_examples.csv",
)

print(
    "MAE in pips — primary. "
    "Positive % means the short-MA version improved."
)
display(VAR_MAE_TABLE.round({
    "Zero MAE": 6,
    "Original MAE": 6,
    "Short-MA MAE": 6,
    "Short vs original (%)": 3,
}))

print("RMSE in pips — secondary.")
display(VAR_RMSE_TABLE.round(6))

print(VAR_EXAMPLE_DATE, "| Athens time")
display(VAR_PRICE_EXAMPLES)

print(
    "* marks an interpolated outcome. "
    "All comparisons use matching observations."
)
print(
    "Training is in-sample; April-June is development; "
    "July-September is fresh validation."
)

MAE in pips — primary. Positive % means the short-MA version improved.


,Period,Horizon,Zero MAE,Original MAE,Short-MA MAE,Short vs original (%)
0,Training,30s,0.720984,0.719777,0.719681,0.013
1,Training,60s,1.037208,1.034797,1.035009,-0.020
2,Development,30s,0.437998,0.437392,0.437384,0.002
3,Development,60s,0.630940,0.630148,0.630117,0.005
4,Fresh validation,30s,0.464081,0.463547,0.463524,0.005
5,Fresh validation,60s,0.665815,0.665305,0.665214,0.014


RMSE in pips — secondary.


,Period,Horizon,Zero RMSE,Original RMSE,Short-MA RMSE
0,Training,30s,1.104054,1.103682,1.103579
1,Training,60s,1.553466,1.552584,1.552653
2,Development,30s,0.723397,0.723253,0.723305
3,Development,60s,1.015175,1.015050,1.015023
4,Fresh validation,30s,0.763791,0.763886,0.763894
5,Fresh validation,60s,1.070910,1.071409,1.071294


Monday, 01 July 2019 | Athens time


,Time,Current mid,Original +30s,Short MA +30s,Outcome +30s,Original +60s,Short MA +60s,Outcome +60s
0,10:00:00,1.132705,1.132710,1.132711,1.132595,1.132709,1.132707,1.132610
1,10:00:10,1.132520,1.132523,1.132523,1.132565,1.132523,1.132523,1.132835
2,10:00:20,1.132555,1.132556,1.132556,1.132585,1.132562,1.132559,1.132830
3,10:00:30,1.132595,1.132597,1.132597,1.132610,1.132598,1.132597,1.132955


* marks an interpolated outcome. All comparisons use matching observations.
Training is in-sample; April-June is development; July-September is fresh validation.


In [24]:
# Record the variation and execution scope.
# This records execution, not an improvement or overfitting conclusion.
assert all(
    v == 0
    for row in VAR_BUILD_CHECKS
    for k, v in row.items()
    if k != "month"
)
assert all(
    v == 0
    for row in VAR_EVALUATION_CHECKS
    for k, v in row.items()
    if k not in ("period", "month")
)

VAR_SUMMARY = {
    "notebook": "06_gradient_boosted_regression.ipynb",
    "section": "short_ma_variation",
    "run_id": VAR_RUN_ID,
    "fingerprint": VAR_FP,
    "parent_run_id": "regression_06_v1",
    "parent_fingerprint": VAR_PARENT_FP,
    "original_numeric_inputs": 29,
    "reduced_numeric_inputs": len(VAR_COLUMNS),
    "retained_ma_periods": list(VAR_MA_PERIODS),
    "stored_numeric_columns": len(r.FEATURES),
    "new_lightgbm_models": 2,
    "original_models_refitted": False,
    "objective": "regression_l1",
    "primary_metric": "MAE in pips",
    "secondary_metric": "RMSE in pips",
    "training": "2018",
    "early_stopping": "January-March 2019",
    "development_comparison": "April-June 2019; retrospective",
    "fresh_validation": "July-September 2019",
    "fresh_label_information_end_exclusive_ns": (
        VAR_PHASES["Fresh validation"][1]
    ),
    "model_available_ns": VAR_READY_NS,
    "model_readiness": (
        "research-clock assumption; zero modeled inference latency"
    ),
    "replay_forecasts_exported": (
        "July-September only; all issue times"
    ),
    "original_artifacts_unchanged": True,
    "original_development_scores_reproduced": True,
    "nonzero_machine_checks": 0,
    "final_2026_used_for_modeling": False,
    "statistical_significance_tested": False,
    "overfitting_conclusion": "pending results review",
    "human_review": "pending",
}

r.tg.write_json(
    VAR_REPORTS / "execution_summary.json",
    VAR_SUMMARY,
)

var_decision = """
## Short-MA input variation
- Raymond Fendi requested testing the shorter MA input combination after weak baseline gains.
- Keep SMA 15 and EMA 15 distance inputs; remove the eight longer-MA distances.
- Other feature groups, labels, 2018 training and January-March stopping rules remain unchanged.
- Retrain only the two reduced models, with the original parameters and budget.
- April-June is development evidence; July-September is fresh validation.
- Require fresh-validation label information before October 1, 2019.
- The reduced model's information cutoff includes the April-June-informed input choice.
- Its research-clock availability begins July 1; April-June predictions are retrospective.
- Keep original models and upstream data; use separate versioned result files.
- Compare matched observations using MAE primary and RMSE secondary.
- Fewer inputs do not by themselves establish better forecasting or prove overfitting.
- No significance test or automatic model promotion is performed.
- Further changes informed by July-September require a later fresh development block.
- Keep 2026 reserved.
"""

var_contribution = """
## Short-MA variation
- Raymond Fendi proposed input experiments to improve forecasting and selected the shortest MA pair.
- Agreed July-September 2019 as the fresh validation period.
- Requested keeping the experiment in Notebook 06 with an explanatory Markdown cell.
- OpenAI Codex verified the shortest pair is SMA 15 / EMA 15 and supplied the appended cells.
- This executed variation retrained two models and produced matched comparison reports.
- Interpretation, improvement and any overfitting conclusion remain pending review.
"""

var_state = f"""
## Current model-input experiment
- Notebook: 06_gradient_boosted_regression.ipynb, appended short-MA section.
- Run: {VAR_RUN_ID}; fingerprint: {VAR_FP}.
- Models: outputs/models/{VAR_RUN_ID}.
- Fresh input tables: outputs/features/{VAR_RUN_ID}.
- Fresh forecasts: outputs/forecasts/{VAR_RUN_ID}.
- Reports: reports/tables/regression/{VAR_RUN_ID}.
- Machine checks passed; performance interpretation remains pending.
- Original regression_06_v1 models and data were preserved.
- No upstream numerical outputs were invalidated.
- Next: review matched MAE/RMSE, monthly stability and baseline-relative training/later errors.
- Deferred diagnostics: real/interpolated outcome splits, eligible-row missing rates and median calibration.
- No risk-action rules were changed; 2026 remains reserved.
"""

for var_file, var_marker, var_text in (
    (
        "docs/design_decisions.md",
        "short_ma_variation_v1",
        var_decision,
    ),
    (
        "CONTRIBUTIONS.md",
        "short_ma_variation_v1",
        var_contribution,
    ),
    (
        "docs/project_state.md",
        "short_ma_variation_v1",
        var_state,
    ),
):
    r.util.write_section(
        VAR_ROOT / var_file,
        var_marker,
        var_text.strip(),
    )

print("SHORT_MA_VARIATION_EXECUTION_SUMMARY")
print("New models:", VAR_SUMMARY["new_lightgbm_models"])
print("Inputs per new model:", VAR_SUMMARY["reduced_numeric_inputs"])
print("Fresh validation:", VAR_SUMMARY["fresh_validation"])
print("Nonzero checks:", VAR_SUMMARY["nonzero_machine_checks"])
print("Reports:", VAR_REPORTS)
print("STOP HERE. Tell Codex: Short-MA variation outputs ready.")

SHORT_MA_VARIATION_EXECUTION_SUMMARY
New models: 2
Inputs per new model: 21
Fresh validation: July-September 2019
Nonzero checks: 0
Reports: C:\Users\Ijzd4\Desktop\Inventory-Aware FX Pricing and Hedging Simulator\reports\tables\regression\regression_06_short_ma_v1
STOP HERE. Tell Codex: Short-MA variation outputs ready.


In [25]:
# # Feature-group removal experiment

# - Test the current 21-input models, removing one group at a time.
# - Train on all of 2018.
# - Use January–March 2019 for early stopping.
# - Test the 12 removals on October–December 2019.
# - Negative change in MAE means removal helped.
# - Use five-active-date bootstrap blocks, with one-date sensitivity.
# - Adjust the 12 tests together. Inconclusive groups stay.
# - Check the selected combination on January–March 2020.
# - Keep 2026 reserved.

# Results describe predictive value for these models and periods.
# They do not establish causality or universal feature importance.
# No model is promoted automatically.

In [26]:
# Settings. Run these NEW cells in order.
from pathlib import Path
import ast
import importlib
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path(
    r"C:\Users\Ijzd4\Desktop\Inventory-Aware FX Pricing and Hedging Simulator"
)
assert (ROOT / "functions" / "regression" / "fx_regression.py").is_file()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

GROUP_RUN_ID = "regression_06_group_tests_v1"
GROUP_MEMORY = "2GB"
GROUP_THREADS = 2
GROUP_WORKING_GB = 3.0

GROUP_RESAMPLES = 9999
GROUP_ALPHA = 0.05
GROUP_BLOCK_DAYS = 5
GROUP_SENSITIVITY_DAYS = 1
GROUP_SEED = 1729

TEST_PARTS = {}

In [27]:
# Helper: inputs, availability checks and reference models.
TEST_PARTS["core"] = r'''
import copy
import gc
import hashlib
import json
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import pyarrow as pa

from functions.regression import fx_regression as r

REFERENCE_RUN = "regression_06_short_ma_v1"
REFERENCE_FP = "650d88ee4e8ae4272136e4e9936f039aee522d00a6482c8e0bc93292cf4c4faf"

PHASES = {
    "test": (r.tg.ns("2019-10-01"), r.tg.ns("2020-01-01")),
    "confirmation": (r.tg.ns("2020-01-01"), r.tg.ns("2020-04-01")),
}

def project(table, columns):
    return np.column_stack([
        r.tg.get(table, c, np.nan) for c in columns
    ]).astype(np.float32, copy=False)

def eligible(table, phase, horizon):
    lower, upper = PHASES[phase]
    t = r.tg.get(table, "issue_ns")

    return (
        (t >= lower)
        & (t < upper)
        & r.tg.get(table, f"usable_{horizon}")
        & (
            r.tg.get(table, f"label_available_ns_{horizon}", 0)
            < upper
        )
    )

def check(ctx, built=False):
    r.check_snapshot(ctx)

    assert (
        r.util.sha(__file__)
        == ctx["config"]["test_helper_sha"]
    )

    r.check_features(ctx["training"])
    r.check_features(ctx["reference_features"])

    for item in ctx["config"]["reference_artifacts"]:
        assert (
            r.util.sha(ctx["root"] / item["path"])
            == item["sha"]
        ), item["path"]

    if built:
        r.check_features(ctx)

def load_reference(ctx):
    result = {}

    for h in ("30s", "60s"):
        path = ctx["reference_models"] / f"{h}_metadata.json"
        meta = r.tg.fx.load_json(path)

        model_path = (
            ctx["reference_models"] / f"{h}_lightgbm.txt"
        )

        assert meta["fingerprint"] == REFERENCE_FP
        assert meta["features"] == ctx["config"]["model_features"]
        assert r.util.sha(model_path) == meta["model_sha"]

        assert (
            meta["training_cutoff_ns"]
            < meta["model_available_ns"]
            <= PHASES["test"][0]
        )

        model = lgb.Booster(
            model_str=model_path.read_text(encoding="utf-8")
        )
        assert model.feature_name() == meta["features"]

        result[h] = {
            "model": model,
            "meta": meta,
            "kind": "boost",
        }

    return result

def predict(bundle, table):
    if bundle["kind"] == "zero":
        return np.zeros(table.num_rows)

    x = project(table, bundle["meta"]["features"])
    result = bundle["model"].predict(x)

    assert np.isfinite(result).all()
    return result

def memory_check(ctx):
    reference = load_reference(ctx)
    rows = []

    for h, bundle in reference.items():
        meta = bundle["meta"]
        n = meta["training_rows"] + meta["early_stopping_rows"]

        estimate = (
            6 * n * (4 * len(r.FEATURES) + 8) / 1024**3
        )

        rows.append({
            "Horizon": h,
            "Estimated working GiB": estimate,
            "Budget GiB": ctx["working_gb"],
        })

    frame = pd.DataFrame(rows)

    assert frame["Estimated working GiB"].le(
        ctx["working_gb"]
    ).all(), "Review the memory budget before fitting."

    r.tg.write_csv(
        frame, ctx["reports"] / "memory_estimate.csv"
    )
    return frame

def load_training(ctx, horizon):
    x, y, fit_info = r.load_role(
        ctx["training"], "train", horizon
    )
    sx, sy, stop_info = r.load_role(
        ctx["training"], "early_stopping", horizon
    )
    return x, y, fit_info, sx, sy, stop_info
'''

In [28]:
# Helper: prepare a separate run and build six monthly input files.
TEST_PARTS["inputs"] = r'''
def prepare(
    root, run_id, memory="2GB", threads=2, working_gb=3.0,
    resamples=9999, alpha=0.05, block_days=5,
    sensitivity_days=1, seed=1729,
):
    root = Path(root).resolve()

    assert run_id and Path(run_id).name == run_id
    assert run_id not in (REFERENCE_RUN, "regression_06_v1")
    assert 0 < alpha < 1 and resamples >= 999
    assert block_days > 0 and sensitivity_days > 0

    ref_features = root / "outputs" / "features" / REFERENCE_RUN
    ref_models = root / "outputs" / "models" / REFERENCE_RUN
    ref_reports = (
        root / "reports" / "tables" / "regression" / REFERENCE_RUN
    )

    ref_cfg_path = ref_features / "configuration.json"
    ref_cfg = r.tg.fx.load_json(ref_cfg_path)
    summary = r.tg.fx.load_json(
        ref_reports / "execution_summary.json"
    )

    assert summary["fingerprint"] == REFERENCE_FP
    assert summary["nonzero_machine_checks"] == 0
    assert len(ref_cfg["model_features"]) == 21
    assert ref_cfg["params"]["objective"] == "regression_l1"

    original = (
        root / "outputs" / "features" / "regression_06_v1"
    )
    original_cfg_path = original / "configuration.json"

    assert (
        r.util.sha(original_cfg_path)
        == ref_cfg["parent_config_sha"]
    )

    original_cfg = r.tg.fx.load_json(original_cfg_path)

    assert {
        k: list(v) for k, v in r.PERIODS.items()
    } == original_cfg["periods_ns"]
    assert r.GROUPS == original_cfg["features"]

    training = {
        "root": root,
        "feature_dir": original,
        "reports": (
            root / "reports" / "tables" / "regression"
            / "regression_06_v1"
        ),
        "months": [
            str(m)
            for m in pd.period_range(
                "2018-01", "2019-06", freq="M"
            )
        ],
        "config": original_cfg,
        "fingerprint": ref_cfg["parent_fingerprint"],
        "memory": memory,
        "threads": threads,
    }

    r.check_evidence(training)
    r.check_snapshot(training)
    r.check_features(training)

    artifacts = copy.deepcopy(ref_cfg["parent_artifacts"])

    for path in [
        ref_cfg_path,
        original_cfg_path,
        *[
            ref_models / f"{h}_{suffix}"
            for h in ("30s", "60s")
            for suffix in ("metadata.json", "lightgbm.txt")
        ],
        *[
            ref_features / "_checkpoints" / f"{m}.json"
            for m in ("2019-07", "2019-08", "2019-09")
        ],
    ]:
        artifacts.append({
            "path": path.relative_to(root).as_posix(),
            "sha": r.util.sha(path),
        })

    assert len({
        p["path"] for p in artifacts
    }) == len(artifacts)

    months = [
        str(m)
        for m in pd.period_range(
            "2019-10", "2020-03", freq="M"
        )
    ]

    sample = root / "data" / "processed" / r.SAMPLE_RUN
    target = root / "data" / "processed" / r.TARGET_RUN
    entropy = (
        root / "outputs" / "features" / "entropy"
        / r.ENTROPY_RUN
    )

    raw = {
        p["year_month"]: p
        for p in r.tg.fx.load_json(
            sample / "configuration.json"
        )["inputs"]
    }

    additions = []

    for month in months:
        cp = sample / "_checkpoints" / f"{month}.json"
        saved = r.tg.fx.load_json(cp)

        assert saved["fingerprint"] == r.EXPECTED["sampling"]
        assert all(v == 0 for v in saved["checks"].values())

        for kind in ("sampled_10s", "candles_3min"):
            additions.append((
                kind,
                sample / kind / f"{month}.parquet",
                saved["outputs"][kind],
                cp,
            ))

        cp = (
            root / "data" / "processed" / "etl_01_v3"
            / "_checkpoints" / f"observed_{month}.json"
        )

        assert (
            r.util.sha(cp)
            == raw[month]["checkpoint_sha256"]
        )

        additions.append((
            "observed",
            root / raw[month]["path"],
            raw[month],
            cp,
        ))

    for day in pd.date_range(
        "2019-10-01", "2020-03-31", freq="D"
    ):
        date = day.strftime("%Y-%m-%d")

        for kind, folder in (
            ("targets", target),
            ("entropy", entropy),
        ):
            cp = folder / "_checkpoints" / f"{date}.json"
            saved = r.tg.fx.load_json(cp)

            assert saved["fingerprint"] == r.EXPECTED[kind]
            assert all(
                v == 0 for v in saved["checks"].values()
            )

            additions.append((
                kind,
                folder / f"{date}.parquet",
                saved["output"],
                cp,
            ))

    inputs = copy.deepcopy(ref_cfg["inputs"])

    for kind, path, expected, cp in additions:
        info = r.util.file_info(path)

        assert all(
            info[k] == expected[k] for k in info
        ), str(path)

        inputs.append({
            "kind": kind,
            "path": path.relative_to(root).as_posix(),
            **info,
            "checkpoint": cp.relative_to(root).as_posix(),
            "checkpoint_sha": r.util.sha(cp),
        })

    assert len({
        (v["kind"], v["path"]) for v in inputs
    }) == len(inputs)

    groups = ref_cfg["model_feature_groups"]

    assert len(groups) == 6
    assert [
        c for columns in groups.values() for c in columns
    ] == ref_cfg["model_features"]

    cases = []

    for h in ("30s", "60s"):
        for i, (group, columns) in enumerate(groups.items()):
            keep = [
                c for c in ref_cfg["model_features"]
                if c not in columns
            ]
            cases.append({
                "id": f"omit_{h}_{i}",
                "horizon": h,
                "group": group,
                "features": keep,
            })

    config = copy.deepcopy(ref_cfg)

    config.update({
        "definition": "group_significance_v1",
        "run_id": run_id,
        "inputs": inputs,
        "reference_run": REFERENCE_RUN,
        "parent_run_id": REFERENCE_RUN,
        "parent_fingerprint": REFERENCE_FP,
        "parent_config_sha": r.util.sha(ref_cfg_path),
        "parent_artifacts": artifacts,
        "model_available_ns": PHASES["test"][0],
        "architecture_information_cutoff_ns": (
            PHASES["test"][0] - 1
        ),
        "reference_fingerprint": REFERENCE_FP,
        "reference_artifacts": artifacts,
        "test_helper_sha": r.util.sha(__file__),
        "hypotheses": cases,
        "evaluation_periods_ns": PHASES,
        "comparison_scope": (
            "Q4 2019 group tests; Q1 2020 combined confirmation"
        ),
        "resamples": resamples,
        "alpha": alpha,
        "bootstrap_seed": seed,
        "primary_block_days": block_days,
        "sensitivity_block_days": sensitivity_days,
        "bootstrap": (
            "noncircular moving blocks of consecutive "
            "nonempty Athens dates"
        ),
        "interval": (
            "pointwise basic block-bootstrap; not simultaneous"
        ),
        "p_value": (
            "two-sided equal-tail centered-null bootstrap; "
            "finite-resample correction"
        ),
        "multiplicity": (
            "Holm across 12 primary tests; "
            "separate 2-test confirmation family"
        ),
        "selection": (
            "negative delta MAE and primary Holm p <= alpha"
        ),
        "zero_feature_candidate": "zero-change forecast",
        "uncertainty": (
            "conditional on fixed fitted models; "
            "approximate within each quarter"
        ),
    })

    config = json.loads(json.dumps(config))
    fingerprint = hashlib.sha256(
        json.dumps(
            config, sort_keys=True, separators=(",", ":")
        ).encode()
    ).hexdigest()

    feature_dir = root / "outputs" / "features" / run_id
    model_dir = root / "outputs" / "models" / run_id
    forecast_dir = root / "outputs" / "forecasts" / run_id
    reports = (
        root / "reports" / "tables" / "regression" / run_id
    )

    for path in (
        feature_dir,
        feature_dir / "_checkpoints",
        model_dir,
        reports,
        forecast_dir / "reference",
        forecast_dir / "selected",
    ):
        path.mkdir(parents=True, exist_ok=True)

    path = feature_dir / "configuration.json"

    if path.exists():
        old = r.tg.fx.load_json(path)
        changed = [
            k for k in config if old.get(k) != config[k]
        ]
        assert not changed, (
            f"Configuration changed {changed}; "
            "use a new RUN_ID."
        )
    else:
        r.tg.write_json(path, config)

    ctx = {
        "root": root,
        "feature_dir": feature_dir,
        "model_dir": model_dir,
        "forecast_dir": forecast_dir,
        "reports": reports,
        "config": config,
        "fingerprint": fingerprint,
        "run_id": run_id,
        "months": months,
        "training": training,
        "reference_models": ref_models,
        "reference_features": {
            "feature_dir": ref_features,
            "months": ["2019-07", "2019-08", "2019-09"],
            "fingerprint": REFERENCE_FP,
        },
        "memory": memory,
        "threads": threads,
        "working_gb": working_gb,
    }

    check(ctx)
    r.tg.write_csv(
        pd.DataFrame(inputs),
        reports / "input_inventory.csv",
    )
    return ctx

def build(ctx):
    check(ctx)

    evidence = r.tg.fx.load_json(
        ctx["reports"] / "synthetic_checks.json"
    )
    assert evidence["passed"]
    assert evidence["fingerprint"] == ctx["fingerprint"]

    records = []

    with duckdb.connect(config={
        "memory_limit": ctx["memory"],
        "threads": str(ctx["threads"]),
    }) as con:
        for month in ctx["months"]:
            path = ctx["feature_dir"] / f"{month}.parquet"
            cp = (
                ctx["feature_dir"] / "_checkpoints"
                / f"{month}.json"
            )

            if cp.exists():
                saved = r.tg.fx.load_json(cp)

                assert (
                    saved["fingerprint"] == ctx["fingerprint"]
                )
                assert (
                    r.util.file_info(path) == saved["output"]
                )
                assert all(
                    v == 0 for v in saved["checks"].values()
                )
            else:
                table, checks, _, _ = r.make_month(
                    ctx, month, con
                )

                counts = {
                    "month": month,
                    "rows": table.num_rows,
                }

                for phase in PHASES:
                    for h in ("30s", "60s"):
                        counts[f"{phase}_{h}"] = int(
                            eligible(table, phase, h).sum()
                        )

                r.tg.write_parquet(table, path)

                saved = {
                    "fingerprint": ctx["fingerprint"],
                    "checks": checks,
                    "counts": counts,
                    "output": r.util.file_info(path),
                }

                r.tg.write_json(cp, saved)
                print(f"{month}: inputs saved.")

            records.append(saved)

    check(ctx, built=True)

    r.tg.write_csv(
        pd.DataFrame([s["counts"] for s in records]),
        ctx["reports"] / "feature_counts.csv",
    )
    r.tg.write_csv(
        pd.DataFrame([
            {
                "month": s["counts"]["month"],
                **s["checks"],
            }
            for s in records
        ]),
        ctx["reports"] / "feature_checks.csv",
    )

    return memory_check(ctx)
'''

In [29]:
# Helper: fit the omissions and score matching forecasts.
TEST_PARTS["models"] = r'''
def fit_case(ctx, case, references, data=None):
    key = case["id"]
    h = case["horizon"]
    columns = case["features"]

    path = ctx["model_dir"] / f"{key}.txt"
    meta_path = ctx["model_dir"] / f"{key}.json"

    if meta_path.exists():
        meta = r.tg.fx.load_json(meta_path)

        assert meta["fingerprint"] == ctx["fingerprint"]
        assert meta["features"] == columns
        assert meta["model_available_ns"] == case["ready_ns"]
        assert (
            meta["training_cutoff_ns"]
            < meta["model_available_ns"]
        )
        assert (
            meta["selection_report_sha"]
            == case.get("selection_report_sha")
        )

        if meta["kind"] == "zero":
            return {
                "model": None,
                "meta": meta,
                "kind": "zero",
            }

        assert r.util.sha(path) == meta["model_sha"]

        model = lgb.Booster(
            model_str=path.read_text(encoding="utf-8")
        )
        assert model.feature_name() == columns

        return {
            "model": model,
            "meta": meta,
            "kind": meta["kind"],
        }

    started = time.perf_counter()
    reference = references[h]
    ref_meta = reference["meta"]
    new_fit = False

    if not columns:
        model, kind, trees = None, "zero", 0
        fit_info = ref_meta["fit_information_cutoff_ns"]
        stop_info = ref_meta[
            "early_stopping_information_cutoff_ns"
        ]
        n_train, n_stop = 0, 0

    elif columns == ctx["config"]["model_features"]:
        model = reference["model"]
        kind = "reference_copy"
        trees = ref_meta["best_iteration"]
        fit_info = ref_meta["fit_information_cutoff_ns"]
        stop_info = ref_meta[
            "early_stopping_information_cutoff_ns"
        ]
        n_train = ref_meta["training_rows"]
        n_stop = ref_meta["early_stopping_rows"]

    else:
        if data is None:
            data = load_training(ctx, h)

        full_x, y, fit_info, full_stop, stop_y, stop_info = data
        indices = [r.FEATURES.index(c) for c in columns]

        x = np.ascontiguousarray(full_x[:, indices])
        stop_x = np.ascontiguousarray(full_stop[:, indices])

        assert len(y) == ref_meta["training_rows"]
        assert len(stop_y) == ref_meta["early_stopping_rows"]
        assert np.isfinite(x).any(axis=0).all()

        train_data = lgb.Dataset(
            x, label=y, feature_name=columns
        )
        stop_data = lgb.Dataset(
            stop_x,
            label=stop_y,
            reference=train_data,
            feature_name=columns,
        )

        model = lgb.train(
            copy.deepcopy(ctx["config"]["params"]),
            train_data,
            num_boost_round=ctx["config"]["maximum_trees"],
            valid_sets=[stop_data],
            valid_names=["early_stopping"],
            callbacks=[
                lgb.early_stopping(
                    ctx["config"]["early_stopping_rounds"],
                    first_metric_only=True,
                    verbose=False,
                ),
                lgb.log_evaluation(0),
            ],
        )

        assert model.best_iteration >= 1
        trees = int(model.best_iteration)
        kind, new_fit = "boost", True
        n_train, n_stop = len(y), len(stop_y)

        del train_data, stop_data, x

    ready = case["ready_ns"]
    cutoff = max(fit_info, stop_info, ready - 1)
    assert cutoff < ready

    if kind != "zero":
        text = model.model_to_string(num_iteration=trees)
        restored = lgb.Booster(model_str=text)

        if new_fit:
            assert np.allclose(
                model.predict(stop_x[:1000]),
                restored.predict(stop_x[:1000]),
                rtol=0,
                atol=1e-10,
            )

        r.save_text(path, text)
        model = restored

    meta = {
        "fingerprint": ctx["fingerprint"],
        "case_id": key,
        "horizon": h,
        "group": case.get("group"),
        "features": columns,
        "kind": kind,
        "new_fit": new_fit,
        "best_iteration": trees,
        "training_rows": n_train,
        "early_stopping_rows": n_stop,
        "training_period": (
            "Not fitted: zero-change benchmark"
            if kind == "zero"
            else "January-December 2018"
        ),
        "objective": "regression_l1",
        "estimate": "conditional median point forecast",
        "fit_information_cutoff_ns": fit_info,
        "early_stopping_information_cutoff_ns": stop_info,
        "selection_information_cutoff_ns": ready - 1,
        "training_cutoff_ns": cutoff,
        "model_available_ns": ready,
        "model_identifier": f"{ctx['run_id']}_{key}",
        "inference_latency_ns": 0,
        "selection_report_sha": case.get("selection_report_sha"),
        "model_sha": (
            None if kind == "zero" else r.util.sha(path)
        ),
        "fit_minutes": (
            time.perf_counter() - started
        ) / 60,
    }

    r.tg.write_json(meta_path, meta)
    gc.collect()

    return {
        "model": model,
        "meta": meta,
        "kind": kind,
    }

def fit_omissions(ctx):
    check(ctx, built=True)
    memory_check(ctx)

    references = load_reference(ctx)
    result, rows = {}, []

    for h in ("30s", "60s"):
        cases = [
            dict(c, ready_ns=PHASES["test"][0])
            for c in ctx["config"]["hypotheses"]
            if c["horizon"] == h
        ]

        pending = any(
            not (
                ctx["model_dir"] / f"{c['id']}.json"
            ).exists()
            for c in cases
        )

        data = load_training(ctx, h) if pending else None

        for case in cases:
            bundle = fit_case(
                ctx, case, references, data
            )
            result[case["id"]] = bundle
            meta = bundle["meta"]

            rows.append({
                "Group": case["group"],
                "Horizon": h,
                "Inputs": len(meta["features"]),
                "Trees": meta["best_iteration"],
                "Training examples": meta["training_rows"],
            })

            print(f"{h} | omit {case['group']}: ready.")

        del data
        gc.collect()

    check(ctx, built=True)

    report = pd.DataFrame(rows)
    r.tg.write_csv(
        report,
        ctx["reports"] / "omission_fit_summary.csv",
    )
    return result

def score_omissions(ctx, models):
    check(ctx, built=True)
    references = load_reference(ctx)

    daily, monthly, checks = [], [], []

    for month in ("2019-10", "2019-11", "2019-12"):
        with duckdb.connect(config={
            "memory_limit": ctx["memory"],
            "threads": str(ctx["threads"]),
        }) as con:
            table = r.read(
                con,
                [ctx["feature_dir"] / f"{month}.parquet"],
                [
                    *r.BASE,
                    *r.FEATURES,
                    "input_information_max_ns",
                ],
                "issue_ns",
            )

        times = r.tg.get(table, "issue_ns")
        dates = np.asarray(
            pd.to_datetime(times, unit="ns", utc=True)
            .tz_convert(r.TZ)
            .strftime("%Y-%m-%d")
        )

        issue_checks = {
            "month": month,
            "late_inputs": int(np.count_nonzero(
                r.tg.get(table, "input_information_max_ns")
                > times
            )),
        }

        for h in ("30s", "60s"):
            mask = eligible(table, "test", h)
            actual = r.tg.get(
                table, f"target_change_{h}_pips", np.nan
            )[mask]

            ref_pred = predict(references[h], table)
            ref_error = ref_pred[mask] - actual

            assert len(actual) and np.isfinite(actual).all()

            days, inverse = np.unique(
                dates[mask], return_inverse=True
            )
            counts = np.bincount(inverse)

            cases = [
                c for c in ctx["config"]["hypotheses"]
                if c["horizon"] == h
            ]

            for case in cases:
                bundle = models[case["id"]]

                assert np.all(
                    times >= bundle["meta"]["model_available_ns"]
                )

                pred = predict(bundle, table)
                error = pred[mask] - actual

                sums = np.bincount(
                    inverse,
                    weights=(
                        np.abs(error) - np.abs(ref_error)
                    ),
                )

                for day, total, count in zip(
                    days, sums, counts
                ):
                    daily.append({
                        "date": day,
                        "case_id": case["id"],
                        "horizon": h,
                        "group": case["group"],
                        "count": int(count),
                        "difference_sum": float(total),
                    })

                monthly.append({
                    "month": month,
                    "case_id": case["id"],
                    "horizon": h,
                    "group": case["group"],
                    "count": len(actual),
                    "reference_absolute_error": float(
                        np.abs(ref_error).sum()
                    ),
                    "without_absolute_error": float(
                        np.abs(error).sum()
                    ),
                    "reference_squared_error": float(
                        np.square(ref_error).sum()
                    ),
                    "without_squared_error": float(
                        np.square(error).sum()
                    ),
                })

                issue_checks[
                    case["id"] + "_late_model"
                ] = int(np.count_nonzero(
                    times
                    <= bundle["meta"]["training_cutoff_ns"]
                ))

        assert all(
            v == 0 for k, v in issue_checks.items()
            if k != "month"
        ), issue_checks

        checks.append(issue_checks)
        print(f"{month}: all 12 comparisons scored.")

    daily = pd.DataFrame(daily)
    monthly = pd.DataFrame(monthly)

    monthly["delta_mae_pips"] = (
        monthly["without_absolute_error"]
        - monthly["reference_absolute_error"]
    ) / monthly["count"]

    assert monthly.groupby(
        ["month", "horizon"]
    )["count"].nunique().eq(1).all()

    assert daily.groupby(
        ["date", "horizon"]
    )["count"].nunique().eq(1).all()

    for case_id, item in monthly.groupby(
        "case_id", sort=False
    ):
        d = daily.loc[daily["case_id"].eq(case_id)]

        assert int(d["count"].sum()) == int(
            item["count"].sum()
        )
        assert np.isclose(
            d["difference_sum"].sum(),
            (
                item["without_absolute_error"]
                - item["reference_absolute_error"]
            ).sum(),
            rtol=0,
            atol=1e-7,
        )

    r.tg.write_csv(
        daily,
        ctx["reports"] / "paired_daily_test_losses.csv",
    )
    r.tg.write_csv(
        monthly,
        ctx["reports"] / "monthly_omission_scores.csv",
    )
    r.tg.write_csv(
        pd.DataFrame(checks),
        ctx["reports"] / "omission_checks.csv",
    )

    check(ctx, built=True)
    return daily
'''

In [30]:
# Helper: paired bootstrap, multiple-test correction and selection.
TEST_PARTS["statistics"] = r'''
def holm(p_values):
    p_values = np.asarray(p_values, dtype=float)

    assert np.isfinite(p_values).all()
    assert np.all((p_values >= 0) & (p_values <= 1))

    order = np.argsort(p_values)
    ranked = p_values[order] * (
        len(p_values) - np.arange(len(p_values))
    )
    adjusted = np.minimum(
        1.0, np.maximum.accumulate(ranked)
    )

    result = np.empty_like(adjusted)
    result[order] = adjusted
    return result

def block_test(sums, counts, block, draws, alpha, seed):
    sums = np.asarray(sums, dtype=float)
    counts = np.asarray(counts, dtype=float)

    assert sums.shape == counts.shape and sums.ndim == 2
    assert np.isfinite(sums).all()
    assert np.isfinite(counts).all()
    assert np.all(counts >= 0)

    active = counts.sum(axis=1) > 0
    sums, counts = sums[active], counts[active]
    n, k = sums.shape

    assert n >= 2 * block, (
        "Too few active dates for the chosen block length."
    )
    assert np.all(counts.sum(axis=0) > 0)
    assert np.all(sums[counts == 0] == 0)

    # Preserve forecast weighting: sum of losses / sum of rows.
    observed = sums.sum(axis=0) / counts.sum(axis=0)
    centered = sums - counts * observed

    null = np.empty((draws, k))
    rng = np.random.default_rng(seed)
    done, rejected = 0, 0
    blocks = (n + block - 1) // block

    while done < draws:
        batch = min(256, draws - done)

        starts = rng.integers(
            0, n - block + 1,
            size=(batch, blocks),
        )
        index = (
            starts[:, :, None] + np.arange(block)
        ).reshape(batch, -1)[:, :n]

        denominator = counts[index].sum(axis=1)
        good = np.all(denominator > 0, axis=1)
        rejected += int((~good).sum())

        if not good.any():
            assert rejected < 100 * draws, (
                "Bootstrap cohorts need review."
            )
            continue

        index = index[good]
        denominator = denominator[good]

        values = (
            centered[index].sum(axis=1)
            / denominator
        )

        take = min(len(values), draws - done)
        null[done:done + take] = values[:take]
        done += take

    assert rejected <= max(1, draws // 100), (
        "Too many zero-count bootstrap draws; review cohorts."
    )

    # Basic intervals invert the centered error distribution.
    lower = observed - np.quantile(
        null, 1 - alpha / 2, axis=0
    )
    upper = observed - np.quantile(
        null, alpha / 2, axis=0
    )

    lower_tail = (
        1 + (null <= observed).sum(axis=0)
    ) / (draws + 1)
    upper_tail = (
        1 + (null >= observed).sum(axis=0)
    ) / (draws + 1)

    p = np.minimum(
        1.0, 2 * np.minimum(lower_tail, upper_tail)
    )

    return {
        "delta": observed,
        "lower": lower,
        "upper": upper,
        "p": p,
        "active_dates": n,
        "rejected_draws": rejected,
    }

def infer(ctx, daily, descriptors, stem, family_size):
    ids = [case["id"] for case in descriptors]

    assert len(ids) == family_size
    assert len(set(ids)) == family_size

    sums = daily.pivot(
        index="date",
        columns="case_id",
        values="difference_sum",
    )
    counts = daily.pivot(
        index="date",
        columns="case_id",
        values="count",
    )

    dates = sorted(set(sums.index) | set(counts.index))

    # Zero here means no scored forecasts on that date.
    # It does not fill any market data or missing feature.
    sums = sums.reindex(
        index=dates, columns=ids
    ).fillna(0).to_numpy(dtype=float)

    counts = counts.reindex(
        index=dates, columns=ids
    ).fillna(0).to_numpy(dtype=float)

    results = {}

    for name, block in (
        ("primary", ctx["config"]["primary_block_days"]),
        (
            "sensitivity",
            ctx["config"]["sensitivity_block_days"],
        ),
    ):
        value = block_test(
            sums,
            counts,
            block,
            ctx["config"]["resamples"],
            ctx["config"]["alpha"],
            ctx["config"]["bootstrap_seed"] + block,
        )

        adjusted = holm(value["p"])
        rows = []

        for i, case in enumerate(descriptors):
            delta = float(value["delta"][i])
            significant = bool(
                adjusted[i] <= ctx["config"]["alpha"]
            )

            if significant:
                evidence = (
                    "Removal improves MAE"
                    if delta < 0 else "Group helps"
                )
            else:
                evidence = "Inconclusive"

            if stem == "confirmation":
                if significant:
                    evidence = (
                        "Candidate improves MAE"
                        if delta < 0
                        else "Candidate worsens MAE"
                    )
                else:
                    evidence = "Inconclusive"

            rows.append({
                "case_id": case["id"],
                "horizon": case["horizon"],
                "group": case.get(
                    "group", "Selected composite"
                ),
                "delta_mae_pips": delta,
                "ci_low": float(value["lower"][i]),
                "ci_high": float(value["upper"][i]),
                "p_raw": float(value["p"][i]),
                "p_holm": float(adjusted[i]),
                "evidence": evidence,
                "drop_candidate": bool(
                    stem != "confirmation"
                    and delta < 0
                    and significant
                ),
                "block_days": block,
                "active_dates": value["active_dates"],
                "scored_forecasts": int(
                    counts[:, i].sum()
                ),
                "rejected_draws": value["rejected_draws"],
                "interval_scope": (
                    "pointwise; not simultaneous"
                ),
            })

        frame = pd.DataFrame(rows)

        r.tg.write_csv(
            frame,
            ctx["reports"]
            / f"{stem}_{name}_inference.csv",
        )
        results[name] = frame

    return results["primary"], results["sensitivity"]

def inference_display(frame):
    return pd.DataFrame({
        "Group": frame["group"],
        "Horizon": frame["horizon"],
        "Delta MAE (pips)": frame[
            "delta_mae_pips"
        ].map(lambda v: f"{v:+.7f}"),
        "95% interval": [
            f"[{a:+.7f}, {b:+.7f}]"
            for a, b in zip(
                frame["ci_low"], frame["ci_high"]
            )
        ],
        "Adjusted p": frame["p_holm"].round(4),
        "Evidence": frame["evidence"],
    })

def select_and_fit(ctx, inference):
    check(ctx, built=True)
    memory_check(ctx)
    references = load_reference(ctx)

    report_path = (
        ctx["reports"] / "omissions_primary_inference.csv"
    )

    assert inference["case_id"].tolist() == [
        c["id"] for c in ctx["config"]["hypotheses"]
    ]

    saved = pd.read_csv(report_path)

    assert np.array_equal(
        saved["drop_candidate"].to_numpy(),
        inference["drop_candidate"].to_numpy(),
    )
    assert np.allclose(
        saved["delta_mae_pips"],
        inference["delta_mae_pips"],
        rtol=0,
        atol=1e-12,
    )

    report_sha = r.util.sha(report_path)
    selected, rows, specifications = {}, [], []

    for h in ("30s", "60s"):
        chosen = inference.loc[
            inference["horizon"].eq(h)
            & inference["drop_candidate"],
            "group",
        ].tolist()

        removed = {
            c
            for group in chosen
            for c in ctx["config"]["model_feature_groups"][group]
        }
        columns = [
            c for c in ctx["config"]["model_features"]
            if c not in removed
        ]

        case = {
            "id": f"selected_{h}",
            "horizon": h,
            "features": columns,
            "ready_ns": PHASES["confirmation"][0],
            "selection_report_sha": report_sha,
        }

        selected[h] = fit_case(
            ctx, case, references
        )

        specifications.append({
            **case,
            "removed_groups": chosen,
        })
        rows.append({
            "Horizon": h,
            "Removal candidate groups": (
                ", ".join(chosen) or "None"
            ),
            "Remaining inputs": len(columns),
            "Candidate type": selected[h]["kind"],
        })

    r.tg.write_json(
        ctx["reports"] / "selected_specs.json",
        specifications,
    )

    check(ctx, built=True)
    return selected, pd.DataFrame(rows)

def synthetic_checks():
    assert np.allclose(
        holm([0.01, 0.04, 0.03]),
        [0.03, 0.06, 0.06],
    )
    checks = {"holm_known_example": True}

    n = np.arange(1, 21, dtype=float)
    counts = np.column_stack([n, n])
    sums = -0.1 * counts

    result = block_test(
        sums, counts,
        block=5, draws=199, alpha=0.05, seed=17,
    )

    assert np.allclose(result["delta"], [-0.1, -0.1])
    assert np.allclose(result["lower"], [-0.1, -0.1])
    assert np.allclose(result["upper"], [-0.1, -0.1])
    assert result["p"][0] == result["p"][1]

    checks[
        "paired_common_draws_and_constant_difference"
    ] = True

    sums = np.column_stack([
        np.arange(20, dtype=float),
        np.arange(20, dtype=float),
    ])

    result = block_test(
        sums, counts, 1, 199, 0.05, 17
    )

    assert abs(
        result["delta"][0] - 190 / 210
    ) < 1e-12

    checks["observation_weighted_daily_ratio"] = True

    result = block_test(
        np.zeros_like(counts),
        counts, 5, 199, 0.05, 17,
    )

    assert np.array_equal(
        result["p"], [1.0, 1.0]
    )
    assert np.all(result["lower"] == 0)
    assert np.all(result["upper"] == 0)

    checks[
        "identical_forecasts_are_not_significant"
    ] = True

    cut = PHASES["test"][1]
    table = pa.table({
        "issue_ns": [cut - r.STEP, cut - r.STEP],
        "usable_30s": [True, True],
        "label_available_ns_30s": [cut, cut - 1],
    })

    assert np.array_equal(
        eligible(table, "test", "30s"),
        [False, True],
    )

    checks["quarter_information_cutoff"] = True

    return {
        "passed": all(checks.values()),
        "checks": checks,
    }
'''

In [31]:
# Helper: fresh confirmation, forecast export and documentation.
TEST_PARTS["confirmation"] = r'''
def forecast_table(table, bundles, predictions):
    times = r.tg.get(table, "issue_ns")

    assert np.all(
        r.tg.get(table, "input_information_max_ns")
        <= times
    )

    reference = r.tg.get(
        table, "reference_mid", np.nan
    )

    data = {
        c: table[c]
        for c in (
            "issue_ns",
            "reference_quote_ns",
            "reference_mid",
            "reference_age_ns",
        )
    }

    for h in ("30s", "60s"):
        meta = bundles[h]["meta"]

        assert (
            meta["training_cutoff_ns"]
            < meta["model_available_ns"]
        )
        assert np.all(
            times >= meta["model_available_ns"]
        )

        data.update({
            f"predicted_change_{h}_pips": pa.array(
                predictions[h]
            ),
            f"expected_eurusd_rate_{h}": pa.array(
                r.tg.predicted_mid(
                    reference, predictions[h]
                )
            ),
            f"forecast_available_ns_{h}": pa.array(
                times + meta["inference_latency_ns"]
            ),
            f"model_identifier_{h}": pa.array(
                [meta["model_identifier"]] * len(times)
            ),
            f"training_cutoff_ns_{h}": pa.array(
                np.full(
                    len(times),
                    meta["training_cutoff_ns"],
                    dtype=np.int64,
                )
            ),
        })

    result = pa.table(data)

    assert result.num_columns == 14
    assert result.num_rows == table.num_rows
    return result

def confirm(ctx, selected):
    check(ctx, built=True)
    references = load_reference(ctx)

    specs = r.tg.fx.load_json(
        ctx["reports"] / "selected_specs.json"
    )
    assert {
        v["horizon"] for v in specs
    } == {"30s", "60s"}

    selection_sha = r.util.sha(
        ctx["reports"] / "omissions_primary_inference.csv"
    )

    assert all(
        v["selection_report_sha"] == selection_sha
        for v in specs
    )

    daily, monthly, checks, examples = [], [], [], []

    columns = [
        *r.BASE,
        *r.FEATURES,
        "input_information_max_ns",
    ]

    for month in ("2020-01", "2020-02", "2020-03"):
        with duckdb.connect(config={
            "memory_limit": ctx["memory"],
            "threads": str(ctx["threads"]),
        }) as con:
            table = r.read(
                con,
                [ctx["feature_dir"] / f"{month}.parquet"],
                columns,
                "issue_ns",
            )

        times = r.tg.get(table, "issue_ns")
        dates = np.asarray(
            pd.to_datetime(times, unit="ns", utc=True)
            .tz_convert(r.TZ)
            .strftime("%Y-%m-%d")
        )

        month_checks = {
            "month": month,
            "late_inputs": int(np.count_nonzero(
                r.tg.get(table, "input_information_max_ns")
                > times
            )),
        }

        saved_predictions = {
            "reference": {},
            "selected": {},
        }

        for h in ("30s", "60s"):
            ref = references[h]
            candidate = selected[h]

            assert (
                candidate["meta"]["selection_report_sha"]
                == selection_sha
            )
            assert (
                candidate["meta"]["case_id"]
                == f"selected_{h}"
            )

            predictions = {
                "Reference 21": predict(ref, table),
                "Selected candidate": predict(
                    candidate, table
                ),
                "Zero change": np.zeros(table.num_rows),
            }

            if candidate["kind"] == "reference_copy":
                assert np.allclose(
                    predictions["Reference 21"],
                    predictions["Selected candidate"],
                    rtol=0,
                    atol=1e-10,
                )

                # These are the same fitted model.
                # Avoid treating serialization rounding as a change.
                predictions["Selected candidate"] = (
                    predictions["Reference 21"].copy()
                )

            saved_predictions["reference"][h] = (
                predictions["Reference 21"]
            )
            saved_predictions["selected"][h] = (
                predictions["Selected candidate"]
            )

            for label, bundle in (
                ("reference", ref),
                ("selected", candidate),
            ):
                month_checks[
                    f"{label}_{h}_late_model"
                ] = int(np.count_nonzero(
                    times
                    <= bundle["meta"]["training_cutoff_ns"]
                ))

            mask = eligible(
                table, "confirmation", h
            )
            actual = r.tg.get(
                table, f"target_change_{h}_pips", np.nan
            )[mask]

            assert len(actual) and np.isfinite(actual).all()

            errors = {
                key: value[mask] - actual
                for key, value in predictions.items()
            }

            days, inverse = np.unique(
                dates[mask], return_inverse=True
            )
            counts = np.bincount(inverse)

            differences = (
                np.abs(errors["Selected candidate"])
                - np.abs(errors["Reference 21"])
            )
            sums = np.bincount(
                inverse, weights=differences
            )

            daily.extend({
                "date": day,
                "case_id": f"selected_{h}",
                "horizon": h,
                "count": int(n),
                "difference_sum": float(value),
            } for day, n, value in zip(
                days, counts, sums
            ))

            for model, error in errors.items():
                monthly.append({
                    "month": month,
                    "horizon": h,
                    "model": model,
                    "count": len(actual),
                    "absolute_error": float(
                        np.abs(error).sum()
                    ),
                    "squared_error": float(
                        np.square(error).sum()
                    ),
                })

            if month == "2020-01":
                lower = r.tg.ns(
                    "2020-01-06T10:00:00"
                )
                chosen = np.flatnonzero(
                    mask
                    & (times >= lower)
                    & (times < lower + 40 * r.NS)
                )

                for i in chosen:
                    examples.append({
                        "Athens time": pd.Timestamp(
                            times[i], unit="ns", tz="UTC"
                        ).tz_convert(r.TZ).strftime(
                            "%d %b %Y %H:%M:%S"
                        ),
                        "Horizon": h,
                        "Current mid": r.tg.get(
                            table, "reference_mid"
                        )[i],
                        "Reference forecast": (
                            r.tg.get(
                                table, "reference_mid"
                            )[i]
                            + predictions["Reference 21"][i]
                            * r.PIP
                        ),
                        "Candidate forecast": (
                            r.tg.get(
                                table, "reference_mid"
                            )[i]
                            + predictions[
                                "Selected candidate"
                            ][i] * r.PIP
                        ),
                        "Observed future mid": r.tg.get(
                            table,
                            f"target_future_mid_{h}",
                            np.nan,
                        )[i],
                        "Interpolated outcome": bool(
                            r.tg.get(
                                table,
                                f"endpoint_is_interpolated_{h}",
                                False,
                            )[i]
                        ),
                    })

        # Export every issue-grid row.
        # Future-label validity never selects exported forecasts.
        for label, bundles in (
            ("reference", references),
            ("selected", selected),
        ):
            forecasts = forecast_table(
                table, bundles, saved_predictions[label]
            )
            r.tg.write_parquet(
                forecasts,
                ctx["forecast_dir"] / label
                / f"{month}.parquet",
            )

        assert all(
            v == 0 for k, v in month_checks.items()
            if k != "month"
        ), month_checks

        checks.append(month_checks)
        print(
            f"{month}: confirmation scored; "
            "all-grid forecasts saved."
        )

    monthly = pd.DataFrame(monthly)
    daily = pd.DataFrame(daily)

    assert monthly.groupby(
        ["month", "horizon"]
    )["count"].nunique().eq(1).all()

    for h in ("30s", "60s"):
        d = daily.loc[daily["horizon"].eq(h)]

        a = monthly.loc[
            monthly["horizon"].eq(h)
        ].groupby("model")[
            ["absolute_error", "count"]
        ].sum()

        assert int(d["count"].sum()) == int(
            a.loc["Reference 21", "count"]
        )
        assert np.isclose(
            d["difference_sum"].sum(),
            a.loc["Selected candidate", "absolute_error"]
            - a.loc["Reference 21", "absolute_error"],
            rtol=0,
            atol=1e-7,
        )

    r.tg.write_csv(
        daily,
        ctx["reports"]
        / "paired_daily_confirmation_losses.csv",
    )
    r.tg.write_csv(
        monthly,
        ctx["reports"]
        / "monthly_confirmation_scores.csv",
    )
    r.tg.write_csv(
        pd.DataFrame(checks),
        ctx["reports"] / "confirmation_checks.csv",
    )

    totals = monthly.groupby(
        ["horizon", "model"], sort=False
    )[[
        "count",
        "absolute_error",
        "squared_error",
    ]].sum().reset_index()

    totals["mae"] = (
        totals["absolute_error"] / totals["count"]
    )
    totals["rmse"] = np.sqrt(
        totals["squared_error"] / totals["count"]
    )

    r.tg.write_csv(
        totals,
        ctx["reports"] / "confirmation_metrics.csv",
    )

    example_frame = pd.DataFrame(examples)

    r.tg.write_csv(
        example_frame,
        ctx["reports"]
        / "confirmation_price_examples.csv",
    )

    descriptors = [
        {
            "id": f"selected_{h}",
            "horizon": h,
        }
        for h in ("30s", "60s")
    ]

    primary, sensitivity = infer(
        ctx, daily, descriptors, "confirmation", 2
    )

    check(ctx, built=True)

    lean = totals.rename(columns={
        "horizon": "Horizon",
        "model": "Model",
        "count": "Forecasts",
        "mae": "MAE (pips)",
        "rmse": "RMSE (pips)",
    })[[
        "Horizon",
        "Model",
        "Forecasts",
        "MAE (pips)",
        "RMSE (pips)",
    ]]

    return (
        lean, primary, sensitivity, example_frame
    )

def finish(ctx):
    check(ctx, built=True)
    reports = ctx["reports"]
    checks = {}

    for filename, index in (
        ("feature_checks.csv", "month"),
        ("omission_checks.csv", "month"),
        ("confirmation_checks.csv", "month"),
    ):
        frame = pd.read_csv(reports / filename)
        checks[filename] = int(
            frame.drop(columns=index).to_numpy().sum()
        )

    evidence = r.tg.fx.load_json(
        reports / "synthetic_checks.json"
    )

    assert evidence["passed"]
    assert evidence["fingerprint"] == ctx["fingerprint"]
    assert all(
        v == 0 for v in checks.values()
    ), checks

    fits = pd.read_csv(
        reports / "omission_fit_summary.csv"
    )
    assert len(fits) == 12

    selected = r.tg.fx.load_json(
        reports / "selected_specs.json"
    )

    summary = {
        "notebook": "06_gradient_boosted_regression.ipynb",
        "experiment": (
            "group removal significance and combined confirmation"
        ),
        "run_id": ctx["run_id"],
        "fingerprint": ctx["fingerprint"],
        "reference_run_id": REFERENCE_RUN,
        "reference_fingerprint": REFERENCE_FP,
        "training": "all of 2018",
        "early_stopping": "January-March 2019",
        "group_test_period": "October-December 2019",
        "confirmation_period": "January-March 2020",
        "omission_comparisons": 12,
        "selected_removals": {
            v["horizon"]: v["removed_groups"]
            for v in selected
        },
        "primary_loss": "MAE; regression_l1",
        "secondary_metric": "RMSE",
        "bootstrap_draws": ctx["config"]["resamples"],
        "primary_block_active_dates": (
            ctx["config"]["primary_block_days"]
        ),
        "sensitivity_block_active_dates": (
            ctx["config"]["sensitivity_block_days"]
        ),
        "multiplicity": ctx["config"]["multiplicity"],
        "nonzero_machine_checks": sum(checks.values()),
        "synthetic_checks_passed": True,
        "confirmation_forecasts_include_unscorable_rows": True,
        "final_2026_period_accessed": False,
        "human_review": "pending",
        "model_promoted": False,
        "uncertainty": ctx["config"]["uncertainty"],
        "source_stability_check": (
            "file metadata and checkpoint/artifact hashes; "
            "no bulk content hashes"
        ),
    }

    r.tg.write_json(
        reports / "execution_summary.json", summary
    )

    marker = (
        f"notebook06_group_tests_{ctx['run_id']}"
    )

    text = (
        f"\n- Experiment: {ctx['run_id']}; "
        f"fingerprint {ctx['fingerprint']}.\n"
        "- Raymond Fendi requested input-removal tests "
        "to investigate MAE improvement.\n"
        "- Agreed: 21-input reference, six grouped omissions "
        "at each horizon; 2018 training and January-March "
        "2019 stopping remain fixed.\n"
        "- Q4 2019 selects removals; Q1 2020 checks the "
        "combined candidate. 2026 remains reserved.\n"
        "- MAE is primary, RMSE secondary. "
        "Undefined inputs remain missing.\n"
        "- Five-active-date paired bootstrap is primary; "
        "one-date sensitivity; Holm correction across 12 "
        "tests and a separate two-test confirmation family.\n"
        "- Intervals are pointwise and approximate, "
        "conditional on fixed models and the evaluation "
        "quarter. Inconclusive does not mean irrelevant.\n"
        "- Selected groups: "
        + json.dumps(summary["selected_removals"])
        + ".\n"
        "- Execution checks passed. Human interpretation "
        "and any model promotion remain pending; no upstream "
        "definitions or models were replaced.\n"
    )

    for path in (
        ctx["root"] / "CONTRIBUTIONS.md",
        ctx["root"] / "docs" / "design_decisions.md",
        ctx["root"] / "docs" / "project_state.md",
    ):
        r.util.write_section(path, marker, text)

    return summary
'''

In [32]:
# Check and save this experiment's dedicated helper.
GROUP_SOURCE = "\n\n".join(
    TEST_PARTS[name].strip()
    for name in (
        "core",
        "inputs",
        "models",
        "statistics",
        "confirmation",
    )
) + "\n"

tree = ast.parse(GROUP_SOURCE)

definitions = [
    node.name
    for node in tree.body
    if isinstance(node, ast.FunctionDef)
]
assert len(definitions) == len(set(definitions)), (
    "Duplicate helper functions."
)

compile(GROUP_SOURCE, "group_significance.py", "exec")

GROUP_HELPER = (
    ROOT / "functions" / "regression"
    / "group_significance.py"
)

if GROUP_HELPER.exists():
    assert (
        GROUP_HELPER.read_text(encoding="utf-8")
        == GROUP_SOURCE
    ), (
        "This helper already has different content. "
        "Keep it for review before replacing it."
    )
else:
    GROUP_HELPER.write_text(
        GROUP_SOURCE,
        encoding="utf-8",
        newline="\n",
    )

importlib.invalidate_caches()

from functions.regression import group_significance as group_tests
group_tests = importlib.reload(group_tests)

print("Group-test helper ready.")

Group-test helper ready.


In [33]:
# Verify existing artifacts and run hand-checkable synthetic checks.
GROUP_CTX = group_tests.prepare(
    ROOT,
    GROUP_RUN_ID,
    memory=GROUP_MEMORY,
    threads=GROUP_THREADS,
    working_gb=GROUP_WORKING_GB,
    resamples=GROUP_RESAMPLES,
    alpha=GROUP_ALPHA,
    block_days=GROUP_BLOCK_DAYS,
    sensitivity_days=GROUP_SENSITIVITY_DAYS,
    seed=GROUP_SEED,
)

GROUP_EVIDENCE = group_tests.synthetic_checks()
GROUP_EVIDENCE["fingerprint"] = GROUP_CTX["fingerprint"]

group_tests.r.tg.write_json(
    GROUP_CTX["reports"] / "synthetic_checks.json",
    GROUP_EVIDENCE,
)

display(pd.DataFrame({
    "Check": GROUP_EVIDENCE["checks"].keys(),
    "Passed": GROUP_EVIDENCE["checks"].values(),
}))

print("Synthetic checks passed; reference artifacts verified.")

,Check,Passed
0,holm_known_example,True
1,paired_common_draws_and_constant_difference,True
2,observation_weighted_daily_ratio,True
3,identical_forecasts_are_not_significant,True
4,quarter_information_cutoff,True


Synthetic checks passed; reference artifacts verified.


In [34]:
# EXPENSIVE: build October 2019–March 2020 inputs in monthly batches.
# Valid saved months are checked and reused after an interruption.
display(
    group_tests.build(GROUP_CTX).round(2)
)

2019-10: inputs saved.
2019-11: inputs saved.
2019-12: inputs saved.
2020-01: inputs saved.
2020-02: inputs saved.
2020-03: inputs saved.


,Horizon,Estimated working GiB,Budget GiB
0,30s,1.69,3.0
1,60s,1.69,3.0


In [35]:
# EXPENSIVE: fit 12 omissions.
# Every fitted omission uses the full agreed 2018 training period.
# Completed models are checked and reused after an interruption.
GROUP_MODELS = group_tests.fit_omissions(GROUP_CTX)

display(pd.read_csv(
    GROUP_CTX["reports"] / "omission_fit_summary.csv"
))

30s | omit Price changes: ready.
30s | omit Displayed sizes: ready.
30s | omit Spread: ready.
30s | omit MA distances: ready.
30s | omit Sample entropy: ready.
30s | omit Conditional entropy: ready.
60s | omit Price changes: ready.
60s | omit Displayed sizes: ready.
60s | omit Spread: ready.
60s | omit MA distances: ready.
60s | omit Sample entropy: ready.
60s | omit Conditional entropy: ready.


,Group,Horizon,Inputs,Trees,Training examples
0,Price changes,30s,15,254,1960094
1,Displayed sizes,30s,19,190,1960094
2,Spread,30s,20,319,1960094
3,MA distances,30s,19,387,1960094
4,Sample entropy,30s,16,378,1960094
5,Conditional entropy,30s,16,305,1960094
6,Price changes,60s,15,108,1959311
7,Displayed sizes,60s,19,136,1959311
8,Spread,60s,20,240,1959311
9,MA distances,60s,19,176,1959311


In [36]:
# Test paired forecast errors on October–December 2019.
GROUP_DAILY = group_tests.score_omissions(
    GROUP_CTX, GROUP_MODELS
)

GROUP_PRIMARY, GROUP_SENSITIVITY = group_tests.infer(
    GROUP_CTX,
    GROUP_DAILY,
    GROUP_CTX["config"]["hypotheses"],
    stem="omissions",
    family_size=12,
)

print("Primary result: negative Delta MAE favors removing the group.")
print(
    "Intervals are pointwise; the evidence column "
    "uses Holm-adjusted p-values."
)

display(
    group_tests.inference_display(GROUP_PRIMARY)
)

print("One-active-date sensitivity — does not control selection:")

display(
    group_tests.inference_display(GROUP_SENSITIVITY)
)

2019-10: all 12 comparisons scored.
2019-11: all 12 comparisons scored.
2019-12: all 12 comparisons scored.
Primary result: negative Delta MAE favors removing the group.
Intervals are pointwise; the evidence column uses Holm-adjusted p-values.


,Group,Horizon,Delta MAE (pips),95% interval,Adjusted p,Evidence
0,Price changes,30s,+0.0000359,"[-0.0000305, +0.0000876]",1.0000,Inconclusive
1,Displayed sizes,30s,+0.0003510,"[+0.0002810, +0.0004061]",0.0024,Group helps
2,Spread,30s,+0.0000164,"[+0.0000077, +0.0000268]",0.0140,Group helps
3,MA distances,30s,+0.0000285,"[-0.0000110, +0.0000736]",1.0000,Inconclusive
4,Sample entropy,30s,-0.0000105,"[-0.0000283, +0.0000119]",1.0000,Inconclusive
5,Conditional entropy,30s,-0.0000091,"[-0.0000301, +0.0000128]",1.0000,Inconclusive
6,Price changes,60s,+0.0001196,"[-0.0000124, +0.0002488]",0.6966,Inconclusive
7,Displayed sizes,60s,+0.0002408,"[+0.0001566, +0.0002974]",0.0024,Group helps
8,Spread,60s,+0.0000091,"[-0.0000131, +0.0000299]",1.0000,Inconclusive
9,MA distances,60s,-0.0000329,"[-0.0001534, +0.0000928]",1.0000,Inconclusive


One-active-date sensitivity — does not control selection:


,Group,Horizon,Delta MAE (pips),95% interval,Adjusted p,Evidence
0,Price changes,30s,+0.0000359,"[-0.0000246, +0.0000959]",1.0000,Inconclusive
1,Displayed sizes,30s,+0.0003510,"[+0.0002848, +0.0004155]",0.0024,Group helps
2,Spread,30s,+0.0000164,"[+0.0000050, +0.0000266]",0.0520,Inconclusive
3,MA distances,30s,+0.0000285,"[-0.0000110, +0.0000697]",1.0000,Inconclusive
4,Sample entropy,30s,-0.0000105,"[-0.0000320, +0.0000090]",1.0000,Inconclusive
5,Conditional entropy,30s,-0.0000091,"[-0.0000275, +0.0000110]",1.0000,Inconclusive
6,Price changes,60s,+0.0001196,"[-0.0000146, +0.0002570]",0.6930,Inconclusive
7,Displayed sizes,60s,+0.0002408,"[+0.0001811, +0.0003010]",0.0024,Group helps
8,Spread,60s,+0.0000091,"[-0.0000171, +0.0000342]",1.0000,Inconclusive
9,MA distances,60s,-0.0000329,"[-0.0001379, +0.0000784]",1.0000,Inconclusive


In [37]:
# Fix the combined candidate before reading confirmation outcomes.
# Only significant MAE improvements qualify; inconclusive groups stay.
# EXPENSIVE if a combined candidate requires a new fit.
GROUP_SELECTED, GROUP_RECIPES = group_tests.select_and_fit(
    GROUP_CTX,
    GROUP_PRIMARY,
)

display(GROUP_RECIPES)

print("Recipes saved. No model has been promoted.")

,Horizon,Removal candidate groups,Remaining inputs,Candidate type
0,30s,None,21,reference_copy
1,60s,None,21,reference_copy


Recipes saved. No model has been promoted.


In [38]:
# Fresh confirmation: January–March 2020.
# Compare matching eligible forecasts.
# Export both horizons without filtering by future-label validity.
(
    GROUP_METRICS,
    GROUP_CONFIRMATION,
    GROUP_CONFIRMATION_SENSITIVITY,
    GROUP_EXAMPLES,
) = group_tests.confirm(
    GROUP_CTX,
    GROUP_SELECTED,
)

display(GROUP_METRICS.round({
    "MAE (pips)": 7,
    "RMSE (pips)": 7,
}))

display(
    group_tests.inference_display(GROUP_CONFIRMATION)
)

print("One-active-date confirmation sensitivity:")

display(
    group_tests.inference_display(
        GROUP_CONFIRMATION_SENSITIVITY
    )
)

print("Four consecutive issue times, where valid, at each horizon:")

display(GROUP_EXAMPLES.round({
    "Current mid": 6,
    "Reference forecast": 6,
    "Candidate forecast": 6,
    "Observed future mid": 6,
}))

2020-01: confirmation scored; all-grid forecasts saved.
2020-02: confirmation scored; all-grid forecasts saved.
2020-03: confirmation scored; all-grid forecasts saved.


,Horizon,Model,Forecasts,MAE (pips),RMSE (pips)
0,30s,Reference 21,483637,0.771636,1.403941
1,30s,Selected candidate,483637,0.771636,1.403941
2,30s,Zero change,483637,0.771388,1.402968
3,60s,Reference 21,483445,1.107780,1.982425
4,60s,Selected candidate,483445,1.107780,1.982425
5,60s,Zero change,483445,1.107146,1.980258


,Group,Horizon,Delta MAE (pips),95% interval,Adjusted p,Evidence
0,Selected composite,30s,+0.0000000,"[+0.0000000, +0.0000000]",1.0,Inconclusive
1,Selected composite,60s,+0.0000000,"[+0.0000000, +0.0000000]",1.0,Inconclusive


One-active-date confirmation sensitivity:


,Group,Horizon,Delta MAE (pips),95% interval,Adjusted p,Evidence
0,Selected composite,30s,+0.0000000,"[+0.0000000, +0.0000000]",1.0,Inconclusive
1,Selected composite,60s,+0.0000000,"[+0.0000000, +0.0000000]",1.0,Inconclusive


Four consecutive issue times, where valid, at each horizon:


,Athens time,Horizon,Current mid,Reference forecast,Candidate forecast,Observed future mid,Interpolated outcome
0,06 Jan 2020 10:00:00,30s,1.116470,1.116470,1.116470,1.116535,False
1,06 Jan 2020 10:00:10,30s,1.116525,1.116524,1.116524,1.116545,False
2,06 Jan 2020 10:00:20,30s,1.116495,1.116495,1.116495,1.116560,False
3,06 Jan 2020 10:00:30,30s,1.116535,1.116535,1.116535,1.116500,False
4,06 Jan 2020 10:00:00,60s,1.116470,1.116469,1.116469,1.116500,False
5,06 Jan 2020 10:00:10,60s,1.116525,1.116523,1.116523,1.116515,False
6,06 Jan 2020 10:00:20,60s,1.116495,1.116496,1.116496,1.116555,False
7,06 Jan 2020 10:00:30,60s,1.116535,1.116535,1.116535,1.116620,False


In [39]:
# Record execution and Raymond's design contribution.
# Human review remains pending.
GROUP_SUMMARY = group_tests.finish(GROUP_CTX)

print("NOTEBOOK_06_GROUP_TESTS_EXECUTION_SUMMARY")
print(json.dumps(GROUP_SUMMARY, indent=2))

print("\nReports:", GROUP_CTX["reports"])
print(
    "STOP HERE. Return the summary "
    "and the tables from Cells 35–37."
)

NOTEBOOK_06_GROUP_TESTS_EXECUTION_SUMMARY
{
  "notebook": "06_gradient_boosted_regression.ipynb",
  "experiment": "group removal significance and combined confirmation",
  "run_id": "regression_06_group_tests_v1",
  "fingerprint": "f761bd52ecbf0f47277b5b617626c45a75644d7519394a01c3887bb8fc9138b1",
  "reference_run_id": "regression_06_short_ma_v1",
  "reference_fingerprint": "650d88ee4e8ae4272136e4e9936f039aee522d00a6482c8e0bc93292cf4c4faf",
  "training": "all of 2018",
  "early_stopping": "January-March 2019",
  "group_test_period": "October-December 2019",
  "confirmation_period": "January-March 2020",
  "omission_comparisons": 12,
  "selected_removals": {
    "30s": [],
    "60s": []
  },
  "primary_loss": "MAE; regression_l1",
  "secondary_metric": "RMSE",
  "bootstrap_draws": 9999,
  "primary_block_active_dates": 5,
  "sensitivity_block_active_dates": 1,
  "multiplicity": "Holm across 12 primary tests; separate 2-test confirmation family",
  "nonzero_machine_checks": 0,
  "syntheti

In [40]:
# Sizes-and-spread variation
# - Two LightGBM models: 30-second and 60-second price changes.
# - Only bid size, ask size and spread enter these models.
# - Train on all of 2018; stop on January-March 2019.
# - Compare on April-June 2020 with the 21-input models and zero change.
# - Q4 2019 and Q1 2020 already informed this choice.
# - MAE is primary; RMSE is secondary. Keep 2026 reserved.
# - This experiment does not establish that omitted inputs are irrelevant.

In [41]:
# Settings. Run these new cells in order.
from pathlib import Path
import ast
import importlib
import json
import sys

import pandas as pd
from IPython.display import display

SS_ROOT = Path(
    r"C:\Users\Ijzd4\Desktop\Inventory-Aware FX Pricing and Hedging Simulator"
)

assert (
    SS_ROOT / "functions" / "regression" / "group_significance.py"
).is_file()

if str(SS_ROOT) not in sys.path:
    sys.path.insert(0, str(SS_ROOT))

SS_RUN_ID = "regression_06_sizes_spread_v1"
SS_MEMORY = "2GB"
SS_THREADS = 2
SS_WORKING_GB = 3.0

SS_PARTS = {}

In [42]:
# Helper: verify inputs and prepare a separate experiment.
SS_PARTS["core"] = r'''
import copy
import gc
import hashlib
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa

from functions.regression import fx_regression as r
from functions.regression import group_significance as g

SOURCE_RUN = "regression_06_group_tests_v1"
SOURCE_FP = "f761bd52ecbf0f47277b5b617626c45a75644d7519394a01c3887bb8fc9138b1"

COLUMNS = [
    "bid_size_million_eur",
    "ask_size_million_eur",
    "spread_pips",
]

MONTHS = ["2020-04", "2020-05", "2020-06"]
START = r.tg.ns("2020-04-01")
END = r.tg.ns("2020-07-01")

def eligible(table, h):
    t = r.tg.get(table, "issue_ns")
    info = r.tg.get(
        table, f"label_available_ns_{h}", 0
    )

    return (
        (t >= START)
        & (t < END)
        & r.tg.get(table, f"usable_{h}")
        & (info >= t)
        & (info < END)
    )

def check(ctx, built=False):
    g.check(ctx, built=built)
    assert (
        r.util.sha(__file__)
        == ctx["config"]["variation_helper_sha"]
    )

def synthetic_checks():
    cut = END

    table = pa.table({
        "issue_ns": [
            cut - r.STEP,
            cut - r.STEP,
            START - r.STEP,
        ],
        "usable_30s": [True, True, True],
        "label_available_ns_30s": [
            cut, cut - 1, START
        ],
    })

    assert np.array_equal(
        eligible(table, "30s"),
        [False, True, False],
    )

    base = g.synthetic_checks()
    assert base["passed"]

    base["checks"][
        "fresh_quarter_information_cutoff"
    ] = True

    base["checks"]["exactly_three_named_inputs"] = (
        len(COLUMNS) == 3
        and len(set(COLUMNS)) == 3
    )

    base["passed"] = all(base["checks"].values())
    return base

def prepare(
    root, run_id,
    memory="2GB", threads=2, working_gb=3.0,
):
    root = Path(root).resolve()

    assert run_id and Path(run_id).name == run_id
    assert run_id not in (
        SOURCE_RUN,
        g.REFERENCE_RUN,
        "regression_06_v1",
    )

    source_dir = (
        root / "outputs" / "features" / SOURCE_RUN
    )
    source_cfg_path = (
        source_dir / "configuration.json"
    )
    source_reports = (
        root / "reports" / "tables"
        / "regression" / SOURCE_RUN
    )

    old = r.tg.fx.load_json(source_cfg_path)
    summary = r.tg.fx.load_json(
        source_reports / "execution_summary.json"
    )

    assert summary["fingerprint"] == SOURCE_FP

    assert hashlib.sha256(
        json.dumps(
            old,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest() == SOURCE_FP

    assert summary["nonzero_machine_checks"] == 0
    assert (
        old["test_helper_sha"]
        == r.util.sha(g.__file__)
    )

    ref_dir = (
        root / "outputs" / "features"
        / g.REFERENCE_RUN
    )
    ref_cfg = r.tg.fx.load_json(
        ref_dir / "configuration.json"
    )

    original = (
        root / "outputs" / "features"
        / "regression_06_v1"
    )
    original_cfg_path = (
        original / "configuration.json"
    )

    assert (
        r.util.sha(original_cfg_path)
        == ref_cfg["parent_config_sha"]
    )

    original_cfg = r.tg.fx.load_json(
        original_cfg_path
    )

    assert {
        k: list(v)
        for k, v in r.PERIODS.items()
    } == original_cfg["periods_ns"]

    training = {
        "root": root,
        "feature_dir": original,
        "reports": (
            root / "reports" / "tables"
            / "regression" / "regression_06_v1"
        ),
        "months": [
            str(m)
            for m in pd.period_range(
                "2018-01", "2019-06", freq="M"
            )
        ],
        "config": original_cfg,
        "fingerprint": ref_cfg["parent_fingerprint"],
        "memory": memory,
        "threads": threads,
    }

    r.check_evidence(training)

    artifacts = copy.deepcopy(
        old["reference_artifacts"]
    )

    for path in (
        source_cfg_path,
        source_reports / "execution_summary.json",
        source_reports / "omissions_primary_inference.csv",
        source_reports / "confirmation_metrics.csv",
    ):
        artifacts.append({
            "path": path.relative_to(root).as_posix(),
            "sha": r.util.sha(path),
        })

    assert len({
        v["path"] for v in artifacts
    }) == len(artifacts)

    inputs = copy.deepcopy(old["inputs"])

    sample = (
        root / "data" / "processed"
        / r.SAMPLE_RUN
    )

    raw = {
        v["year_month"]: v
        for v in r.tg.fx.load_json(
            sample / "configuration.json"
        )["inputs"]
    }

    additions = []

    for m in MONTHS:
        cp = sample / "_checkpoints" / f"{m}.json"
        saved = r.tg.fx.load_json(cp)

        assert (
            saved["fingerprint"]
            == r.EXPECTED["sampling"]
        )
        assert all(
            v == 0 for v in saved["checks"].values()
        )

        for kind in (
            "sampled_10s", "candles_3min"
        ):
            additions.append((
                kind,
                sample / kind / f"{m}.parquet",
                saved["outputs"][kind],
                cp,
            ))

        cp = (
            root / "data" / "processed"
            / "etl_01_v3" / "_checkpoints"
            / f"observed_{m}.json"
        )

        assert (
            r.util.sha(cp)
            == raw[m]["checkpoint_sha256"]
        )

        additions.append((
            "observed",
            root / raw[m]["path"],
            raw[m],
            cp,
        ))

    folders = {
        "targets": (
            root / "data" / "processed"
            / r.TARGET_RUN
        ),
        "entropy": (
            root / "outputs" / "features"
            / "entropy" / r.ENTROPY_RUN
        ),
    }

    for day in pd.date_range(
        "2020-04-01", "2020-06-30", freq="D"
    ):
        date = day.strftime("%Y-%m-%d")

        for kind, folder in folders.items():
            cp = (
                folder / "_checkpoints"
                / f"{date}.json"
            )
            saved = r.tg.fx.load_json(cp)

            assert (
                saved["fingerprint"]
                == r.EXPECTED[kind]
            )
            assert all(
                v == 0
                for v in saved["checks"].values()
            )

            additions.append((
                kind,
                folder / f"{date}.parquet",
                saved["output"],
                cp,
            ))

    for kind, path, expected, cp in additions:
        info = r.util.file_info(path)

        assert all(
            info[k] == expected[k] for k in info
        ), str(path)

        inputs.append({
            "kind": kind,
            "path": path.relative_to(root).as_posix(),
            **info,
            "checkpoint": (
                cp.relative_to(root).as_posix()
            ),
            "checkpoint_sha": r.util.sha(cp),
        })

    assert len({
        (v["kind"], v["path"])
        for v in inputs
    }) == len(inputs)

    config = {
        k: copy.deepcopy(old[k])
        for k in (
            "upstream",
            "contracts",
            "dependencies",
            "versions",
            "periods_ns",
            "params",
            "maximum_trees",
            "early_stopping_rounds",
            "model_features",
            "model_feature_groups",
            "test_helper_sha",
            "reference_fingerprint",
        )
    }

    assert (
        config["params"]["objective"]
        == "regression_l1"
    )
    assert len(config["model_features"]) == 21
    assert all(
        c in config["model_features"]
        for c in COLUMNS
    )

    config.update({
        "definition": "sizes_spread_variation_v1",
        "run_id": run_id,
        "parent_run_id": SOURCE_RUN,
        "parent_fingerprint": SOURCE_FP,
        "parent_config_sha": r.util.sha(
            source_cfg_path
        ),
        "reference_run": g.REFERENCE_RUN,
        "reference_artifacts": artifacts,
        "candidate_features": COLUMNS,
        "inputs": inputs,
        "variation_helper_sha": r.util.sha(__file__),
        "evaluation_periods_ns": {
            "fresh_development": [START, END]
        },
        "architecture_information_cutoff_ns": START - 1,
        "model_available_ns": START,
        "inference_latency_ns": 0,
        "comparison_scope": (
            "April-June 2020 fresh development comparison"
        ),
        "selection": (
            "User-requested fixed sizes/spread recipe; "
            "no automatic promotion"
        ),
        "missing_features": (
            "native NaN; valid target rows retained"
        ),
        "primary_block_days": 5,
        "sensitivity_block_days": 1,
        "resamples": 9999,
        "alpha": 0.05,
        "bootstrap_seed": 1729,
        "multiplicity": (
            "Holm across four tests: "
            "two horizons by two benchmarks"
        ),
        "interval": (
            "pointwise basic block-bootstrap; "
            "not simultaneous"
        ),
        "uncertainty": (
            "approximate and conditional on fixed "
            "models and this quarter"
        ),
        "forecast_estimate": (
            "conditional median price-change "
            "point forecast"
        ),
    })

    config = json.loads(json.dumps(config))

    fp = hashlib.sha256(
        json.dumps(
            config,
            sort_keys=True,
            separators=(",", ":"),
        ).encode()
    ).hexdigest()

    feature_dir = (
        root / "outputs" / "features" / run_id
    )
    model_dir = (
        root / "outputs" / "models" / run_id
    )
    forecast_dir = (
        root / "outputs" / "forecasts" / run_id
    )
    reports = (
        root / "reports" / "tables"
        / "regression" / run_id
    )

    for folder in (
        feature_dir / "_checkpoints",
        model_dir,
        reports,
        forecast_dir / "reference",
        forecast_dir / "candidate",
    ):
        folder.mkdir(
            parents=True, exist_ok=True
        )

    path = feature_dir / "configuration.json"

    if path.exists():
        previous = r.tg.fx.load_json(path)
        changed = [
            k for k in config
            if previous.get(k) != config[k]
        ]

        assert not changed, (
            f"Changed configuration {changed}; "
            "use a new RUN_ID."
        )
    else:
        r.tg.write_json(path, config)

    ctx = {
        "root": root,
        "feature_dir": feature_dir,
        "model_dir": model_dir,
        "forecast_dir": forecast_dir,
        "reports": reports,
        "config": config,
        "fingerprint": fp,
        "run_id": run_id,
        "months": MONTHS,
        "training": training,
        "reference_models": (
            root / "outputs" / "models"
            / g.REFERENCE_RUN
        ),
        "reference_features": {
            "feature_dir": ref_dir,
            "months": [
                "2019-07", "2019-08", "2019-09"
            ],
            "fingerprint": g.REFERENCE_FP,
        },
        "memory": memory,
        "threads": threads,
        "working_gb": working_gb,
    }

    check(ctx)

    r.tg.write_csv(
        pd.DataFrame(inputs),
        reports / "input_inventory.csv",
    )
    return ctx
'''

In [43]:
# Helper: build monthly inputs and train exactly two models.
SS_PARTS["models"] = r'''
def build(ctx):
    check(ctx)

    evidence = r.tg.fx.load_json(
        ctx["reports"] / "synthetic_checks.json"
    )
    assert evidence["passed"]
    assert (
        evidence["fingerprint"]
        == ctx["fingerprint"]
    )

    records = []

    with duckdb.connect(config={
        "memory_limit": ctx["memory"],
        "threads": str(ctx["threads"]),
    }) as con:
        for m in MONTHS:
            path = (
                ctx["feature_dir"] / f"{m}.parquet"
            )
            cp = (
                ctx["feature_dir"]
                / "_checkpoints" / f"{m}.json"
            )

            if cp.exists():
                saved = r.tg.fx.load_json(cp)

                assert (
                    saved["fingerprint"]
                    == ctx["fingerprint"]
                )
                assert (
                    r.util.file_info(path)
                    == saved["output"]
                )
                assert all(
                    v == 0
                    for v in saved["checks"].values()
                )
            else:
                table, checks, _, _ = r.make_month(
                    ctx, m, con
                )

                counts = {
                    "month": m,
                    "rows": table.num_rows,
                }
                counts.update({
                    f"valid_{h}": int(
                        eligible(table, h).sum()
                    )
                    for h in ("30s", "60s")
                })

                r.tg.write_parquet(table, path)

                saved = {
                    "fingerprint": ctx["fingerprint"],
                    "checks": checks,
                    "counts": counts,
                    "output": r.util.file_info(path),
                }
                r.tg.write_json(cp, saved)

            records.append(saved)
            print(f"{m}: inputs ready.")

    check(ctx, built=True)

    counts = pd.DataFrame([
        v["counts"] for v in records
    ])
    checks = pd.DataFrame([
        {
            "month": v["counts"]["month"],
            **v["checks"],
        }
        for v in records
    ])

    r.tg.write_csv(
        counts,
        ctx["reports"] / "feature_counts.csv",
    )
    r.tg.write_csv(
        checks,
        ctx["reports"] / "feature_checks.csv",
    )
    return counts

def fit(ctx):
    check(ctx, built=True)
    g.memory_check(ctx)

    references = g.load_reference(ctx)
    bundles, rows = {}, []

    for h in ("30s", "60s"):
        case = {
            "id": f"sizes_spread_{h}",
            "horizon": h,
            "features": COLUMNS,
            "group": "Displayed sizes and spread",
            "ready_ns": START,
        }

        bundles[h] = g.fit_case(
            ctx, case, references
        )
        meta = bundles[h]["meta"]

        assert meta["features"] == COLUMNS
        assert (
            bundles[h]["model"].feature_name()
            == COLUMNS
        )
        assert (
            bundles[h]["model"].num_feature()
            == 3
        )

        rows.append({
            "Horizon": h,
            "Inputs": 3,
            "Training examples": meta[
                "training_rows"
            ],
            "Early-stopping examples": meta[
                "early_stopping_rows"
            ],
            "Trees": meta["best_iteration"],
        })

        print(f"{h}: three-input model ready.")
        gc.collect()

    check(ctx, built=True)

    report = pd.DataFrame(rows)

    r.tg.write_csv(
        report,
        ctx["reports"] / "model_fit_summary.csv",
    )
    return bundles, report
'''

In [44]:
# Helper: compare forecasts, export results and record contributions.
SS_PARTS["evaluation"] = r'''
def evaluate(ctx, bundles):
    check(ctx, built=True)
    references = g.load_reference(ctx)

    monthly, daily, checks, examples = [], [], [], []

    descriptors = [
        {
            "id": f"{h}_vs_{key}",
            "horizon": h,
            "group": label,
        }
        for h in ("30s", "60s")
        for key, label in (
            ("reference", "Vs reference 21"),
            ("zero", "Vs zero change"),
        )
    ]

    for m in MONTHS:
        with duckdb.connect(config={
            "memory_limit": ctx["memory"],
            "threads": str(ctx["threads"]),
        }) as con:
            table = r.read(
                con,
                [
                    ctx["feature_dir"]
                    / f"{m}.parquet"
                ],
                [
                    *r.BASE,
                    *r.FEATURES,
                    "input_information_max_ns",
                ],
                "issue_ns",
            )

        times = r.tg.get(table, "issue_ns")
        dates = np.asarray(
            pd.to_datetime(
                times, unit="ns", utc=True
            ).tz_convert(r.TZ).strftime(
                "%Y-%m-%d"
            )
        )

        month_checks = {
            "month": m,
            "late_inputs": int(
                np.count_nonzero(
                    r.tg.get(
                        table,
                        "input_information_max_ns",
                    ) > times
                )
            ),
            "outside_comparison_period": int(
                np.count_nonzero(
                    (times < START)
                    | (times >= END)
                )
            ),
        }

        saved_predictions = {
            "reference": {},
            "candidate": {},
        }

        for h in ("30s", "60s"):
            candidate = bundles[h]
            reference = references[h]

            assert (
                candidate["meta"]["features"]
                == COLUMNS
            )

            values = {
                "Sizes + spread": g.predict(
                    candidate, table
                ),
                "Reference 21": g.predict(
                    reference, table
                ),
                "Zero change": np.zeros(
                    table.num_rows
                ),
            }

            saved_predictions["reference"][h] = (
                values["Reference 21"]
            )
            saved_predictions["candidate"][h] = (
                values["Sizes + spread"]
            )

            for key, model in (
                ("candidate", candidate),
                ("reference", reference),
            ):
                meta = model["meta"]

                assert (
                    meta["training_cutoff_ns"]
                    < meta["model_available_ns"]
                )

                month_checks[
                    f"{key}_{h}_late_model"
                ] = int(np.count_nonzero(
                    (
                        times
                        < meta["model_available_ns"]
                    )
                    | (
                        times
                        <= meta["training_cutoff_ns"]
                    )
                ))

            mask = eligible(table, h)
            actual = r.tg.get(
                table,
                f"target_change_{h}_pips",
                np.nan,
            )[mask]

            assert len(actual)
            assert np.isfinite(actual).all()

            errors = {
                key: prediction[mask] - actual
                for key, prediction in values.items()
            }

            for key, error in errors.items():
                monthly.append({
                    "month": m,
                    "horizon": h,
                    "model": key,
                    "count": len(actual),
                    "absolute_error": float(
                        np.abs(error).sum()
                    ),
                    "squared_error": float(
                        np.square(error).sum()
                    ),
                })

            days, index = np.unique(
                dates[mask],
                return_inverse=True,
            )
            counts = np.bincount(index)

            for key, benchmark in (
                ("reference", "Reference 21"),
                ("zero", "Zero change"),
            ):
                sums = np.bincount(
                    index,
                    weights=(
                        np.abs(
                            errors["Sizes + spread"]
                        )
                        - np.abs(
                            errors[benchmark]
                        )
                    ),
                )

                daily.extend({
                    "date": day,
                    "case_id": f"{h}_vs_{key}",
                    "horizon": h,
                    "count": int(n),
                    "difference_sum": float(s),
                } for day, n, s in zip(
                    days, counts, sums
                ))

            if m == "2020-04":
                first = r.tg.ns(
                    "2020-04-06T10:00:00"
                )
                take = np.flatnonzero(
                    mask
                    & (times >= first)
                    & (
                        times
                        < first + 40 * r.NS
                    )
                )

                mid = r.tg.get(
                    table, "reference_mid"
                )
                future = r.tg.get(
                    table,
                    f"target_future_mid_{h}",
                    np.nan,
                )
                synthetic = r.tg.get(
                    table,
                    f"endpoint_is_interpolated_{h}",
                    False,
                )

                for i in take:
                    examples.append({
                        "Athens time": pd.Timestamp(
                            times[i],
                            unit="ns",
                            tz="UTC",
                        ).tz_convert(
                            r.TZ
                        ).strftime(
                            "%d %b %Y %H:%M:%S"
                        ),
                        "Horizon": h,
                        "Current mid": mid[i],
                        "Predicted change (pips)": (
                            values[
                                "Sizes + spread"
                            ][i]
                        ),
                        "Predicted EURUSD": (
                            mid[i]
                            + r.PIP
                            * values[
                                "Sizes + spread"
                            ][i]
                        ),
                        "Observed future EURUSD": (
                            future[i]
                        ),
                        "Interpolated outcome": bool(
                            synthetic[i]
                        ),
                    })

        assert all(
            v == 0
            for k, v in month_checks.items()
            if k != "month"
        ), month_checks

        # Future-label validity never selects exported forecasts.
        for key, models in (
            ("reference", references),
            ("candidate", bundles),
        ):
            forecast = g.forecast_table(
                table,
                models,
                saved_predictions[key],
            )
            r.tg.write_parquet(
                forecast,
                ctx["forecast_dir"]
                / key / f"{m}.parquet",
            )

        checks.append(month_checks)
        print(
            f"{m}: scored both horizons; "
            "forecasts saved."
        )

    monthly = pd.DataFrame(monthly)
    daily = pd.DataFrame(daily)

    assert monthly.groupby(
        ["month", "horizon"]
    )["count"].nunique().eq(1).all()

    assert daily.groupby(
        ["date", "horizon"]
    )["count"].nunique().eq(1).all()

    totals = monthly.groupby(
        ["horizon", "model"],
        sort=False,
    )[[
        "count",
        "absolute_error",
        "squared_error",
    ]].sum().reset_index()

    totals["mae"] = (
        totals["absolute_error"]
        / totals["count"]
    )
    totals["rmse"] = np.sqrt(
        totals["squared_error"]
        / totals["count"]
    )

    for case in descriptors:
        h = case["horizon"]
        case_id = case["id"]

        benchmark = (
            "Reference 21"
            if case_id.endswith("reference")
            else "Zero change"
        )

        d = daily.loc[
            daily["case_id"].eq(case_id)
        ]
        scores = totals.loc[
            totals["horizon"].eq(h)
        ].set_index("model")

        assert int(d["count"].sum()) == int(
            scores.loc[benchmark, "count"]
        )

        assert np.isclose(
            d["difference_sum"].sum(),
            scores.loc[
                "Sizes + spread",
                "absolute_error",
            ]
            - scores.loc[
                benchmark,
                "absolute_error",
            ],
            rtol=0,
            atol=1e-7,
        )

    for name, frame in (
        ("performance_summary.csv", totals),
        ("monthly_scores.csv", monthly),
        ("paired_daily_losses.csv", daily),
        (
            "evaluation_checks.csv",
            pd.DataFrame(checks),
        ),
        (
            "price_examples.csv",
            pd.DataFrame(examples),
        ),
    ):
        r.tg.write_csv(
            frame, ctx["reports"] / name
        )

    primary, sensitivity = g.infer(
        ctx,
        daily,
        descriptors,
        stem="confirmation",
        family_size=4,
    )

    month_table = monthly.assign(
        mae=(
            monthly["absolute_error"]
            / monthly["count"]
        )
    ).pivot(
        index=["month", "horizon"],
        columns="model",
        values="mae",
    ).reset_index()

    month_table.columns.name = None
    month_table = month_table.rename(
        columns={
            "month": "Month",
            "horizon": "Horizon",
        }
    )

    r.tg.write_csv(
        month_table,
        ctx["reports"] / "monthly_mae.csv",
    )

    metrics = totals.rename(columns={
        "horizon": "Horizon",
        "model": "Model",
        "count": "Forecasts",
        "mae": "MAE (pips)",
        "rmse": "RMSE (pips)",
    })[[
        "Horizon",
        "Model",
        "Forecasts",
        "MAE (pips)",
        "RMSE (pips)",
    ]]

    check(ctx, built=True)

    return (
        metrics,
        month_table,
        primary,
        sensitivity,
        pd.DataFrame(examples),
    )

def finish(ctx):
    check(ctx, built=True)
    failures = {}

    for name in (
        "feature_checks.csv",
        "evaluation_checks.csv",
    ):
        frame = pd.read_csv(
            ctx["reports"] / name
        )
        failures[name] = int(
            frame.drop(
                columns="month"
            ).to_numpy().sum()
        )

    evidence = r.tg.fx.load_json(
        ctx["reports"]
        / "synthetic_checks.json"
    )

    assert evidence["passed"]
    assert (
        evidence["fingerprint"]
        == ctx["fingerprint"]
    )
    assert all(
        v == 0 for v in failures.values()
    ), failures

    report = {
        "notebook": (
            "06_gradient_boosted_regression.ipynb"
        ),
        "run_id": ctx["run_id"],
        "fingerprint": ctx["fingerprint"],
        "reference_run": g.REFERENCE_RUN,
        "candidate_features": COLUMNS,
        "horizons": ["30s", "60s"],
        "training": "all of 2018",
        "early_stopping": (
            "January-March 2019"
        ),
        "fresh_development_comparison": (
            "April-June 2020"
        ),
        "primary_metric": "MAE",
        "secondary_metric": "RMSE",
        "objective": "regression_l1",
        "comparisons": 4,
        "multiplicity": (
            ctx["config"]["multiplicity"]
        ),
        "nonzero_machine_checks": sum(
            failures.values()
        ),
        "synthetic_checks_passed": True,
        "forecasts_exported_without_future_label_filter": True,
        "final_2026_period_accessed": False,
        "model_promoted": False,
        "human_review": "pending",
        "uncertainty": (
            ctx["config"]["uncertainty"]
        ),
    }

    r.tg.write_json(
        ctx["reports"]
        / "execution_summary.json",
        report,
    )

    text = (
        f"\n- Run: {ctx['run_id']}; "
        f"fingerprint {ctx['fingerprint']}.\n"
        "- Raymond Fendi requested a two-model "
        "variation using only bid size, ask size "
        "and spread following the reported "
        "group-removal results.\n"
        "- This is a requested joint-removal "
        "experiment; the omitted groups have "
        "not been established as irrelevant.\n"
        "- Training remains all of 2018; stopping "
        "remains January-March 2019. Parameters, "
        "MAE objective and valid training rows "
        "remain fixed.\n"
        "- April-June 2020 is the fresh development "
        "comparison; Q4 2019 and Q1 2020 informed "
        "this recipe and are already used "
        "development evidence. 2026 remains reserved.\n"
        "- Compare both horizons with frozen "
        "21-input models and zero change; paired "
        "five-active-date bootstrap, one-date "
        "sensitivity, Holm family of four.\n"
        "- MAE is primary and RMSE secondary. "
        "Missing inputs remain missing. Forecasts "
        "retain availability and information cutoffs.\n"
        "- Execution checks passed. Results await "
        "human interpretation; no model was promoted "
        "and no upstream output was replaced.\n"
    )

    for path in (
        ctx["root"] / "CONTRIBUTIONS.md",
        ctx["root"] / "docs" / "design_decisions.md",
        ctx["root"] / "docs" / "project_state.md",
    ):
        r.util.write_section(
            path,
            f"notebook06_{ctx['run_id']}",
            text,
        )

    return report
'''

In [45]:
# Save this variation's helper.
SS_SOURCE = "\n\n".join(
    SS_PARTS[k].strip()
    for k in ("core", "models", "evaluation")
) + "\n"

tree = ast.parse(SS_SOURCE)

names = [
    n.name
    for n in tree.body
    if isinstance(n, ast.FunctionDef)
]
assert len(names) == len(set(names)), (
    "Duplicate helper definitions."
)

compile(
    SS_SOURCE,
    "sizes_spread_variation.py",
    "exec",
)

SS_HELPER = (
    SS_ROOT / "functions" / "regression"
    / "sizes_spread_variation.py"
)

if SS_HELPER.exists():
    assert (
        SS_HELPER.read_text(encoding="utf-8")
        == SS_SOURCE
    ), (
        "The existing variation helper differs. "
        "Keep it for review before replacing it."
    )
else:
    SS_HELPER.write_text(
        SS_SOURCE,
        encoding="utf-8",
        newline="\n",
    )

importlib.invalidate_caches()

from functions.regression import sizes_spread_variation as ssv
ssv = importlib.reload(ssv)

print("Sizes-and-spread helper ready.")

Sizes-and-spread helper ready.


In [46]:
# Verify prior outputs and synthetic checks before processing.
SS_CTX = ssv.prepare(
    SS_ROOT,
    SS_RUN_ID,
    memory=SS_MEMORY,
    threads=SS_THREADS,
    working_gb=SS_WORKING_GB,
)

SS_EVIDENCE = ssv.synthetic_checks()
SS_EVIDENCE["fingerprint"] = SS_CTX["fingerprint"]

ssv.r.tg.write_json(
    SS_CTX["reports"] / "synthetic_checks.json",
    SS_EVIDENCE,
)

display(pd.DataFrame({
    "Check": SS_EVIDENCE["checks"].keys(),
    "Passed": SS_EVIDENCE["checks"].values(),
}))

display(
    ssv.g.memory_check(SS_CTX).round(2)
)

print("Candidate inputs:", ssv.COLUMNS)

,Check,Passed
0,holm_known_example,True
1,paired_common_draws_and_constant_difference,True
2,observation_weighted_daily_ratio,True
3,identical_forecasts_are_not_significant,True
4,quarter_information_cutoff,True
5,fresh_quarter_information_cutoff,True
6,exactly_three_named_inputs,True


,Horizon,Estimated working GiB,Budget GiB
0,30s,1.69,3.0
1,60s,1.69,3.0


Candidate inputs: ['bid_size_million_eur', 'ask_size_million_eur', 'spread_pips']


In [47]:
# EXPENSIVE: build three monthly comparison files.
# Stored comparison columns also support the frozen reference.
# Only the three named columns enter the candidate models.
display(
    ssv.build(SS_CTX)
)

2020-04: inputs ready.
2020-05: inputs ready.
2020-06: inputs ready.


,month,rows,valid_30s,valid_60s
0,2020-04,259200,166254,166188
1,2020-05,267840,158697,158634
2,2020-06,259200,166254,166188


In [51]:
# EXPENSIVE: train missing models or load validated saved models.
SS_MODELS, SS_FIT_REPORT = ssv.fit(SS_CTX)
display(SS_FIT_REPORT)

# Evaluate both horizons on April-June 2020.
(
    SS_METRICS,
    SS_MONTHLY,
    SS_PRIMARY,
    SS_SENSITIVITY,
    SS_EXAMPLES,
) = ssv.evaluate(SS_CTX, SS_MODELS)

print("Lower MAE/RMSE is better:")
display(SS_METRICS.round({
    "MAE (pips)": 7,
    "RMSE (pips)": 7,
}))

print("Monthly MAE in pips:")
display(SS_MONTHLY.round(7))

print(
    "Primary tests: negative Delta MAE "
    "favors the sizes/spread model."
)
print(
    "Pointwise intervals; Holm adjustment "
    "covers all four comparisons."
)
display(
    ssv.g.inference_display(SS_PRIMARY).rename(
        columns={"Group": "Comparison"}
    )
)

print("One-active-date sensitivity:")
display(
    ssv.g.inference_display(SS_SENSITIVITY).rename(
        columns={"Group": "Comparison"}
    )
)

print("Predicted versus observed EURUSD:")
display(SS_EXAMPLES.round({
    "Current mid": 6,
    "Predicted change (pips)": 5,
    "Predicted EURUSD": 6,
    "Observed future EURUSD": 6,
}))

30s: three-input model ready.
60s: three-input model ready.


,Horizon,Inputs,Training examples,Early-stopping examples,Trees
0,30s,3,1960094,476052,797
1,60s,3,1959311,475860,107


2020-04: scored both horizons; forecasts saved.
2020-05: scored both horizons; forecasts saved.
2020-06: scored both horizons; forecasts saved.
Lower MAE/RMSE is better:


,Horizon,Model,Forecasts,MAE (pips),RMSE (pips)
0,30s,Sizes + spread,491205,0.755641,1.128385
1,30s,Reference 21,491205,0.755400,1.129039
2,30s,Zero change,491205,0.755263,1.128142
3,60s,Sizes + spread,491010,1.086150,1.602966
4,60s,Reference 21,491010,1.085756,1.603952
5,60s,Zero change,491010,1.086040,1.602911


Monthly MAE in pips:


,Month,Horizon,Reference 21,Sizes + spread,Zero change
0,2020-04,30s,0.811410,0.811808,0.811442
1,2020-04,60s,1.158846,1.159566,1.159401
2,2020-05,30s,0.664188,0.664336,0.663801
3,2020-05,60s,0.956403,0.956565,0.956456
4,2020-06,30s,0.786456,0.786630,0.786389
5,2020-06,60s,1.136138,1.136427,1.136373


... (2 outputs left out)

One-active-date sensitivity:


,Comparison,Horizon,Delta MAE (pips),95% interval,Adjusted p,Evidence
0,Vs reference 21,30s,+0.0002412,"[+0.0000594, +0.0004332]",0.0188,Candidate worsens MAE
1,Vs zero change,30s,+0.0003781,"[+0.0002860, +0.0004676]",0.0008,Candidate worsens MAE
2,Vs reference 21,60s,+0.0003942,"[+0.0000142, +0.0007928]",0.0402,Candidate worsens MAE
3,Vs zero change,60s,+0.0001095,"[+0.0000575, +0.0001602]",0.0008,Candidate worsens MAE


Predicted versus observed EURUSD:


,Athens time,Horizon,Current mid,Predicted change (pips),Predicted EURUSD,Observed future EURUSD,Interpolated outcome
0,06 Apr 2020 10:00:00,30s,1.081005,0.00000,1.081005,1.081305,False
1,06 Apr 2020 10:00:10,30s,1.081305,0.05000,1.081310,1.081350,False
2,06 Apr 2020 10:00:20,30s,1.081160,0.05000,1.081165,1.080925,False
3,06 Apr 2020 10:00:30,30s,1.081305,0.00000,1.081305,1.080705,False
4,06 Apr 2020 10:00:00,60s,1.081005,0.00000,1.081005,1.080705,False
5,06 Apr 2020 10:00:10,60s,1.081305,0.04907,1.081310,1.081000,False
6,06 Apr 2020 10:00:20,60s,1.081160,0.04846,1.081165,1.081065,False
7,06 Apr 2020 10:00:30,60s,1.081305,0.00146,1.081305,1.080915,False


In [52]:
# Save the execution record and Raymond's requested experiment.
# Model adoption and human interpretation remain pending.
SS_SUMMARY = ssv.finish(
    SS_CTX
)

print("NOTEBOOK_06_SIZES_SPREAD_EXECUTION_SUMMARY")
print(json.dumps(
    SS_SUMMARY,
    indent=2,
))

print("\nReports:", SS_CTX["reports"])
print(
    "STOP HERE. Tell Codex: "
    "Sizes-and-spread outputs ready."
)

NOTEBOOK_06_SIZES_SPREAD_EXECUTION_SUMMARY
{
  "notebook": "06_gradient_boosted_regression.ipynb",
  "run_id": "regression_06_sizes_spread_v1",
  "fingerprint": "ba65a1f9828279dc1e34fa16cc94b8eb92f6c167fd414ee17ca8a97ff095f62b",
  "reference_run": "regression_06_short_ma_v1",
  "candidate_features": [
    "bid_size_million_eur",
    "ask_size_million_eur",
    "spread_pips"
  ],
  "horizons": [
    "30s",
    "60s"
  ],
  "training": "all of 2018",
  "early_stopping": "January-March 2019",
  "fresh_development_comparison": "April-June 2020",
  "primary_metric": "MAE",
  "secondary_metric": "RMSE",
  "objective": "regression_l1",
  "comparisons": 4,
  "multiplicity": "Holm across four tests: two horizons by two benchmarks",
  "nonzero_machine_checks": 0,
  "synthetic_checks_passed": true,
  "forecasts_exported_without_future_label_filter": true,
  "final_2026_period_accessed": false,
  "model_promoted": false,
  "human_review": "pending",
  "uncertainty": "approximate and conditional on